# Pindah akun/VM Colab: backup CPU → restore bobot → gitar berlapis

**VM dan kuota GPU tidak dapat dipindahkan antar akun.** Yang dipindahkan adalah file bobot yang masih ada atau backup yang sudah tersimpan. Tombol **Sambungkan tanpa GPU** cukup untuk backup, tetapi mungkin menghubungkan ke VM CPU baru yang kosong. Notebook memeriksa keberadaan bobot dahulu; tidak menganggap filesystem lama pasti masih ada.

1. Pada Colab lama yang masih punya file model, salin sel notebook ini ke sesi aktif, biarkan `ACTION="backup"`, lalu Run all. Tidak perlu CUDA atau GPU. Backup hanya 34 aset model pinned (sekitar **13.86 GiB / 14.9 GB**), bukan seluruh VM, runtime venv, kredensial, cache ganda atau hasil audio.
2. Simpan arsip di Drive/lokasi persisten dan beri akun baru akses ke file tersebut. Pada akun baru, buat shortcut file/folder bersama di MyDrive dan isi `ARCHIVE_PATH` jika lokasinya berbeda; Drive akun baru tidak otomatis melihat MyDrive akun lama.
3. Pada akun baru, pilih runtime GPU yang tersedia, ubah `ACTION="prepare"`, lalu Run all. Bobot lokal dipakai jika lengkap; selain itu arsip dipulihkan dengan verifikasi hash. Runtime Python dibangun untuk VM baru jika perlu; `.venv` lama tidak dipindah karena path/dependensi tidak portabel.

Source/engine/style gitar sudah tersemat. Setelah prepare, notebook melanjutkan dua lapis gitar dengan opsi preview30 atau full90 seperti notebook 21. Default `ALLOW_MODEL_DOWNLOAD=False` menjaga agar path backup yang salah tidak memicu unduhan bobot baru. Jika VM lama sudah kosong dan tidak ada backup, ubah ke True untuk unduhan baru. Akun tujuan tetap memerlukan akses GPU Colab yang tersedia. `DRY_RUN=True` menyiapkan source/inventaris/plan tanpa mount Drive, instalasi, transfer atau inferensi.


In [ ]:
from pathlib import Path, PurePosixPath
import base64, gzip, hashlib, importlib.util, json, shutil, subprocess, sys, uuid

ACTION = "backup"  # "backup" pada VM lama (CPU boleh); "prepare" pada VM GPU akun baru
DRY_RUN = False
USE_DRIVE = True
ARCHIVE_PATH = None  # Isi path arsip/shortcut Drive yang dapat diakses akun ini jika berbeda
ALLOW_MODEL_DOWNLOAD = False  # True hanya bila cache/backup memang tidak tersedia
ROOT_HINT = globals().get("ROOT_HINT", globals().get("ROOT"))
BASE = Path(globals().get("WORKSPACE_BASE", "/content" if Path("/content").is_dir() else Path.cwd())).resolve()
BASE.mkdir(parents=True, exist_ok=True)
assert ACTION in {"backup", "prepare"}
# Embedded verified bridge source, compatible engine and selected style MP3; no model weights.
# BEGIN EMBEDDED BRIDGE SOURCE
BRIDGE_SOURCE = {
  "configs/acoustic-drums.json": {
    "sha256": "bcfd73641fddbda1fb0975febfe28cccb3a3121ec48cec1d877933e64a6d70ca",
    "gzip_base64": "H4sIAAAAAAAC/0WRTWoDMQyF9zmFmPUk9IdQ6K60m56hlKDYmoyoxzKWJiGU3r3yTEo3Br/PenqSvzcAHcfuGToMMqtx2MY6T7o933d9g5NESisnNSoO9tsjKq040pkDNR7miKtWKgVWltzkIQna48NKOKu5O2VraGm0AiWK6tqHXwDu/Pxc9NPMEXOggwZMrc3T7u7mNFClhXioVrpfgQytYzyYHEKZXfeGtJCAxW6h3lUSGkX4GxpaFvhicyXIVDBzS7mDF6g+dkXOFHvIaHPFBKcqcia4sI1eFL560IyVAHOEkETdeeTtiNbDICnJhfMJbCQojFmgUB2kTm0wMJ4c9lDGitqeNY94zThx0B28JsK6phvZtAdDH3iYEwycki6vtfjfwJHsQpRXI/LSt7ZekJyuHlzgLL5Cv1eYKEn0mf9/Q3fd5mfzC1x9T/sNAgAA"
  },
  "configs/assets-base.json": {
    "sha256": "b87ff04e90500281d47e37eea933dd756cca895df4eea796f9d4c7a98c2f1dde",
    "gzip_base64": "H4sIAAAAAAAC/+2bS3MbyQ3H7/4ULJ0tGd0N9GNv3sSpzSGVxx5TKRa6gZboFR9FUtp4t/a7B6RJxbWRZHHopMa1OfA1D8z0rzHA/z8kf341mVxs2o3OeXqv681subj4ZuJe7xbPl6K39umCm262urq8d3RZeaMX+9Wb5d266W79n3uftRnfTr67u76eLa4nf7A9Jm//8sfJPsRkrlsW3vLryWq2WKhM1no/2x1q8zFSn93qxgL93T5MJj/vn23xWlfL6Ux2R3j7u3eX39spvHnbdP/GXdF+38OGH8PttnQlJtcRokQIzsfiY3M1qWdpLhZutbGo9//ee3f0Bc/3I/nrj7oIl+/mVUVsIJdwFb99wyIq0+3yB11srt5v7DgP+4qBmS14ezi4cWw/rJazxXbz5tRQm9lPu1NIkB4W8e31cj3b3sx3sa9n28t6u6yXmxt2n5zB7NrOYbdBJezFBVLE5jQ7wKRZgFoFn7jn2iAUVbnY7/rL63HCbje8nW51vrrlrV69ny3e83DczwY7AEfn4kDisZTesxgA1xVTzuRJS3O91By5FLXpjj7AyIkvF312fWZiPxrkQNhysgwk7F2V5HxL0TmU5j3l1KJqzcg1osuxFfFI4yY817UN6Gr7z+1wwI/FOPK1E84UBiIOLmAhclLAMEPswVNX6M7VWnMV6gySWho54l2judpw162V1uV6cwbpp0MdgbviKFv6PVE4DLWn+BhswJAUqRWkGCoUdaHVxBGtevRmRAomLrYsICcFy3boLUsTzOpDH3nt3qx0JwMO/W0659WZVeWzAQ8TEt3Q7OfmQwNm9hiQ7D0WDAFYayqNcigUkrXSMG7wez6GYn0m76fiPOQ9+pCATs160Z5ir5BjSa6lHFPNvhh59tFHpyFLkYQK3oXiKdSQbQKKmGbBSsBfCfzpF+mknwl3mApCwIEpL8I9lgzZO2TELuKjWJZTjcYiRKv12ac0cur3y8b1TNSPxTjwNQAxh6FVxap56OqAW7ARqiNuuQfK2gmjT6GYUmwUy2gQ/+3d29//6d3VXF4E85GtH9KyDBXTHJyjps0bumpSTwisLhN06d04tpw0xlzGk5ZHV0zf/XR5O790V2moVTwx0jFFvUspD3WLlBESgcsqyk2iNLOHrgb1rdZeIIeoldGNGvdQs3hyrAevGPNA3uCqlhogiWkNQdPYyQadW+nVIbHPUYF7xXHzPrHBvTjGg1PMNJCvM9uSizPjYgkMUdn7YmamtFwAk0AREDZrPmq+JxrFl4b4bfrER+kMsoknRjrgNqVsbd6Rz6fq5e6ilYYinIKSNu1WHnpPGYScNcbiGaADRTHBLKmE0FIqKtRikyQ1j3oKzjGJg+Idsz97dGXojSjeW5VolwBDD7lqhKyQrGC1XX0JZudhd3WMGv0Am3hKmKMuweBdCeXUrA/EnWILnZrJEPatUcrGPXMuIaUEJTsThdhdDtyo5WbzwB1rFbQX+TrQT79ED32ZR3Sm8Lz56VMnYldFoInkYjLFHHsq2MkHoujVZKJlvjVYXzg2jDmykuNcc7NHZdJCo56IE33jS0P8Nm3jp18Mbu/WdTlYIX4mwjGjy2D9HRkqNbCsNZYOA/iCqYv1T0DXkwqkiJLb6Nnerfcgp4f1U1t/tfpwFuzPhXxQ5y7AQPwlEtvYg1kcZ5Lc9DiVliRDNM2O2WaiixLQiPGfpx1fFOdoM83WZyvOT93ie6aHdrORqaTOZjGBQ6kSQjFxWKOa4VcTKbloQA5W1G1JEB9qBG1R0XtJY8c/W1x/mqXT/arh6f/CmIdpKTaQoXe3OjsXPRODaw2pJcv/HhoFZ0CyT74xkAMc8QRsdmubTnf3Rxbbq9V2IPRn4hydU0ZnxunU5OeUshYRky2dNFdv5b4RmQ+zq0mCyzmgadMKQbjWJj2BRhWT9BKty9bRsD+1if6/ZX7kds96sgB5cp9jLfZD70h1tLKbqo9OxLScii8NrA5oKuDJmcBrVXyGUeGTWe93u3DT1Yftct1upsPa3smxjpd+SGgyI598z0TYWUZaL8OGIModipbqsSRUKiCusdlGiw521ZNAztg6Y1UpXEo9YRYe/TXcf0yFWqXxzndtVnHyXv5g70yJNVLAkCVyFYdf4vuYX5/R81/QDL6DXarT4uxBtSJhUDOCls9WPkuonYL62oD6uEjy6np6fTcTtpYzRCjseT4Z5PijvUJDhQEgZrZRRGNqrYljTIxQAHISiTnbGL0vvY6L6jk2b0/0+ZaFQ21GZUOGXpOH1BWs6vZkNS+WrAjdeSuDwaOTMdL8QsbuE7xfpa/7XwA/38rtKf83nRy6lHoEIHAcBatk62DCDFzZRG3qJsSokTW3llslDxSaWL9jyIgS6wiB/9po7Q42OMNfFvHo3IhwqIyDnlsniFATscdorqFn82qmVELiYFXbEh1zGBfw883anvLYvZo9/2P/FwJZ/ri4XbJM64ft/r8EDi1ELi6he/XLq38B/WBJB+cwAAA="
  },
  "configs/assets-xl-base.json": {
    "sha256": "7c6e217c4f044edf7c648d8c8d743b7b0be08af5fe74d2d203812ee85170e1ef",
    "gzip_base64": "H4sIAAAAAAAC/+1bTXMbuRG9769g+WzJaKABNHJzEqc2h1Q+9phKsRrohkyvRKpEymvv1v739NCk4iSSzRnaqdnaqEoSOR89g9eN1++Bw5++WSyebdtrveHlW73brjbrZ79ZwPNh881G9NrePeOm253eXryFePHu+qLyVp/tj9hu7u+aDof8ufdVW/H14tv7q6vV+mrxBztp8fIvf1zsoyxudMfCO36+uF2t1yqLO327Gq62/RCpr651a4H+bm8Wi5/2f23znd5ulisZrvDyd68uvrO7ePGy6f4FXMb9uYcDP4QbjoSSMnR0SZIL4FPxqUHN6lkapMKtNhb1/l9nD1df881+JH/9Qdfh4tVNVREbyIW7TL99wSIqy93me11vL99s7ToP54phs1rz7nBxg7J9f7tZrXfbF2NDbVc/DreQXX7YxNdXm7vV7vXNEPtqtbuo15t6sX3N8NEdrK7sHoYDasReIERFbKAEDrOSuNiq85k71eZCUZVn+1N/fj5PsNtr3i13enN7zTu9fLNav+HpcH8y2AFwBEgTEU+l9E5iAEBXzETRRy0NeqmUuBS1dCcf3MwR36z76urMwn40yAFhq8kyEWEPVTL4lhMASvM+Um5JtRJyTQiUWhGPcd4I3+idDehy9243HeDHYhzxtRumGCZCHCBgiRGkOIPZpR587Oo6QK2VqsTOTnLLM4d4aDSXW+66M2rd3G3PQPrpUEfAoUAkK78niMOg9jE9BrbDkBVjKxhTqK4ohFYzJzT26M0QKZi52LaAnNVZtbveSJogqQ995ty9vdVBBhz62/KGb89klc8GPCQkwdTq5+ZDc8zsMWC011gwBMdac2mRQokhWysN8wZ+j49BcXcm3k/Feah79CG7OLbqRXtOvTpKJUPLlHIlXwx59skn0EBSJKM6D6H4GGogS0AR0yxYo+NfCPjLL9JJPxPukIqIDieWvAj3VMiRB2TELuKTWJXHmgyLkIzryec8c9TfbhrXM6F+LMYBXwMgUZjKKsbmoSs4bsFGqBC5UQ+RtEdMPodiSrHFVGYD8d9evfz9n15d3shJYD5y9ENZlqlimgNAbNq8QVdN6kl0xsvRdendcGyUNSUq8ynLozGO3/54cX1zAZd5qlUcGelYoh5ypqluMRK6HB2QinKTJM3sIdSgvtXai6OQtDLCrOGeahZHx3rwiokm4u2gaqnBZTGtIWgaO9ugqZVeASN7Suq4V5w33iMb3MkxHpwixYn4gtkWKmDGxQrYJWXvi5mZ0qg4zOKKOGGz5rPGd6RRPDXEr9MnPorOJJs4MtIBblPK1uYhehqrlzsko4YinINGbdqNHnrP5CSCNcbi2bnuYhITzJJLCC3nohJbapKl0qxTcI5JnBTvWP3kEcrUhSjeW5VkU4BdD1Q1OVKXjbDawC/B7LwbZsesoZ9gE8eEOeoSDB5KKGOrPkTuMbXQYzMZwr61mMlwJ6YScs6uEJgoxA4UuMVGzfLAHWsVtH/yy4B++SV66GkeEUzhefPTYxMxsIhrIlRMpphjzwV79CHG5NVkolW+NVhfODVMlFgjMFVq9ls5aomzTsRI33hqiF+nbfz4s8Hd/V3dTFaIn4lwrOgyWX8ndjU2Z1VrWAIG5wvmLtY/HULPKi4nFGqzx/b+bg/k8rB/afsvb9+fBfbnQj6ocwhuIvwlRbaxB7M4YJLc9HgsLQu5ZJodyTLRRaOLM4b/PO14UpyjzTRbT0bOTy3xfaKHdrORueTOZjEdh1IlhGLisCY1w68mUqhoQA5G6rYliA81OW1J0XvJc4d/tb76uEqX+13Ty//EmIe0FBvI1NWtzgDJc2QHrWFs2eq/hxYDGCDks2/sIjiccQK2w96my2F9ZL27vN1NBP0TcY7OiRDMOI0tfs6ZtIiYbOlRqXqj+xaj+TCbTRKAKKBp0+qCcK1NenaaVEzSS7IuW2eD/dgm+v+W+QG3t6yjBciT5xy52E9dkepotJurTyBiWk7Fl+aMBzQX5yOYwGtVPLlZwSer3u+HcMvb97vNXXu9nNb2Rsc6Tv2Q0WQGjV4zEQarSOtl2NCJcndFS/VYMmosTqCx2UaL7mzWR3FE2DpjVSlcSh2RhaceiPuvbHjvmrF+0l6NlCKZqe2tgtcKycVixtXeeoQv8ZHMIzf1iY9pMOPUhZDeCYur1r7AJJspCeMG40/g3s23mOiIPjvxODs8+fZqeXW/ErbeM0UxHFF9Ms4BW2syU0Vy6ADInqBK6iWTJE5YIhfTxzmSSecimNnPDttzXN8R10+RsI+TWbh2riTeg7IGyiX0avPepZSLgIsmCyRhrH2mmH4ht/fvIJ/s9/xk2DWimtHz3Tkka3YhmpqIVBopmz7zxtUJfJ4fTexb1IWzH7jY9P0LPMv0HaE/PfCRpMv+eRmMaXwvNMVmfZ5Tsjsxh92rS13FWiB0QPEOI6g2iQGbQ05FLFHe5eCCdh9mnBT/tZLiRyQFh2W+0e5E1JQHkI27mwWvKaYm7GyaxBq9ACO5jkb5TtGBCeygaXhAB1xgn8c89/A/T0r4WkkJo5JCbrRqVMQatQ2fXbpazZ0Low8hxpx97TUkIoEQnFmeapOqdYkJbOrU6l0vEmecFPxaScGTk0JoYiiNX8QqBdWkZFIYVgtb8UhUQ++dqdrwKSkJD3K++sBo1mbIiGMBLzaV6jyT8jFYl6u16Lvz9NJpQY/fKgHMMPV5wUChm35Cj917b+xl5irl3EyXBjG/OTyeZdNonrD/5+reu+vlcL1zRNTJcY86ym6VpqLfkxYjJGoZpHTVzkrkWnaxIaF5e+1QzJDNDv3zlwuPeM99wdD+/mP/PTbZ/LC+3rAs6/vd/gttwUgyGnlRCt/8/M0/AZIVRDxvNwAA"
  },
  "configs/clean-electric-guitar-pick-layer.json": {
    "sha256": "7aaff6ec3d7fb3a8c439d4b6d9e2baca3edc65835e3b59e64891d9b6694303bb",
    "gzip_base64": "H4sIAAAAAAAC/22QQU/DMAyF7/sVVs9tNZgmJH4AiDNHhCaTuKtFGkex2zEh/jtJO25cKuU9P3+v/t4BNOybR2hcIIwdBXKW2XXnmQ1zl9h9dgGvlLvl0LR1fBJPoSbQkRqlbrk7dh+otNmeFna0bpw9blrK5FhZYpWHIGiH+83hqJbniaJVa4NujhJ5LeJbeQDsy/d91cuMx+jopA5D5Tz0+9uqgTKtTmlVo8fNkKEi/cnk5NJc9EKk1XGY7NbqRSWgkYf1DvB3B9gqATonU8LItSpwBBsJLpindg3kFuboWU1y3WESCWRYhzLdavXwmjArQUSbM4ZwhXrdihwle4hipIDRw7kwAsFEQXypMHAIChe2EeirnFKVF4KEs5K2UOgLxzNkkQkGKVUhj1cbp1v1Hp4kBLmsZRJjlI1XosZTTVZkGjNqefTw/N8PSwzXvtn97H4BIIS0EzECAAA="
  },
  "configs/clean-electric-guitar-picked.json": {
    "sha256": "aaa620f12c1cc9db21a56d4c45ca942de68346e2e1ba4fe46a1b61cdcd7b392e",
    "gzip_base64": "H4sIAAAAAAAC/11SwYrdMAy8v68QOSdh27IU+gGFXkpvPZTyUG29xKxjGUvONpT+e2UnhdJLIDMazTDyrxvAEPzwAQYXCdNEkZyW4KalBsUy5eBeyE/722Fsoxt7im0aHYlSnvY3z9MPFDppT3tw1LdVjyeWC7kggVODH5FR313LQhItdaOkjToNT0aIvBj4zX4Anuz7veM24zE5uovD2Hzez0/XqgcV6oylatLnk+BHs/R35bvL1XBzpM44zHql+iQcUclD7wD+dgBnJEDneMuYQos6w1cs2wi+HCPU5IMol6b9X/UadAXJWIQg2NwefMUYD8ixtk7BrVw8KCcSwORhse2RAEumZQksIxgFVlFIC6ACglqCGT6j1oLRIEX3cmoTK0GxCHYKE9JPa10k7AQZa4falKVxBA8uoKsxARPDRpH9McNHjpFf/yF6PBOaqQUYIa8FpUexTf5IuAUnM3zpD6QHEAscrRUTl/XQ1QagX/hcoIU2jmzVnZ1Z+a3GupmJPSG5ihsvf0u5s51Z5uH2+/YH5wMIqKkCAAA="
  },
  "configs/clean-electric-guitar-strum-layer.json": {
    "sha256": "885c989481abca270fcea2baa77cd843c73efcde4ad3e2f6d42ad9eeee05d104",
    "gzip_base64": "H4sIAAAAAAAC/21Qy07EMAy871dYPbcVsFoh8QEgLvwAQiuTuFtLaRwlbpcK8e/ksdy4RPKMPTOZ7wNAx7Z7gs44Qj+QI6ORzXBZWTEOSeO6DA53isN27Pqyv4glV07QUFIKw3Z/Gj4xUaMtbWyoSq4WGxYiGU4svsCTE9TjQ2PYVwvyWqjm2phEZFMG3/MAcJffj4rnHYve0DkZdMXncby7SU0UqTI5VTk9NUKmYmnPKmcT1oxnR6qMwaC3VK9JHCpZqEXAXxEQ513nBVoyQGNkCei5JAb2oDPBFePS17vYw+otJ5VYpFQ8gUx1KdIt3QhvqGtEB5es4QjMLNFCrWFhf4Er6wz0lTtLiTcCu3tc2CRAn9dCbh0myVG8ZN14SzbCszgn1+oVOJNNN/WgXGXLdZgjpjyM8PLff8S7fewOP4df78QTcBgCAAA="
  },
  "configs/clean-electric-guitar.json": {
    "sha256": "0dceb443e8cb071057c6367e11290463217123569c0263e359685f10825728c3",
    "gzip_base64": "H4sIAAAAAAAC/11SwWrkMAy9z1eInDOh7VIW9raHLfTcYymDamsmZhzLWEpmQ+m/r+zMwDIXE78nvfck52sH0AXf/YLORcK0p0hOS3D70xwUy3557PpaM7GnWMvQkShlI573nyi00Z6W4KjJzB43LBdyQQKnCh8jo/542piQRMs8UdJKbU4bI0ReDHy3C8CDnR8NtxqPydFBHMbq83N4uEodqVBjLFVtfd4IPlZLf1A+uDwbbo7UGIdZr6lehSMqeWjDw2142CIBOsdTxhRq1AF+wwXL1Lfi0sOcfBDlUvvvO3PE1eCEOheMcYVL0BFOJhMJcnDn6jly8bBwcCGdBDB5kIxFCMq46jiZWluTDPDCMfIFdKzNmBgylSOXqa5k05EeNEwm1EMeC4p9NUW/JjQloy30UlHzsCbr/k9uosh+HeBt/rwFBFRFd+5vM0BiJfDkcG3C9NfeVyQsBFg0uNkWaVsd4M/dKjjF1VQY7G+RK9iDr4P1V3vLsrA9rAzd7nv3D5AvL82UAgAA"
  },
  "configs/electric-bass.json": {
    "sha256": "4bacacb4f9fb1a9e66c98958869e2756878e6420910b2af065bea6522cb6ef63",
    "gzip_base64": "H4sIAAAAAAAC/0WQy27bQAxF9/4KQpts7MBpERTorst8QxAYDIeyiMwLQ44DI8i/lyOp6EYY3SPx3OHXAWBSWjjh5cZNpeTpNzwdRyzBjxNHJmtCp3dUnVaQSuA4GBKrcT3dnp4H5g0Hvgnx4NQDblltTLJPn+ZY0H7+2IhktdYTZxvov0SZg3r06i8AZ3++rfm1S8BMfFHCOCy/Hs/7oJkbr8Q7jV+fN1DmIQwXKxeq3XP38UoIq+2dXrRENA4wS776IuweGf7dHUYtcLNhAyQqqWKW0fkR/kBo9yNktN4wQo2dPnyMlczwKbYARcYW74DNhPomQTOkDz2CLqUZ5GIMzW2+QwXM/kEP8u4Neq3cYMGWShbSoVNJ1UnqKr6BtVoUl0kG9Ekt7eFp79v4Kr6RBnOJsXz69cAWhiqYy4Pus/0GJmmwYQ/3jGnopsP34S8y88VlIwIAAA=="
  },
  "configs/electric-context.json": {
    "sha256": "f70cd046214eb82fcc29bc6da216329d6229b13708df12e6aa0552e2cb998854",
    "gzip_base64": "H4sIAAAAAAAC/02QP0/EMAzF934KqzOtDiQQfzYkYGFgYT65qduzmqa92EGHEN8dNy1wU+SXZ7+f/VUAlNyW91CSJ6eRXdWgSOWmoHTS8mIxbMXe4bwXsqoV67jb5c8RY8/hTL/JcoctnYm7+vo265JzeApL5iuLUqAIc6SOYqQW9EA2uVo7QTj0nqrZkCCSJK8P9s5TVLNKakQxKKOHMLEQoMLl1Q622Hql/0DPLf5G/s/uiNoG3QAjKZoBwR3IDTYYe+QgmllWXhOftvvA4wJjwEJaw3PyvpIp9NacwgD9sk4OA1z4CUc4JiPQTxBl7w3/mDgSvLy9A53IpT+zz9ewjeuy+C5+AFLyNqSbAQAA"
  },
  "configs/mac-24gb.json": {
    "sha256": "16fb27fa5f89e4c5965452192ae1b0a3f5878d3d24ad44953521302668b4a429",
    "gzip_base64": "H4sIAAAAAAAC/02MsQ4CIRBEe76CUHuGK2z8FWPICnsJycESdrnG+O/CYWE5783MW2ltEgXczV0b8MiCZTnW2/ICRnMZOuARPQ6fCk8UM0ttCbMM3Ks/zoiBO3rY55lp23aC4IScL62LvsJ52ipIpOwYPeVztNqrnS8CVf5E5+qjvjXuBAWrAAAA"
  },
  "configs/mac-full-song.json": {
    "sha256": "e69fad3eeef49d4fac70c909418b75643abd8c115d98155c6e466580fd4e9b74",
    "gzip_base64": "H4sIAAAAAAAC/y3LQQqAIBCF4b2nENcJtWjTVSLEnBcImdJYm+juabWc739zCSlViIRVDVJZB85I+ux6PVuGamomnN6h9pD4I79x3o+ALVcu098ZIC40ttN7x2VZoyWTo3HpKKF8QdziAZ/359J3AAAA"
  },
  "requirements-context-chunks.txt": {
    "sha256": "f66d99f5111b5adc05fd368747e875bf36c9f3074984e636df275513fa9e8587",
    "gzip_base64": "H4sIAAAAAAAC/yXJQQrDIBBA0b2nGOhWbLXQlcmq+5wh1SkZojNFx0Ju30BXn8e/wNLShl3bqiR8ZVF8ieyA/KUmXJEVhMvh4ClwXiDuupZyVgV0QyiSdsxQJWOBNlipojM86ueYJ+/Cw8a76Yn+9N7GYLoMzm8qOE8354ON3vwAtgHTaYoAAAA="
  },
  "scripts/ace_context_chunks.py": {
    "sha256": "a94943eb4ce12d307715b010f4b5b6871314e4d52badde7de5a1e463d5990312",
    "gzip_base64": "H4sIAAAAAAAC/7U825LbRnbv8xWQ9gHADAhyRpZscQaq8jrazVaytsr2plLFsFBNoEkiAwI0AM7FiqryD/nDfEnOrYEGCFJyvKuHEdHoPt3n9Ln3abx8+fLb795Pfmr03lmVhyLV6SQpi0Y/NY5+2usq2+miqR1VpM5Oq/pQ6TRwarXb53qi8mxT6NSpm6xJtlmxCS8uftSHWtdOs9VOstXJPbxeqbqeptVhN90cskZVzmNZ3euqDp0PuSqm+6pcaSfVa3XIG6cpnaJ0smKtK10kOrz4Xuu0dorDbv8MEycZ/YcrXWe5Dpz1erfXm8DZ5weeVT9lsJ5i43z3t3/61qkORQMoOOuyciYT/aSTQwNAX758eZHt9mXVOKra7FVVa/O8VfU2z1bm8T/rsjC/d6rZXqyrcufs4Rd0cuTFB3whv+vtocny9umwAvwSXdcXpolQcRTgtG97GXywuV7zHIRrWAOJVW4mSsqq0rlqEPGsSOO9VvcdZJ3rpKmyJF4f8jyGhW8IHhAj17Az3/783omcVzez2ezixx9++BkecOFeHOPUceyHla7L/EF7fggkwX1fXC8vLi5gbwAtdfP6jYeI+/MLB/6l2UbXTST0CqWDT+8es2bLwGlAWO514bnVyvVpRU2l1Y6h4D/cnFVeJvew707W6MrL1W6Vqjl3hGWp1Lue3Xx1iX/8YOW6/pynDw/7FMjh0XCeu9LNoSpkeeFWP/EvWBlj8ljBFDHuKy0ueFD5QQtO2BB1675tNFJWVc8RPoeIVlwf1uvsyXPDZrd3ecq2W8jAUXw8nCFMYbNrj6YIYMOAptFNoPK8fIwLVUR/Unmt/Sv3P4ojSJXe5yrRvBBZeq6ey0PjNWWj8sAROY3ezgJgzWqTFdEb4AuV6ih8/Q28P+g6+r4sgFl0oavNMz0IprSkOvIESMAAAhzNK8nWjqeKZy8D/qkbBbLoPQSrssx9B7arKBun98bLiiZY56Vq/LYDykuY1cCpQBTvwaeNfsBN5tn9lgNkwPXsLpIF3UVvZjPTPrvDdd3xGrGRf91d428ZMLm5NK0zv+OtSmW1dv4N53tfVWXluahQzCCY8X//+39gptqQ8F10XTMRcZ3eTAjjI7krDW9ntdsSaEAF3hcghN8hdIlidxfRK8KJGs6t76fyUCXa2R3qxgHFaFbo1BoWnZrZ/+D8BTDYaKBFmercSQ+VarKyqGHZSX6A5WtQ2XuVpqiXne/1A3TVTwkij2qy0r/ADjRICrUPCeROPWW7wy6ibYOthMUImXxatEhIGlWorjwmDL3xb4k6kUCArYBuhkj45u7683gbfMzWwLqcXKsHsCY56E9YtSqAGrVTMkxHHdKsFHIksKQmusb5hNSyFlCKMA88eDcBIZboLPd4q3id/hSh+QJH7VWSNRmIxgKbr6ADAs2QGxazgOaZXC8ZLC0DmZpeV6rYaI96+MsOe1xYdD23IcvauBNYJGiaLW+dfZlnCXUwb+g3gYGfwGoOSrDMDS1mkoGcNifltPkCOW06sewkc3bX3BHNpmRGEOcGccZVEBz88SKqwQzp1Kt149Gbs4L4R2QjUHUOWui6ZXheRuAcigxYFI09wiQPBPDIgLORf9G6yd4PNuCaN8meWu8UAC02LIYTpPgCNrHtAE7JRjeRab9iBm+HXXZ7t8jg/bQ+7Lx+23zpd2TLS+Q+rwUnzB/w7MPBvaHbLNplRTd0MLOAYCabZP6lgO4AADfk5bttNh9RfKWzBhcuW4GfsTK0R5evGmoElsCp6Ns822WN0TsWuXCh2yxAVPMy4MYeFQLExn4LAkerDWQot17dDHAo96TJogXvQ8Ov+0zHiN5FvS530TZb2qQQSHObo1HgaO3yMrjXzxE7HE4zV6vaayYGGZHJ58gFL+4BeC+NDeXcFiaK42AGYNYM/ZI6ysEb9ZgxiSBX1wEudToFK0WCsxgn43J0yR3g/qo9NvAgu4g1P7CamPlX4Wx2c9nDa9qnN+l1wRM9Ex4fF1pVMY9wh2DdlcpRo6QxCm9HC2TcUO3B3UuRif1OqZlWnsi/GPYmzpbm7aG4b7UgiXfgqWDl4+Zr8J91hT7fr9meBCUgaSER7LYBFF7VBACYOHAWqAmqemI8NtMrVO0dDYz9jICU6YRGM5naHhDFaAkC2IiYIX7rtHRkIAwMah9ddP2e3HkWuGgyYgLvzpU8Qh93vgqg1/7QmJeMgLRRD/gT9PbM+ueSFxCbJbnzkWV+6vnHH12O4mKkpTsnBeUSaeJ1pUAhu3MmlCvIxaCLYuOEzI3feNFfBWmMrpd4lS76U10rPgWuEaTYcIg7N79gTiIgTEP/fxL/FwOBmMx+LxBRjQrqKqrXHCmQV6/yR/Vcxzdp9HMFViRtnvc6csnwvbrpPLi6ehGRRQMmQzgYxuw1hD0vohtj+4p9Zxyxjx+CA+993sfkIcCJEMqWEHfd//OvPZ/FBCoAUvBDOw1xMkQJcbn2wP8vZJbuRYStIYik53aNbtC17iGIA4ENPn7yj7u5GIu7fot+9449BfB4uUvgYsCO/3PMDgHXiKd62O/ZONM2WdDqOUX9gUNQAscAsfHuepvAjNICMci7170zBLDXGkW8yLlAkgh3rB9j0W2VRMrYHFPiwXISeDf6b4bwBJEjgNw+DvLo3d+FkkwzjNxrIpnFLa2EC/ae22EVYlhqtjd8VA/mNyUNXH8g0AZfz7WxMEC4TaBIh1EwvF2eyzsV7+HRwGiTFthHQPXaXP/TYkQwTGKC+X1IAualaJSlWhAjYtC+G9vrB1VlqieCgrO8gJWXVQYKT+VuzyOT90bGeFXhhx/f//T+559GBOtvxX1RPhYOToQJNsmbmWn8IYf1AS6k3/LiVJeOdJjn8WqKOAOnPDRgbL4ssUA5I8kuXAa2kIgCgYklh8caW/r2KRZZBPv8xvnnRJICBFn5u7ezEar+malox5fgT+GeSFz8dhhkw1T99b6wFmwiElnJi45hTk5t+ABd7Qz41lFma3GnZVJm6Mji64+2Dp93v4Mh/837z59aLAZ7gZEk4k3RZB+Nzg58iQHYAvukHfQOMRqMBrSn+YXRmM84z8ZNVuoxoGbuYjXTeG4Nd/dpBtLOKUq275T1jct7yahxb9mpGFOUkeQnZT7qsEGPlnyjmNysiOFPPfa63CvuHHLGLwTXWFeeaA1O84ZJuX9GSTAyNATZORo8/dH7F1Fvmeecih+ko0MjwVJgVJE6j1tMHLMsY94bYzmLnZJcq8Jg5gpGqGYljfSdKsoiSwBsyxrocZPqAXDTXfaEUMsifw6dP/PyIUjHDDXNhGxaPehUwBlsYK9pTZz8RgjgkSZbORrAqAhBTcBDhsAKzxDIMkzQMjhkY2pOSnUJ9LA6FN7C5YQ/8OWkKOsmzdCGTMhoIJWoHf9kbgAsbbYbGoAJqYUD0FE/GjolMPYGxydzBf/tk11cX7/JNY8lWvrLgE42iPGYxqzgLO+UO3ZJHBS0NiNktGe31zvdKHQDI8oZg4+a1h6JQdvXJ9eWE8t+z7aYsWyQGN+Yuc0F7uqzfT9MPZOSMVCdtNS1SRIlW9o8a4spU4l8a9kkQri/qhSM2UqrBtTJYulfDZYMth7jcHrXh3KUTmJ/X5tY1KNIxcKKjgEi2oMFx33UY8Ih9zf+HOO/XBcedfH57ZV5uxzaVQoXPPD/618q+n8H28pnDZeXN76sB0UlktS8BXsksx4gFgGvvRtrzjBqYKudih90VWMgegMsp5pDHbniBrhBqyp4V/uqI5C9J8NsC/oxs/e4xLAI82zQMz+BZWktu3ME0FJr9iqG2i4sgNzBcedRdI4mAS4ExL6X00E0PMBIupq23BU631m2XfLgtbgwtRg6TjL3kuN/gegcA3U0fxxw3pKXkBVgvJ757PNQk4785aBybDsUsE3ZOtNp6H7eLTntRMxxbxdDO76M+g1fYMbtbGff7yJFMmgcGlbKs1Tlr7robGBNxj1uR7pXAyCjdtE6GRS+Gk59OzSdgw4BL6Sn5ASUvHkR8Qxf4pEYK5keyDjipJauYuofYbqMPrq4NHfOMzLfuqJV5ejxE/ur3WmisbEk0xzdUChhu+HYIN73kRh46zJPdRVYYQydR0r71Ao7RsWNp1u4dtPSH241nuonoFWjc+AE0z5AaVz2jr/2fHYQ07F09qDjppT1dvNSntM+QH4RmVWc83f+3C7LWBqzmQBO1yDj4k2AY9BPLeAk3bl1b29HiAwbuLOoccQNnXxDR3MIY3IPtmUfCSP/oe50jztwcQtXNn2w579vrxiy2f2zR5dDBBDCFLW52bqT+4ROT4wkszN7g+P4bpW3ZHMtb4lQO3KRAG9rX8maok/UGtSRPcCSFHGmmfUyOlEtMVXa6F4cwzUBHIZMR0TPJr2Y467AYrANFih7B8RtexGNCuK5rejc+i+VHSHWwmQQly+iYW6FVkaOesg54uECe/njs+v7QKAJIK/PpKnBeqyBfWoOMaxA5lhfWkSzNO2RxI93s1jQfilelHAlGfnnmJwCW3ME1NKm2zB3R5sOLpve8a84GuTmrL425xr5taCMOPsIN5Ke7Ry3bdzAg02+F2NXzP/29D0S5KyWu1VJA94N9bPhDcyz+1uzZihJu7oHEnYeS7RqK0vNvbpl90yVSYAkfLbhtmEocrBkUJpKJfeYK0UeL+gNO0nih8/dsaDPBUlgoH3eX7QvlmD3NeqKGenuFE+9YhAg9A6+HgXZFq/FsFP72p2/hqHdyQyxzmJ4YAPTNKq+j/GcAjDM9aZ0Px1RYnG0G8uo2zZDyQbCMTqws7ae/NCuK5/itcar9S9tNmpTTn6X20KBxWoD3i7qeQ+vHyhncB9QcY+ZNkTDUXv+Z6POb2lZZiM5A25YxNllNYWeHd/ZaM3t5aOitVDEpQ5o08XFNpDOuOGYL2fUE/Lx4sSWfI4M3zHiQxs6IIDOseCjw3I8BjgC/qOBZ44YJPW5Qi/jgEIDOwem71BUXU9L+2I5WI8ynTUVxQwyMiSfSFNrHox02XVf/WTAIMI1Q3+Hnzo6DZ+xAnARRvvkddk/9j0a29ZJdMILgLweJDyvXU7GgPufgS44bFRGYMOvj7sMNEeHxVCl2LMMzDZI5hD702vuwRHHhAH0OQLMktm2s6ejzOiU+CtQnU6NmE3N2tlnHHC+Oe2Decx58KFg21x7VVmyAY8t/9G22OKzWX5m2/k3nBjd2sY+PmHkZarzAd5FL1aPzsYpF9ZJ4zopmrwrsjXugxt204WYoHJN1a264qpbbLRqbhEMVvkl9x7+CbjhX3/47l/i9//+X9bT93+04uWq3IAMy+mPmdq0ctB71Lnnp9sAem4O+Z+9t5TTB/vBturjp169sDknFZ9V6gWWfR2LNTKUtzVMjVUYS4geyN8z61/z6Pgjd3tJ3V4u57NX6Se3XyRD4+T8oT1y6PLAli2CqWl1gtH8yFFIVLIF16YN7qXjAgYux9IzAzvHw4/DCIog5qMBAQ4gf7cNAAhh93iOntvLM9lu71H/fZUVDdoYMCSOSYwJdCxUCtb5od4yoW4ls6Z7UP7g/KiTEmtUFdVN1LhW1WA5NNj0dYMvCsreVdVhjwflhmCcgQn75VISrqVRz8MxvGPgZoUUFwrdwk1erkBg+HV86Q5dF8uDl0622z62VaIgM2Zp4GV0wyx5kPe2IPTta2Tb1/nosUVvt2Rd9nbdduSQt7crmPB+yLJttzbHcZw4ZivChmb/3GzLgg8qUP2CDMM+rlWWg1exxtLYBKahuxiwdzjvsyH9cW73sFuB9p0dveDTLdkgEFWzOR95BAup3+qKOTdfRdfHAtdS4Tyw4yXw0bnZBjY1ct7XAg3IVp72J8ZPnLqqs9/kPgRykYXPPkchX156H932XPiLKhY+fTaoI+4dgjBK2Khp/8yChr7yvG341Mtv9w6oDdzP6al2K06pqp6OZcNgxoTHOTEw9vZVEds0Beapn0qQRnFO0pVnXy3pHSndzC6LPaiAzfXMw3MqbuXuwbWeXNPhkjg5EENx5nwE0PBsip4PWNTBuXSue4NW6v7mK7+Du8dLWz3Iu7Io+fCMgSkQqujaX8zn3yxDVSMozwJl4gI888Kh/t2brygCgAXzM2Dy1bytRtr+6s6pCMTFe2VlmiVZ8wxxdfipnX0S4V+eXY5Wk6i99ERQA/mTYjkf1hoF4B1vyxSe1uDSL9rVYFG2gJgiBaTo3rsOui7QrKQ417QJm+XlY7DNNlspMP5qNptNb17PuILUanv12id4KoEJJS2Ll7LINTQ3tODtAgDOESC6H9h2Ba7oo50iRCj0xufIdTFb3kWzLybfCo9cmClgLAFaIs5gUW5ztYnQQuPa93y8T1KLnej+AAxYvosQxGX49rW5C1EW6ywl77RltiTP9oSN2iyntMZgFlz7g8pSXCpRB7qxZTGQ3kXh69dd8mGAS9fPVHxS4eijBrqBdrfr+hUaItzWHDTWHi9KzQJ42mfTG67/P2Z9e42ISgn7glCAwQNczBIBgIUbtJqDBK128SNsafkIzLJuggoXFeSkZ4KK/0s0eijBY5a2MYjYBXoxoRfT6c0tFibTmytqYpQihLug5kk+pv2xBHj8DXP6KqJFCYjqJIjqNAjkBi6ChZiW1hbcEEOujlpHsud/wntT4MbAJDVw5HMJ9mJVgscpSThTWPWogsdV1NtcptntKkfSqMtHdbW6fFxxykMFGlBDPegH+N8K47A1X1Qi9edpdanVlV5d6pU/vTH8W1Ud57JGG1FjlzCCxq84CEEhoRn9d6C/vh5kysheR6DbaR24WB98wLUv5ssaMgulJOYDmTXnl0OmqbhOtFmKKcuizvgSAggjGOHsVy30wrg4zzkjjlZ3dcjy9CA3tYB1m8gjhLG8n6v7aWX+lYfP3vUE5/Hl/lV1bE9ng5haquUDQin46DLDWvXe9Mzeh4t8GgMB4nS1hndADa0ss+8SFw46rHyqdmc7S3lQeAhcmq+LYXgMHsxAJwviHoLjjPJ1uX4A35NPM6grAZ+ipoNdE9uJU9Xg9/Du42/faBS+rByrqlLPNTk4AZm8WuRVLh/0w8pb4k6xBfyqV9K+tG6wiEXkof6LqC0Jqcck5j34MM8SzZobUJQma+s8pVoOOPZXXYHO8saOWkAcBwrv1U13O4qVk9QMFA5dnKDl9VE38s+H+yLtX+iYipYY1stLtcGgYP5cWohXSbNPC7QHVEQ/SAWRugxWxt22LlUsA7uN1tulr4iSCzVfLaVGSI3iMl9NTitItAP9uyn21TNr5y18O3vBK5bLXOb38rYod+iDi/ru4dPdaNEK9La5fAlK6jp8LYnPwbWL5fSVP7hSZd1I8hYy27I78ju+i7GkJUZnLz7Z/0ipMBUE/IQXHMjjFT+2959uZpZDj+F/ozADjX4Q6Ghnq/I1f0VASnrM5d9D094BpFu6MEKrZOvodGNlAZoq2/XEdUgioo+xFAMKLRKWGrpr1jaTLm0t+DuzVz0WucJ5R+lD1clXbOVaIP4dm/yedCGI5cVRuscsY/xiH3IlJwSpKAe2sGiIJE6a1YmqUktwyFNEBvJsn4Y1ATOm+b0MLE8nEbdmjDa+fR3taf60eIUnaKiClz2BFfOCyY2a3E2LA/4CphdLQQNxIGT3wWIkUpbl1rCvB7qiSJmgDC8EltSv0I/EDuFQ2PvMOFci+Yzo4O3E3gyjC9RY65FOWc0HbE606iZbLlaTMYYZHTbac1yNET2tncW0Sw9Dc/ttBAmpinzlfw5Hyr99HpXxZqmufNVj6fEsw/CyDqnOfVTto5PRDzdhZaA7dz+gZbcr98TrUVQ/M9lUJfCX+YyJfbqaf/lp9m9d018hptxvsRSD7vFw1D1YY1ZA5J+lHJiV+bP057lvwUd7yIC9k21ZoVf9LNXTWbHp4VDreb4PYGEc2DMv+IGE+biFlrKjyvnrm9nskpzovNzceNV+gSgtp7n8IPfRPJAGM11Gjo9Jpk0pK6XQo8zofqN2I3lmN5I9KOQSdmMixU3oEHLD6sR9R/I/CbEIUSZvkx+rPSNsHERGlP6egLVTBR+lIpEjj9BYnHBLl86dM3lF4fmqNl1Pu6ZL/92bUxc2e/9QodJndob3MnAaegOxCCYfTqDAxamCQyxVY+mZ7KDU0v7EH5dZQ5heO2lJc2vMQmFtBYvLtMl2q0qHznssfZ3ssdazKwrfHDB90mjwWbeq1hZfh5bBIZfJXIYlqnWJEitAA02F+RhAWW5/iIOAZ6EROzxB+Pb1FIfxcRH8YE68Du1ohodf4jiIZGh2d07/BS6m+LHCWKsU8NvxQeucuuJO0jMlbdw5/j154dYxAi40TDO1wQsJWVLTXYlb/IyROjQlyBfIcpKrbEdfrkApdlSSHCqVPAcOkxcJV5NUJ8904NsvNw7dQfSCVyF+z2Fn/xSTTm6xxHX0NHPsGM8bP/obKVxiqwH+8q05F7J958+e4gGix0dkI0d6o0dm4jsBjPGqx/Mnf2PRmpxc22doXQFdW6fVS0vD+9GENBPGSIV1d4SO1v2WXKYHAeJ3dlhoNoIZQ6euiUHlrlR7MDJSrSlDpCt/Oau+ba/0OFljDnIU0HELgs+zuNbWBnIedjqsPr65Jafka/5akqmbXrsfW0b8xBcXeAYyFvVhxVe4P3z31/jmK3cchrvLnugGKo8PX1964deXNnG5XvFK7micAY3sWVPBb+Es2oUF7Qw2l/7OS0p9AjDufnsHKc9Wu/0r8K01DllR2/Xbm/vxsdDVHbmhRNhgjICxKu3YQhSjhQbneCRswreLYRpoOQgrj/eQQoqPNLgnTtaGik8qKdFXDHJ4M4ffXsnb5bmdajefhZnRq0EZ2dzWpUE6QTK5kL7yCYzkDb8mIV+S8EaSDmfO6qwPouhi02zJkn0T9sH08io24CGMkesGfayRE0a0mdxH6FmzjzxUKhIDu4AnNjQAu2kV/AysIRpM8jjwYzziyXUJw2Nkxg5COxhmIJMpsKu9GDl3blsg2MXTNbNWEen/VzbxDjPdIKRPgVg3Co8J4U4afsdL/82ie9oIxax/Y9aup4RbigjMNTK5NdZahPFLXG2UE+gnpiKlEiNLBkd8n7b2OBpR1xYDmcs/J3W7WLERfmZvkfmVMeulqrlJ3CH8QJNJLO4j8ynH8NtqQzh+wKfKS8FWVxmV80ZxnJZJHPu3wBcReJtpGiOHUMcaezYR8OwOooIUp7c8aeuyH4ygkTzMc7lOki4lULuS+T13MmGL4wakuLBWYgB1fFSJJdynhnQ3B4fDzLfwjDAxCHKxA/nCZvR2dmJOTpGdHfvmzNwYu50dHL7+5sTMeC3SQvfMJPbHTMCVzhJM1534ZMmynVk+e3IaLI+YtFURHei2PALCgww/Z+qiosR0Qq6edSXN8mDNOPgQxOisbbnDhKshhhRAm4USQy6IS7eiYT50OAOR7p4TMmRLHAp8fjQtcet55jLeM0B4YTQJmIJjDMqyx6cGffy86NjUUrcCjgCV8iNoVG9AT+0ezczJZ8B5ZOK2yJUnx4/atXv2Zha8nQXXN7PlUNSO4bBlGuNWOSCDzjXoCSIpjqy7K1X4FIquiPhSkWtfw6wRjPnGBnWWoiH6DRIeCAT+vJKIDjf209XcZp84yVC8Tmxpd2q07ukOr4LS++F1y8EXOahPv80uPB/gzPtjVWaeiUFpbBdXdgWVXSEmwYs/Uk+zxRyN3X5J7SWnnboispPFY9TxqHaMWu3Ssd9Snmvt+WdqiLtKMt4PLCc7XyscnMVASs+Yw+ggty0r6xGyrRqjVnnoG/njqrHju8n/mDKxv1d52BiXosbsn3BaBJjLplm15N22jLAsJVrjgEcF8Wn+po4C3ErbDHpx6az1rWABbD4WTOVasOSYfKc4BnTiGN2eOMbPq6H7c/F/Lnr3rcVbAAA="
  },
  "scripts/asset_progress.py": {
    "sha256": "857ea80a06c3356a52731a958b2d668da100bcb0bb07c73506ed30806b222ac3",
    "gzip_base64": "H4sIAAAAAAAC/4VSwWrjMBC96ysGQcEOqZN69xRwDntoD0uhh96N6ow3IopkpHG7NPjfV5KdWCltV6fx8ObNvPfMOX/eW0OkcAedElLfEv4leEUrW9kIkkZDZ80fi84B7QXBm7EHB1KDNoQvxhzA9S8e0gSE6anrqeCcM3nsjCUgeUTG2A5bcPId60CfhSrfMPBPjn3YVnC3Ln8uFj/GfngWqbcaWn6KkNUFUZTtAA/yF/+covwvRbkp7jzF45niU9iI+e0xH1cAKodn9ACehLFGCW/AvVT4NPk1XhGU17XUkuo6c6jaJXR74XAJZEiopbeS0L4KVZX5fHcAFhMu1hM41ucJqL6guqbxh5GHhiSKo/GxGS2bLL+AOuvnspYDnCLdsIH1DeyElXCaM4sr8oEvoVW921fPtsecJRIbodRFYmN6TYkebd6+O8H7GyegqhK5YGwcvE10bKtrD+YVUQraBgONj2kNi4l0lXKGKOevGKTHXrPMhsw5eFdOE338MW4gS9wZ9Q6rpDVvyYf8g23ptjQjr5b9A9Srxy+UAwAA"
  },
  "scripts/audit_audio.py": {
    "sha256": "6a6e7ac3df0436319ea840de67a2dac6174313ef6b924021faff92e90e56f5de",
    "gzip_base64": "H4sIAAAAAAAC/5VZ+4/cthH+ff8KRkBh6aLT7tmtE6yzBa5uUgRIncA12h8WC4ErcfeY0yukdI9e93/vN0PqsQ877QLWSeRwyHl9M0MHQXDb5boVmczuVC7KOleFkBiqxVbtaqPEL+//fvNWZHX1oIzVdfVOVLWnK2qZ62ovaiMq9Sh0tVNGVZlKgiCY6bKpTSuk2TfSWDV+G/ncf/xq66p/L2V7N9uZuhQN3gq9FX7iF5rw77bbNqbOlLXDyPPw2nU6n80+/vzzJ7HiVWGa7nSh0jRKjLJ18aDCKMFpVNXa9c1mhrUJbZboyirThotY2NaEzGEugrauCzu3rSpT9SCLIIrc+Zx6/KYfbz99H0N9VV3pTBb638oLYeoHVUloo6e0d/L1n97G4tHoVqUsOlOarupJnBXSX+utm8IBaXc+SE9j1G+dNooHZ7NZrnbCyrKBmLaVrQ3dh42WM4EfLPG+7qoWRqt2usLO87prr+vdtZHVXvmlVjzq9g4TojVKtmRThQFlhLRCirKzJNv1bx0EbJ+FzeAYbGTawrFNM95mJcDF6lzhbdcVRWqxkD6G/fG+4HWNkvf4sOAKcWxX0kzi5uB5AirvoLyqP6MTiH56B3bOYxJtHduQyaORiH7jpl+vxM3RFBwacnZqGDwS44i8lHtMdSyT3Fq/0zDt5SjlU0iv8Ug/0kyEBGtmcMXPgaJX29eryX5/FjdeIXzCUZ/HVCvx5vU3b7+Z0/PbmZPOGaNQ1eAQPGFU25lKvATeZ5gwWLoFsQgGjaV+GeaGMcz7SRIUMyzvkVp7AlPSSraQ/Q2hNSpgPlV0RLY80rwqrBIf6kqd8PX6STvQpuy7kxP6yd9fsjMyawFi45q52xbHmOy/OOFUKWnSUf1TNuPoZU4HH6MMGj5ECXNikXdGEpMJeqx+kFg0hu5fVQagFW0tdsDalsOiVNJ2RpWK7FU3xEEWxTPpOrujyEXciq3R+V69ssAzQHIN/7iH6VlhIBlCF4hdw0sIixICc6DHALAJoCi7T6GmpmvD9RR2wmC3A9VWBRFc4vohwFMZUxt6ubaqUFkLUQEl8AIMyeUiGDUKirv6McX5jVY870hXJGuWVrJUsffOXdn2r9CVirM7WVXQa7zVrYUTGu8DvG+9oz8kS+BwnNQcbWLRqqd29ckgZiMfHGUpqxyCn0lVNmrvhKpq2yK5BRcE1FP+NFDKhiYWSxYUA62n6E0cbWYetwZTj0DVHwcxvQ6upfFrKbEwd5kRz9fBZnZOzSLv3rz2KsiWkv42WZkOg41u1PLGr+7hfuXycMJPCB5E0+mEks/2uVWf9wd/DK9RCEa5lJbUJkfa+GolAqSKFkcYBe25M+dH2YTDYkLzM5Q3UiOA/kko+T3pPtwFH2qRc0jkLp6W4oWMcAhOwK1UrcxlKxGf5OLroHfGzXoBh7i6upgx+1At4GoWiKGaAsoZo/EHDb1TdDkCMgXWtThM+1hfw6aFUE9wSu1iE6anDNrCZ0V2V9eWglMyYFzDSfeU+IdIhF/kOie+lBKR51UeDoUICG0QJfui3oaBhctSkZXmGhkAXqozm17NjaL6TOVzOpU0mkKaYyE6xrLp7149rwpZbnPJNdeSnwkpBXUSNFC2EAWCONYuhIb0TLSUnceTHyXoCaYwV+g/TykSwwiSqBaCYJ8OcokVnGWvKkUhngfHGdyblFm4ym120TfINUZ7wJ/Utq7vxeIb4cyII3dVnoh/dE0DsLy+pqprTmznbT0oD340ojU8oKtC/IuFc/rRE36sbAOM66vm2/lfPAmqJfhIKe+VkG2rqo7D/7qusGeDXbR6tKQ0yQVzDujJ2to8D24w2a2vYTE0KV9jN+gPNI732OBNfwzqIQsrgjPXOLLKAFEj1bGdvjqyE1X9a1Sqjqg2eq+RiNLsrqvuU0SKesISchTQsJ8MbNcBhhCLG+K4vonFYvOFqA++fyJdQ81ftC92cNADhouNBwRdsU2wy2b0WwSiS7v+YLBSyTKF50ec1JM9lpNq6+066L89sk6QTCPSoTTEaDjm+FADEziLRxE3TGcF7JAs+umF+G7c9TsUeYuT2Pi8qqQvqkRW6GZkAmnR5LwZ9ON0UpDaVsI5ye7VS6+h5eJNfkhv0xeWl2QyHUFbsDmkVqncjdMbRl6NFlS2K8h9x24mdJuwyuMzB1NPKutow9TqfQVnMyqIJrVz3ZlM9eHg9uQhmOckALwR/Am05UKyV6dbBH0D2YHg+oHSPocXUbjuLHRE7OuTnVI3G2x+zwDvHSC4VXN/Dqpa9ip/R1BJB2FwQV7A+UZVTCxCPR+1T84wcy8OJTH0gtTQTlzOyEcemqQNBuYpQtM6D54+h1wlj/Ih4OKbkZVqLue63BoQPRGgeij1E78eJgrmoEpk06gKSYpN6o4Ye8Fj3jIeD+et6VArKe8BfKHvwzmrxEicGgmnvvcVsKtGUHWQVxCLwT8DP+oQzKd9boxXODpokPOpfCK7xpw6KYlhcFrkThgfQSAXMtNJPpYNI1fMvxxOc2lA8UUdyHrDzVNLO1EuGu9CxK1PBPCLIQVAdjHJ34ISRMwWoJsGlzVYiTaBD4u+8YZRM9lZ5dgbLoPaGjBCFWWBNHIYYO5/MQuZ21nzHOZiwRh+jnaxG7gI+BO/RJlmcrbJRcqlY8/FvzToFkBf5aTJIaW4CbRaJdWxi2jOVy0Xihm4REt3FrTaSewZ9MPkCB4ylkeNWK+XAXovs0eKBo6R8k7Xsz7H1fC3XrfsE4cJygKEunJLINjH5Zh4xkA5ub9gmKJq/bPdY3S0wN3ZpWgAbt5+cdm06Rzruv5HvdkzG49W+Yjq2x2PhcseMvtxd1i6L+CXz5SdAefAdHpQLJl+UsM41k7pXmqKab4SQDfjIsgPHI72oPTLW68vXGNsqMxccHk23ezzpMuz8/uLnuPl0wuRzdkSOjzdDekqvEkWsbhZXF2F12/mr+HOtMRBMPFlgJne9gy7OondzRrVhcjRgQulF44hl6bJhV6chx3SO2CaqWuH4mccp5eV3i38JqN/rEZHIRlW9KCbrad0oHiziM5Ys+MkXUP9QHhqRc/E77Vin3Lv0TDqnHV15LinZ4tOnBWaW14+yToY2N5r5KucbEv43F8LQmt94/luWrH3qRqpRWojPsgP8x+rXXDSmhDErSchv+mTIm8eTcCQEtTaJ4uByjEYycaL4XAwdeDWJv5aw325Jf8nue+k3JBvc0oYZNLg3vrCZOhwz+vsmHIQGlo0uoXOdIu+xt33IBcVNXzK0w0dDV/+G75wcP8RkNyaPZeRv/BMmCubGc33WKs0zessTaPJykTmeSr9kjDgxg2Stc+NWlFB+EVa6OWcFgRULfkl/IcWWV9DUpnBp7UJ38zvJu+WUzQXlRyvJzcFkyKnZ0F36j0Lfj9l4aqaMHAe7/rOQHzN/5mR0OOP6MXv1NN6+e0mOi53LjepQwXApb+uTr1vDJXGoCcJd8F7AhMhbaHFCxGtX11K2a82B1F2hRyIjhL3q81yf7DLIDpthph2GiPHoerOEAjxEZHIhV6rc8AhJFNldy+rd8DboivFVlspSoWEqe/xZpUbVvcW0rn/pDpqagzHNvWdXz4AURYSzKg6AtxwjxQGH2//FcRYtO7T2iaKw+CHn36+/TTs7TZ1VBey2iaKLuQQr3MhXnjXg7tEX73w5utXk3QClSZv9wfgnhYoCdHZXt/EN5uB9AvX27T09R+w1APXsOYs2cGqQXTZIn+j3NUQ5Fa/ym7pBT1LzZs4+I+f65PzRjj1X4BfH65uj58kbCcrsL4MYEfEn2TVyL6utnreVXmHxio5OefEf4xs4EDiXok7aXUBM+87PEqZ8D0PHDTl2+Y05SuoNCU4TFN/A+WwcfZfhwMaGiIdAAA="
  },
  "scripts/check_playback.py": {
    "sha256": "5f8b4c24f11d03cabd5d059b3c63899958d88503e50f93755a03878bbbecfe84",
    "gzip_base64": "H4sIAAAAAAAC/31W32/bNhB+91/B5YVyJst2unWGAT20XYsBxdCgCPYSBMRZOkmcaVIjKTtu0f99R0qynSZtHiKLvPvu13d3urq6+hu2yJQ5zBTuUc1dt5ltwDnWKjhuoNiywuxasNIZ7VhlzY4VUDRYMuhKaVJ2kL4xnWc7U6JiUldoUReYXV1dTeSuNdYzsDUhOBzf/yWsSYRqwTdKbthwcUuvoxA50lpToHOnk+PpZ9fJcjL5/OnTXR50EiEqqVCIaWbRGbXHZJqRSdTe3S8fJqSZBVOZ1A6tTxap8zYJ6nPujVFu7jzuBO5B8em0dy2GNzr2+c3d+7QAbbQsQMkvmFqEUrTFbojDmj1qoLhHDdfAze+v04OVHsU5YPIkWIpGR1GL/3XSYjzspfAxXIhaSV80okIsYyUGeQUenRfnukwmkxIrtqNKitbiXuLBJbbTKdWl7fx0PWH0RwW5peyg3SPzDZLZoVQpA+9RdwTLlitWvqUDXQ51D3I7Q4lhpmIncoCPEA52yGqQOlY7GCmg9Z3FPEScKQOlS4Incz5czIdnFgR4KBel0eOjTyjvAcDCIX+iQJFBp7yoCM1nB9jzKCer0VZWo0+48+A7x6e/5KNmyZmxQx0SgqWr4eaePwEVvQx/6PPUeyEdsn9AdfjeWmMT/q4n/aBIz4L4blmfYVY0oGuy2MfQH2a7bSltMtAwv7MdpvgoqXJmm38A5XCIeKxD3uvN+enkHO4l90Iw6bl6FLn1+au07Cx4aXS+XKShJvki+6O3EN+Wi+vrZLZczW8W0xcgT3CjE8RJ9GK5KjfRi+fw4V+PdG7VjEqX3F8SOuFVtWux5tOUz7RxvpSa009C5BhSG17iieSxK0+eBAVPF8sFT0+F4TOoeFrxRtZNS0zMq3y1WLf5Tbo3qtth/jW49S3gge0BQ+8GLCjo9CbcFGugBzWvcMvXCi/gg/yYgNHEaiFiLmIapg8p8aDYxnJORyaCPiYKdTIOhRFDU39MiXc315T94AeriJLhlAYlS74rdPos5z/wYfozot4+n9uMfKt941gpKzI4sPQ8mpIz8cKA6Xsz/cqd6WyBY3usL1op5ZF1QmrxVjjqBV06vn6V8pElfE00OVftFKgI5eFroiaV/9FbiAdiY3wjwlQjlHBwjlw0X/h6dYmljUe+5n9KqAOfZMGMVseM/UUas6DSTyx07ADaU8/GiUVczNhHxJZady9pTvd8YZ0eejdjd42kFBlSJBP9SGeuRZqqljLnPKWGIgtDhfIE0rIaNfbhZvzb2Mw0YPQwAU5zWepkqFncgzYfN2L2xtbkhfa38Twp0RVWtrHPhChNQSvtQi+DshQwqCR8NqN246k/thjX4E9FyaVnonTv8kE+PoKGS4ZQOh38jD19XquR7sMpQxpiz9dRcjkFewj6/QJE+GyIEGHgJ3z84hD817Dfs/DvN1rkDT7er1cPPeoPl1wfu5Uh2g9G0XBe85fu3tJuk7regmZP2i9lT7svZS81H7FH16Q6sKel+gcsWofooc34E1PvzAZY+DBhR5IK2xLYDhU1q2QNjYq2MRrZRiogdcp+KeGSxsTLIxDzIs+lS9mmC1434KTqSV1JDeo7o3eyhC2DEsZvMSdTYnnZEVzY9NAFr7sNNCcsRY4FlMmEKiNEmE9C5DkXIlBXCL7uKTz5H6ueHRgvCgAA"
  },
  "scripts/check_source_roundtrip.py": {
    "sha256": "0e3524dbbcceb34eebf1de4ca449b6a8cfed7cb691e4545fc7686dc9ec6afddb",
    "gzip_base64": "H4sIAAAAAAAC/8VZbVPcOBL+zq9QtF/sXeMBkt27mpSvihDYpJYABSS3VxSlEmPNjIJH9kryMByV/37devF4mAmB/XJUAbbU6m716yOZUnopjCWN5Kom2/8iB2efB5+O/ySmbvVIEKFGdSm0walRPZvVCilIKdxwRu6kndatJVXNS6km5L28zCmlW3LW1NoSricN10bE96+mVvG5NltjXc9Iw+20kjckDJ/BayQx9yY+tq0st7bOT08vSeFoEsbGshKMpbkWpq7mIklzECaUNVe711uwNkfWuVRGaJvsZMbqxDEYEGrrujIDY8WMiTmvaJp6ZXhbyjqqMgKbKDnilfyvCLrqei4UV2CZSDMVo1vWNsBc8Flmpnzv19+yOy2tYG63bp1YIDGbVNKOpmwsRHnDR7eRR8Ut+ICBfUF/2S3SrbJyJphQc6lrNYOdxRUbpvwaI2zbsDD9mLq5B2cF5l5fBhsaCWN6lKyqJxMBtt4qxRg2LNCmCUxk4Oemtelwi8APOPmS3wpip4IYDrLeEams0GBM0hpRwpub6/Z606qyEsuI4aULmAmXykUMMgVhoEWBdssxokyCgtFfS9vkOEvR6bxkVixsAr7DxT5ii7AgKF4O/HB+x+dehhwHMflE2IQay21raPqqoD6oS0pqTbwfE78YJv2SK+oHmJ+m194WTnUujSBfeNWKQ61rndAT2DnhirRqNOUKTIoJ1FTCwlPIH7LcFvWb8DbOZ7el1EmI5uJStyITCwkxUt8WR7wywlP3AzToGryEJujtHGKfg2FfZ2WruZW1KnZ3MjR9ses5zYTlJbe8eIgWGXYmpNmjbQ+DdTaKSrPOJv6HRpl0CEKp04RJxd4xA1ZQJYh6vbaGG4hk81hgzF5YNpYTM/BU2zfciBAWa4xUbQXs5cKXM40SIfTbESpEalXdZwRICLAwZCKU8KrmrspxVRKshRoeMGU4ug9qIcQ3KeV4LNA95Pzk95BNJqffnPxl+vdsFKLO6ZlFe6ch7m2rVeeEkHs9ZZM5FxmWCQiHZQJ+Nr38Q4XHkDT29R6xUBiXQTaGeBaQdfC4WtW7vAvZr9pZc0+4IarpD9taj6ZuwFUOUIVp4Ta/LINeFnOCY5oFfXNVytmr4jWmVRwCfzbiauf6VbGLw+gB1eTSjKUCyyVxpzmvqiR9KskuVjYUDOmqDxQiWQ4URgryjKJDnqF9vc7JnQTv3vWkYIXym4beEfbJZkjaI+q5DQySB15+FZqJOWMG5vmohuc0TXMfQ0Bu+WgKDWvUtPDX+Q3++zVeQ8glAZ6bFX3TJmAlbiAsrJy0dWu41vw+misr7X0jCiAJcZBmfm0aPRJ5Bvvvgv33nrLu4aIRI6xX0Ci1qGP4rGYRXYnhKCLE8F2tb4VOHreOYyjuLvvIl/1DcifkZGrN21AOxWY5xjumViIikRDgXRjXJg8NMW8bSCSRPNwO5y7+b7M5dqQNfRPMDuExM0mKFrpFqoR+Ojt+t3/wx+HJeyh+8HJwenL08ff3H8/h9cMR+/D5HTvYP/hw6F//PLwMr+m39JEmV5H+9Ojo+OPJIb0u6C5dI7o83z+5ODo9/3R4frGB9LkJ2h9wUKY/ClZVJWIm5GDGy4zGWgY1TZsco7zq0MB+a+uQWaf1TQVAZ7nGUTJfgCP9XGg5vt/6XivfXAo3dPLvt+iuJfV69OYu9LdadqgmRvHGTGsEd75xgzA7lSZ0Z4hxQ3iFat87rEM7tZ/dp5bqrXa6J9X7hDYnM67kGDF70C5IjxC0iMIBqJa1HuwfHG5fWNFs7+a/+mBahaxJh10pH4nADFMGs087HNd34o+3tuLPK4oBZ/r7kuPA+goQl7FSeXBwnTtkYDDJE+p0bGqAlGYA9XVAH5VeH2qd7qDRRp6ZH/WbajSwAyuKWcutKzyGf+XDfufMiOWq4XiMGRx/GvjEQE7ZuGrN1CExzwyUKtbzw5d+iFIL3oM+lOCZo6fk430BYKlqwG/uKGMYVkQvI7d1Uoq5BEBLoUfQUNtDfwnlPcejS2gXiBsyAC+iMGPnASf6OwjNM6OBD814dcfvDdsrexustZyAHatiradt7EEoP78MCexKT7GsQjmYQxlsCyY/F74HJqhs9uafOzs7aRKFpXmrzF+tEIBnd2JfWWkCxYOHWFbfLyMiePYwoAB0KJxXfVvEp4iEYGLdk46b2yNkVgu+MADco/QX4YGmxj4pa10gJPCRkTgDpLlXhpUA4lfW+PGiWxoQAnSlZ2CEDfa5wnBhISqhg2wGkf3lvgytrnqgQV8HFOiwArUDzPDYIc06ClvSoVcuEtgyWYUiv71J15D52g/18Q5g3QU8dSzgLUbpt44BRLhD2I9CvLcF5haFaH/aQoBC80sfh5lpb3xmHB2f7l9CqEDEclvQf+9/ob14+CG69y/p9+ITzxRdUXhZoG6+1YjM0iWhv82Asz2U/hwaBZy9ddfXRwI7wgc/uizMfnZWLQCvaocSZotuNvAoVhf3AjEQ5FgZ4XdtvF/O1ifXy9s6jTQcqiyzf5Uzfwru9xQ8Q0RKhpCfwUYYKLKG2l1fPfdIMJ7VoV/MhXMM9oVW8TmXFUp766qJRuhktWwQ5rrKjsdGYqBqg+Pi6WMMxxW86+jFymyR+9Nj/riq/EQOgNyd3/jIQt1x6BdB/kAttYms3ZXKzb0jj8WSNLIRFSiTv7xUhbLTGSzYKqQFC0Uo1K6/W4mQ6csr0eqq/08l8qUHU2vNQm4OhHondTCKDjFRf1ijept7ukatWOHlNaoR/LaY8UXi7QHbxxfs3DfG9Wo4jGY/b5xdwJQDgAt3YlpVDLIb8KjBs2wny90jzaRKdvJ/ZPA7QOHuNIUPBA4UgsBwR9/dQeGa3Z0M/92FI+n27vXA7dNpcPdMDX4ieBXCEX5rvPgIAtwFDmr3ligBiJE0cOrkjy+BbE14A5pqwPRyAqmdr2uK5Tu+/Px6DzHLwP3rSFeu4jbfwT0eZRjLQvnJTlR88Ndz+KdX1PGuBC971s2ymt6btUlw7S90JfDWJntKpc/TKrS+cOT2p7WCdpedNPNuCZylmrCOSbYcWxfl6ljoCYyB/xCJM7bWVt8LNeH6lqshWTNsRvqooD/ez8S+IzZ13SDoqK6Aehg9uUYqFiPRWPIOzkKH7tHFoCECm8zwB/Yac6z9NHPExZg+YJYn7i2F7aN3GPs2JA9u6Fsv111D88c2bAtVDxg/H6r465oZ+CS2CvfhRhfxE06+ryctXpicuXE4nZgRNETnMsbKehQ949flvCwZD0sSur2tW5DmKhd+vXmSFFR9LqlYiFFrBRxhXBoU1EC8AEQAn9AnF/qLqZ6YbCqqZrnbi89nZ+eHFxfhQKMnpgic3D/kZZLu3I9vuWdJpHFY5KRWoucIfw/Wo1t+tepnEt6h+WNPqwpHDQ9L0k4WfuVwhXXt21HS/5DgWcDzBhb4EcaxiEd64GkoVoJYnzrUw+gv+P0txz9voPNPxeJquLt3Hc/Va9+Itl6QNVGf4MpennQfo5Ir7KKrn7HAbg77fu9bILTopZeXTTi9zoSaFxsvAjN3Ou8nNJhn7ZR5drx/Qk5Pjv8Ty83yvshI3rwlXVQSYPuVV1xBberuG1YrzNYWbD9md1FQxjAFGaNDn4pb/wMaiPa7oR0AAA=="
  },
  "scripts/colab_weight_cache.py": {
    "sha256": "b47a37a647673854b6d5240e8f943bf50e8fca88e69cbf484fbb2deca986f73d",
    "gzip_base64": "H4sIAAAAAAAC/81azXfbNhK/669g3APImKLsbJODVPY9N3W2OTTxy0f34PjxQSQksaZIFiBta9387zszAEiQkh2np/XBJonBYDCfvwF8dHT0+uLztCqLnddIXqqVkF618up2WeRqI+T0Rsh8lYvMO3t9Pv3YiNrjSolGeUvR3ApReq+rgi+9P35X0dHR0STf1pVsPC7XNZdKTFay2nppVTbirinypWfGVVvXUihl6TdcbWDYvuaVffpTVaV9rpRmV/Nm4/C6gNfQu2iluKhUfoevdobatE1e2LeGy1VeiO5VbGt611xpW0ktqzUKZpm/AYIL803TbatMFIlRgt2unkuCkMJ2k8mH9+8/eTFJ5ycJLpQkQQR8quJG+EEE+hFloy5Prya/nZ/9ev4BiFnK041ItrzMV0I1Ee6eTSaTTKw8+zHJeMN9XCuYTzz4wceYlqGPCxyPcWZUVDxT9BXW5VmCRvCDYCFFWslMxfdficGqkp7+5OWlh9MvGcqr2JVeAX9KvhWxprpkGciRl7zJQbwrYFfA442IBzbwcUbQzc9XnqWLcpXwJeihbUARHqzOoojh2h0FKAe0iyNfvtAIMsP3smroOVINktzmsBADnaXXdZWDOmeMGBI5MaSd9tvAH8lzJbw/eNGKcykr6bO35Q0v8gxnZm1d5ClvhDaq5251tButC5X/V7Crn068TouXjBfrSoJoW3ZFEoMkPlMb/uLlKxaydd5Ml0W1nMKXUxY8VTgtDy43y/I1iOX1y/SSmR1fogaujMEmeqBppTZvaIhCE3aRFs3xlOWuEQpcJdqIO72YHxg/zMsb8NtK7nxZVU3Y+aXZR9Ix1/Piod921Iu2Vg2stdWui7yc6JgxWCSr5Mxmnelp9JJpdwciECA+WWxzpfJyHV9edW6Muw57XzaiRDnEOmyn1zTFTB+1vhUm3HNa0gk4LAYEOCsvM/0J/A90An8StEgcD91hbqQ8Hn3vOItCibnZQMTrGnbrBExTNbyIVbv1pZ2po9Td1A36CBrJNe896/Rt/G2u7RAyYqoty+b0EjIjpv4Kv2kzzEofDjyTGXEHLKaWtBvVmcPuLWRpta0L0YieO0aEHR6uACMwzi6IZypmSO/limKIghxUohMshiiE5MITd5iBZzADfFKYQaH6Cubp/auIfTUenOUqrYCQnC5ZciV6Lw7Rv6AISpHF76pSGJ+B0vYmB9OLuxzTwdoWQcw/VduA9v9qc4kDrz//ehaCm3g8xUwGApU3mBq4LhymYlCtRMa4eh8BJIwTBosU/C2HwBEqhvV8bWr0Siuk1Q7K6rh3twddF+xrEIG6gGWrYPfOOgOHt8To9VkujdP3Yd8RhF0wXz7gSFc/n8yNY3azuljVBZBKTp42ERbXa7FTvk8WwV+mRgZO4OLEVVVkgkLB1wSh/jNjqmplKlj3/hHj/t+iFJKj1aa/owkUG3qd8/NcT4zWkJ19xlORGOiSaM7JcxaEQyKos+str5O0zXjyfGYkCEZZHfTqa7EPZ7agU/a8N3nEs8zM6k1lop2X12DdS98xyxPMEdaBVj0qr1/oasgTaS2lfqa0Q8O4E/z288mV9UV0Pz1obY3OqL1bF4A7YnmQH7j4pspT8O/L+tEF4xiZdYsWovTNzOBZfDrfL5sfBYQ4L0zZJGClYIW2zBYefkF8lvz29t0nyLYeCAywd+6xYxYCGvkTsASUqtqHqhDaZYaJ1ny9BD1MJmkBq3ivMUOJ7AMUUCG1/THdJEle5k2S+EoUq9Azhcakccw3Gl266QZ/kDoyxPRsJtBzN8kQmDH7eUFEKWwW6uSQI5Z8IeNR5Q9cROOglzi2sIUqludMO/UPwqHh5CHUmTsSRG2NvuevjnDcux/Wya9fTo4iKAEQr+jynS4RnFg9olNMTx2NEfB11BZpaqALHIUcx+g7SGu+DuWhAXdjA30P0u1gxO9XcMFYh7n6PazAHdSG6B3h7UrE4VnsWLxDABgdjsAONBvR688uej+MLD+Zjg/riAkTW2ShPm95k24wJIa8XehvMaEuwgYQcpluoPLtIUMaNYMHQV9IHw3Fo6VqAGIfx5lE32FNXPFwBl78A8wFNjPSRgQMEGJqLWPbeE6fjKp/4el1C71zgT650zhCLby8QY+ibC1vIIGithvZ6SD4fmD7NFSLPlw7sBZrwgjSPhtD2m840/si8/743fiRgXezWwnlkUIVt4ZiHLPIO/NeX3ymPZSlSNGVwFd2UC0AJZXi1oPGvNkBs8j0NVbJpvBur7FSmv45/iRbEZI+k+qa3jrj6PY/Arx3nbSKr4U/5BQA6BDiJ7L88enJix+fP3/Rb1Nv8QOEY77tezHVrgB85ohcnFBATHntqRogA5mr2QjvVuTrTWOFN1vBxjYHP7OCIIBMUC+daEZJhtBM21BNie+Zggjd8gRKm8IInJ8C8EZQAzCf+nb9chV+XxtgMDsmRuNZgT4a+ME7v0uLViGSNRJ5SLwAS604PGReWtU7sBpIBAYEsA9VlnsW9GcAcNHxI93WcLkWTWwYRRV0PT67W0LLDgudYRhMO4BtV8OES8xhX8QQtIvg/RaKTCNKw1jueruhSs1SoTny0UvhU7X8MzZjqKqY3f4Ny3OFE4YeXvMdnp/EPh2lZO22Vr42Q6gg2SWIVbW7HbMvJSA4W64WebmqYrvyJy7fwruvz3j0YES1C7VtFgkGK8NUhH4Umkge5lX0C9rr7ftuwnAGuhy0J+Iu9J1EEWCmEGW7Rfgr/FHOCE9HGFU3DnnZ+Cubri7vievX2f3ANa68e1zmKwtX4BsbJ+gGmPeQGpiOCjXT2cDVxzDd6AGy0Un16scf99iTmR9Mdsa55FJbV8Py+UHcL3VsDdCbrydYXOWeAna5/Lirliwcyh4EBxfas6teGc/j8G9kgIGeLO5SUTfeL9AGndMjhPvIx+0hqv/+I6WnYG4jqy2LvLx2ABrlsu+tWb86+Q2PKDjihKylNldX/IUT6E7AdvVsmPSgiLuZbgCnsQXswgvqUKtiptcALh1qcGtjqHtuaG/WdRu/4QBPLRwxhwH/l3hk8QQsMsAsTyx3Rmmj07khDDpYEu06phhqPpejw56rf1IehyXRGMUgTWVcQwlR6qONhSWIT+AR1JyZ1uUH7yMJSBgoAz7rtuDS24rtEqqgh3cXC1MjoFOXPG3ohE7NMAag5aPD/bLz+olbI/riYL2Eyb+fH6gIZrUYkqf0YSRYrHIJVi7xSN0MhtTBuTCLaDAi3lGDKfUHKvHPYnPqbw619UiuekCmv1Cz/erly3+9+hYAewdcOCUFihq8sLEYhLrfERJxYIVzWYA5ymiRJCEhAt1JBYO96bkRVFJ/jEqwI8cduCQamMDIAKfsUY1QC9JTOH1r8/8ZgC0vq4Ru1KiJITjWyFahWtJW0rmTXcpRB7qptiVWTmPV4cp0DaJHyIygmS3AETzdyu+c6haMj39IGGeee3/Rz+ruLsytgckp7pUGRox1GsPQes1+fdtX1Gfw2BpAt8hC566D+JVTG1tD6GqWGe1JixYP7hvGmzbyHWwn9kU7GyxHXQMkkdUKo5yi2JrwgOkcePkgJlhogu9rI7oDewNMDXrsiud4y5Zuru8BfQM2DSKbfBN4jANQKyM4jEQ+mFPvJ0MRvPOsJJe7uDuj64ZcDH1oQwcxze0GBDI70VmCCgX+Qjii1N4suvk4zIuSsrmVjd5htv9k5cV9+zrKrAVMo8RCqjl3cd8zQQmW8cDYoe4eDEzA7F61Td0284dOgh1NEQrQ5NGwkT6sDPSdh/nSsWhRpdfxYaU9OtHkEZo/X8LM60fJjdDYKgmfJu2zP4Q9R0t2qjh82TCmHsbHQ+DyjM4JxrDSYIBFf9GCvrAHKPc2qkCNdQFQw++EDbUgPa44jk8PO6NGG+PhxwH4P1PPA7C947AP3PcBfH+aC90v3hD0PklniKJ8RoDKdmvzQzfKFraPMv3C6/7dg44E4DsGd8PzEj/RNRa8yh0bXTpq1MjmzCqbhczCjfkAtT94MDG8CbR87HWifQd8Rtbqv+Pb6FTjqYd4yKzvIdicsgPeEuZ4ao96TZI4ZkmyBQUkCbP/byHBGWP7Ly7RmVxDi102F/Tdz4RKZU7uEidJVqVJEjjz0GQJN1PwjolOUsPuFoR1CB5UqDsgFlp14Inrg6ymU4T8LGx2te5nIOmteFs0MV51PDqxK6gPTJ6xtCpX+VrNNHif0u0c/XfKo3ytC/Rs7YHeWsVmDv3BWcp3TlXXKtKqAf33GkE3QJn6LocoTScHT8N+Z1hqOr4GZhhYPjdyCB0bndA4btwjG/x/BUnga9PsC2tMpq9LjNmCsZi22Tggsz6Bcc6c9IIhHsSUTfwCCvr/ABCsMri1JQAA"
  },
  "scripts/compare_bass_tone.py": {
    "sha256": "cc92b244e3f2875518601576e124c0ba4cd9ae0006228b4247e49e3f1d10e9bc",
    "gzip_base64": "H4sIAAAAAAAC/+0b227jxvVdXzHhS6gtTcnrbgNoqwKO4w0W3V272U2LVjWIMTmSWPNWXmwrWwP9h/5hv6TnzJ03WTaaNEXrB4ucy5mZc78MHce5yBjJaN2UNCEsYWFdxuHRNa0qEtKijvMMfrMojmjNCN3QOKtqQkmYp0XCahaRs++/OSVvLk9eEpjEkjhj/mTynu7INSNNkeQ0gkE0yWGZOKtzQjPC7uOqjrMNOcsTek0qVlWwzoIDpSVzfd/3JmuYWp+8DO7y8oaVS1wh+MPFd789/25KStbAJBLXFbllZbyOYYkqb8qQzcomq+OUze5YvNnWlT9xHGcSp0VewrbLDcCvmHoP82Knnv9S5Zl6Tmm9nazLPCUFPCXxNZEdl9ghn6ttU8eJfttV6rFp4kg9/xAX6zhhk8n5u/OzT9+9PQu+Pv34MTg7vfz09uIDWRJ3QuDPeVsBJhCba0ALK6t6lzBNDcKpsWnimpaEhhxLGZwxq31ySqJy52kCFkkT3gCYGtHtCNh3cb0lYcJomewABXUcNmItWtc0vKk8OAruNcuBwiUsShG3QHNCmyi+TpCMBSvJlpZpnsVh5SvIp6SKkQ1I2lRxCMvjRpEDgNLAI1leprLxSO6+ZBsgPQBb50mS3yEP1FtGiphm+ZeVgitWgmMBJXEI7iXaZTTFxZ3JtIvOy+/OP55/Amx+FvOrcMtSGgBrIF85C3LsESeO4MFpcbgDzWkesQR7aMhgZ8XR7fEr7GSOJ4BF7DYOGY4Im4jilKJkYSwhO5JNsR1Fo2yQLtihFqgYiypoWM2vJETARUSzkAUV4Awhf+XP+fQ1Kxlvh33glFfYnK9xiSio8yAsGmj9VDZMQpISCo2D/OVNHiaTScTWIGJ4bhYFsPeK1W6Z5/V0IYA4zvcV42RQo4jEHDyJ8a95N5dhLdJbliBXsPQaziem05SRW5o0DKjkCFqiAAFhUHDEomQG286zdbypZi1i+CiBchJfFKZhk4/rVS4C8kuGiGD3tTudknjNoftxFaCMuVMQGDgICrUfMVbggzvEKFO+CE4X63yxJEPDBHrwr6QxAP49nuy8LPPSdc6VaH6NoinhRPEaKFgRrjna+DyG5xpVXvVajC5vQUhqztmgylBY2J1CuyM2WDKQ6UwCl3SUBJdK0W3rSE91Lwe5wdD7AhYq44iRPAOVIHZaxjSJf4C9Kq1/zdZ5KfhC62saQScQHQiIrB6C/gWFSlND7+u8ySJa7oB4DuhEH1TuLTyrx9Xx4spR6AeFQ0COABRKQ+8wsMKU5KAqWu1+CCvUrlpnitQ73keqS6QQnsKyIBaJmhDkvlo3iTBkINo5YGdHQpYkznRkqxJHZo/YLxt9aIsLlzcnLFNjp+Q3S/Lq+OW+rX4vOAHELC3qnaYEV8+A9XoLthNAkBAUJAXGKiu5w1wRdEnWX375pbZ3QqX5WrMQWmFjoBsmF2W8iTOaXHIywvxWt/8ty1hJcRdiwCRMEJ2K/ZH7RYfbBiR5DVk2AHBxHQQuSMPaIy+ADcDkvHhxc4dPU4MP0bDSSu0KFbp8+aJ80OOqBvSOO/U14C7Iyf4zANj+/ieINkvqOkxXsiIBqJrtPIPyXxDnz5kDP6bveCrFtWR/beKSBevi5CVnhKZk8sSg+hPcygofwByWvAUNpxznb0BPO3wYmJHVFVd4+Co6rsF0syxypmQJolbs6rwMt8EtZc6VYlq3BQlYt26AYVBgHNkTOYQvbA3j9jCI6l3B5FgO2Vd2TpOB893wZGEx5UpoNI2IwGm6EGi2c825xlcmEkcDxxFEw+O0EQmrTad79fh9ITT0kD/LTzOLQCNETB/UUbTdCLZi3Kx5qKbRd42AM5q6aKDlhQeKk4VNzZZvKFgmr8NUyw8g6EYpSzbtWOK2oYmAmcIadDYeE62xWpUcz//593+czI8q2C3YlDCJC62ThzcIvJfSwk1oeh1RUiyEjS6mwOvgjN6CPfXIyNmmQuugUkcLvFrMUVZXoPj4BDTxVQhKsAb6cB2pm+s8T6oZKKU0YOApOFPBrFwdo7eZK08bYg70NblNMkOKEsQuQxWsxp1dvH//9tNHj9yVcc0C7sXr4RALGHjgE0bBX/JrOE5CrUEpDTlBgDh3ajRMDIAjUjDPLWgYWQQsu43LPEM/zxrf7TLzwHo3RSCHdGeA4G7tLaO4BCVT+lrtfsvCmwD7LLA1eEMpOHQ5GjB750m+2bBo0nalBj1AqfH+2qCdiGAUosY15EanGH1Y6cuKqSvp0l5p66ghrBxAMPSArwujeEAWVFv68tWvQJuDoFoDMSDJgs6YfaL6cVgkGJdgYeE5TMlHYOlQEmJuQU+lnYwaYQjIcmTPaoBRopbVV50eqPo80XZ/cIQLsa4U+KkemQp3dY12ywzW3cdz8uul2SM8n8wPd0JR5BW5wUQiN1RGP2Cv9ExhGQjDT+ZE6ArtQ3DR9tMbUDIuTsrqasljDeH/BfmN0GOKazi/gYlWqhhiHmQfVMMiSIKdQKN4gJamkCyLkhVjlxTeFWczGR2Zvw4DLR7lHlhD02+h8dgDq9kCPNph4AMD+rvDcBnPPMKX6tiopdG35xqZvJeBMjrQGAhnIYTkYZkj/bZNdgNyXHFe2DTglmQ1CJrvCNfHKDhXKnDQpxvt24jwyZN0ETSSKgy53Sg0o9P5/npmoQXVMTZM/rYh/wLUvnN0lNL7I5CiI8VRQImTOf4/OlLB7pEIdoU5UIqkEwlfTbtYJghCWt4jaYklDLNdbZmF0NblzkgNSLDc+KIFWqBphamGCLxGxr1No2bdtn52pcWzoLZtOZINzfmit3+j1XXaQOnqN+8uTj/pmKQ/s73Cct94Q+mVfMR0AyAvdLzHgkZFDO13g81+oaBAvHZlVJA2Lq7sB/bIbpcD5k/YF0+glKuRqU0TZPFBuhQlKE4I2t6dfiAXH979cTHiCnkAQjhoRHv6Dircptp2lrO8ekF13YO2o+SJtHauYVDAZma0kLVpKx3R5jg9su+rKr9R+N4YIZrhyhZ1o9pHnFfUMcaB7ZsFx+wO4MNZ+ysCu+gx6zyJOMMNomE+nwenAXoEAfoBcxMRAB81CaLSOFuugOUR7nhxh9dGlJwhhQfxIVpkKDAYRwCkEd9CSPSoS3GIERVLzuIMzj2TuxMZ5lAY5TWNEyCdOYXK63CXAsejeySanKsWp4XCxfpsZe16oifzhdCFqb9+orCvH7mOHEkd1rS6CXg8BXYqYZvcOcxEaqgqn0kTlXckDvjf1wkLRHI3/kHBkiGOU2/j7AZOr5uGYTcVA4NUBwYXCoDqSRlsf6AdXAwwjhumuh5shsJYUqF/5YjclHPF2emGiWQRT1DyAApaPPkaZ5pEPpjZtHLtuHGYc07DugFTbkTD8ILIBA4kAod8BIuZRGKUe+vqDKLJYiXMAvA2qVu4L85PBjLMY2Sr95rW4TaoIIgSQ467AxCr4GhEecqlGUZJB4Tj9kAU4MwZX+oAFPQPLcz3qAqW/TOVadinfAcTLrq3ycItcM9zIx2VVtFgRvzQPdqoJwxADgvcY9HTc2AOerM2uMEBj3L/pYosuK4kCrGgUjBR3OZzy5+0yW4ifGkmgNpah2KKANP6qBMNL6iKQZBw79m/o7dALaXHlib+wkLlcu5/5UHQdR/oASdzS3PzAp6PZQJeP9B7cNL4XkIeWFl12kyHiPSbAmukrjD2S8vOe1oqluoB9iyc2UAy6VL+DqtL2CziQxBn2cKRCX4ESpCI1uHj+6AuKXi1su2V2Da7D1lRI3XYOX9E9QUhM0PiLh45mbSBnhi9XDuf0ca4/A3TshkFXzB4WJDPvOnBRhXykchhYLI4sTz1Z8Q3vLIpS6z+n+LiDZJxgGTcWZGeBQxHOb9HvxhcplLUnpcaytvL4JvzN+9OP51/M+Xp8jLcxreWJkTLgSdEm+EObHCER/sM5I2othEWsH0wWTHXS+5xz6QXI7VmW6rl2XyOeoM4PJ3H/7cKUdJ5loUoWaoXwaROWHn6EsDz8qCPgntC1nIPlJ84gWnLJlD/oHymkOsn5DX/VzKauSw0dfwFRWLuMbSWHHUYZMoHAx0Jc+XoNJBVRlG9jwZz7YHdfBef0U55dUshakdfYJjQv3iw566CFTeMWBAycvcAp9pBgxUfnMwfpo+nZLt1VFOFQPMyw32R+ezszbfkK1VjVcwAtHUMLbTjZsjRsZLKExsN3G0ueFro/mOE7X2Mfa32lzYV3j+C0ArMSDt+5zgCgBIzj0Xs6siBDttHkDAcuA8F7R2QvegdheI5obs9wrosI0Zxz7svEAOuDg43LKJvp5l+e6KOxduzTPv0IHoJeLPx8KZ3jwCVlyShcNh5hkBx/oxnyLkHPRPd3CdQ6BVHcUXPtLP1TmSwJycyOmnvmdV++/dZeodUmTdxTlH+BRsTr4VePUA0W65MWy5BSVaA8EpDbIc8gKbOgnBiiThlrtVNIwGI68qeJ9THwHueV+RziAauIpwqRvohOjgO1LHMXZEhJjO5iNb9huXwva2DOJImiKsd4Tch622s72u+hq3mubzSpa0lR7m62Smop5WvSn0Ehbog8fnR3NHJ/Hm5qj3pqoF808PESuzsR+yTkjzypAO5nj0YPyTTs47vlQoHP6bOw1zdItJKFXWZpaWDTuyrbI+Kf4V9Tq2YEwBjD9nvJEhfYiUbrg6/oGiTqnf70DZEnc6rhx+jkPjksqFGrLgtkZc7mIM+vOqYev3a4qimHAdv26QFGQnKx1TYoqO7estoRzR47JKnvSUlF4tBLdRbZOwm63653ucwjpZJ8Zp570L5a6zinP/OI0XMk4fbeA2BG2Z+oZeiZePeY3XHWOGTMzUd3KYyT1SmkuI14euSmZopv1WE1XaAWeeiskrmeFlaODnGQ9IgInC9yjTO8KJjCMOaCnRmDQR4TX5/eo5XWwuYsFHXy/EIoFXK3d7qrAiS4+o51Vku4t6wx+D8ZMVa5yvR8f8S7I9Ygv151VtBIvnXA+arDzuT+bQCq575762varAHxWm65ZnVVXZP+Z27dpTWxlE7ry3jERG59Zb/d5RaDXEGq62tPQzUXV0z3wrgdONYDDdUaWgt9bQg5NEaw5ne5dPrsuL6ukfouhYI3us8WigbHmAwrytIn8G1VC4m+pZiRR/ewKMkfxMrq1dAuuy33VQxRLW0Kpmu6dMRw7T/qYL6kggpIXd2ZJWY0VaXYcAzk87DXvq1z63KjmNRsxnxKBXN/Wu9RY5VtLgVYGWHqSL0nc1FRH3g/0CR0ETfLU9vKkpmQ+kIdFMOj9cfQ5e6SK8+aNO+SqfEZpwLC03Ijgm9ZonHK0a8RuFq/9Dx2lEI5sp1XQKVu4pEOppgJBjplshbFT0cZrkpa+cz39fDSOUOw8kDanb418qlS5SL4w6u95x6nZVTUpW7QTl5chmvm6xqgd1Xzguu83o7VNNTHf9lhb39rvEjhT1uEdEA/0hFvf7mTIg1UtRrN+vRutY3UtM7vCj4M6v/pcB6rgTHv6lFG6u+r/VPyw1PM1zyHjdiopqGNAmCKA+DYGrN9GkUBVROcSHaQF8WjoXMt8T6HorZmsIp+JsbcF0EMEzRz5d5htXx1X7IzNwV1OD3zrB052ETrCDNOgB+Lbl80jcePDY+nX1tgtUM6Kl9nJHVgYgH71QGBDAeHFykDagg8FBAuTTKocXvqfCyjADAfxAEeDQ6MQdvvjpzK+jjPSZ+xZgcW9RxOnGIWIGZD+EMJrl0CrRYOTgPIVoUnRlaia+AhuyjLiLzrcCb3pbKGENsVTnw4HauD+AXXvhlt4//fgkct2X3q8Xxy6uuXbFcfldD9tp4MsVxFbDz7lbUzlr3r2QM18epbj0Er2caJ8oJqmwU8s9RDRpf95Euss0Wkz8PtUaBPwWt6s5BF6tWob+FiUOwLOJf7lNGTVpUrljNA0MQgZgsX07tcc4FB7jQCRj8GCzGbxyFaeTf4gUBKscgcAQlhKac/Aua2y7AakEAAA=="
  },
  "scripts/compare_decoders.py": {
    "sha256": "93191a8ee5eb12b7132d5e8a5b735070103994129cca8a3532d598a996bd14f4",
    "gzip_base64": "H4sIAAAAAAAC/6VZ21LjSBJ95ytqax5GmpFlLnPZcIcfgIGdju0GAujZ3XAQikIq22pkSaGSAC/Bv+/JukiybJje3X5orFJdsjJPZp5Mcc4vc8liES9lwk5YmpdNHbACY0k6nzcqLXJWCqUCVi9lzhI5F01WM5En7PTqC57jIknzBSvmLE1kXqexyFgmavxUIed8L12VRYUF1aIUlZLu+asqcvd7Jeql+12ovXlVrHBmvczSe2aHr3pT1LKp06x9Wiv3s05X7f5NkyZ7e9eXl7dsqld7UTRPMxlFflhJVWSP0vNDiESCzg7u9rBPSIeGaa5kVXv7AVN15ekdxozXRZGpsarlKpKPIuO+b+QUTZIWTsrr49uzAMrMi5z0kP5b2stUxaPMRR5LN/P08vPnj7c3mLyU8UPUlDhLihXOXIrDn38J2FOV1jLSWtJbVE3u1hpjRV+L+06EOtoQhAyg6qiSZSbWZlZc5LV8rqO4WOHWKTZu98PJOEub3sxdiTh6KqqHeVY8tbO0oKsikVkEYUjXAUlFG64Ah1ZMehPJ/DGtinwF7boNdrwya5Ssm9LtOZxdruul04HRUQRtxlKp3swoKxYLCXvvAZ/QtiS7engBJDc1ruVP9hj+AY83uSjVsgDQ8mwNwD+l9ZKgzYoqXaQ5wKukTMaQqQas1QeWy0dZMdw7na/NxCxh8rmUVUpX0BinvXunOcRhqAe2wAxagbpxvZo2nWqnCLNCJMqjxYS7zlwhveW0UCQRmdLzzVoAAWuxw4zjp+J3s/07/QKKIsi7l/aRm5fpnEbDhaw9rmpRN4r77C9TxhcylxUAkXDohDY3c5x+onjZ5A9ASyKfzYoDO2/GkwYLETH4HY0f7Ruta+2IVEn2h8gaeVZVReXxM6gwximMbphJ+pUXtbwvige2/ysz0A2MfY72RwqBJqcINU8rVXPf3cFeasY1Ms3BXAAftSxHjwc/j+6Fkvom7UwysDIzZ1psGuB3dxuTEAXqqiELm5l6Yn/wvbudmnCKNWNnA4qmsrJyz6F5gGrKjJHn3+/v70fH0cvWIa8RyfbSk/L1e4M2qSgMT3vBwDO7BnRs0NlWPsu4IaNEKl3ksHMluQWOKpoqlg6t5hA9xO+G+ISm7ZGpYheUHqAs2MvuEaYKwQZRJ32UUV1o5NMME808M8lv9WieI/P2m1R5Yg8aWynipcjh8VafxqfC1UOSVp6N6dPbqkGEks8pAmHxMD0XmZK+9QsdIMjj+rHPuiZ5nZvBA3tuAAwGDKAVce3SJMyIG4qVHJl8Z5Ih7Nr5LNbHRbmOJK3TErXqtKppZcGpVpdP4pH/T7qCklx0M3BTVj8mY4YkCoHRIgVHGm3iEMprJji0QQwCYVp0EtFLI5SFng67U/bi4sYECstEnmt9tWFm4pwJY23wADDwghIrQSRoL2P+DS472amBje20/Jhofmzt5/KqTlIpTbSJd0YxQu8FbiNr1R1p7eLyPqLOPF2osZmmo4kNxFuHUfgiXRChyuUTYGJDKZzvQ48dKfEIe1uOxBqFB0JSS6dkqH/DYkSsANvCmRPZqGBxJtIV8S2BWPgc8lctR8cXejAeJo/AGs8ZEsEgt0M2dxoBorhsvGHm/IS8ZLImZcEyJXuzP47P4AwkCiywZC41Ez00W+nJsSgp7nSXBvnAFdvcWajQsoKwKRPM8V4e5HrySNjWusFTYJ5w2A4iASqH66+U55NrYTbN8/jnq08nx6d/P7v4jQf0cHp5cf7xb799vMbj7+fR719OotPj09/PzOM/z27to//qD+SaufmX5+efPl6cISkgzxzwrWm318cXN+eX15/Prm92TrasJW9W5ZoJxfKyP1wXVbzsDwD8eaIzCOaquUkfRIYMPQcoQp35Wjp03NSFzHUguizuM5C2bo1hbwbKbj7YDZhNN+dRSLgV0CZ7dNXCApAklrURBfqc5W3k7aAtFhRvb2Hej+3fN/fRqYmEmQ18Weftb3ZnSlf2qJkLbIZNOPTy92LvZ5Gnc7AOl5loO1csWU94ImsXdesNHYkR+bp38xwpxQ914iJAE/5pyEAanhmBlotaR2TEL3tIZJNPlGGZzN1bmr7jje8P73IO057pI13iRQjq1QoiI8WvTT5FIELeAvgeIZUpBgFFRAEKex03thfcLHFaU6AiSopqfHx6NrohsnYQ/kwSU1w260ywiByF3QKGqzLzcm1cy6nizRXthFaBg2zcO1Jn4BYQ9kWLrI7b9A/dXLNx2jeSHRsnHZPQPEON7U02Sc/KIW7Tgf4c6dsuRBCrcGSVEMjcxjNOd1J9kbWv0Ty6nUKFZOl+CIepakVkHRAle5dFCtuMEU3GvAc2+mcizrs40NRkxzmBHTZiU6iikh0aeGez8UAebjkgnuG2ctWIWqcxJb4KRFWde7XJjg4/9JogdZqIBzx/FaA6DyIHPBcpADvPGrXsUTucAHm2o3BI0RW1q6xhd6ROj0iQuwGKw6wAN9AoUhFlWbNlCDKdyMc0llPyZR4k9bqUU50mQislYgVyoyXqhlxMkVY0IPqIDuDEKOijMo0fMtknxOQAtC7ME1ALKt00wPUQcFtKVJRtpdcfPtTDuEVo4Ba6WKOBGxFgc+Smb8L87CS4DU7vrNszfYAjPiY1wUoboUGZewpFvY100RSNElUl1p4REYrOVVko6e0Hh8GBb3WHFU5xRmMEXa0zYhUhgtccdBEcQ1ZWp08odqE3PYhp1K4KY5lmnotARhmjg7vxLz/5NhSRHpLILCVB912Isu8886rnGznMTiAYLm4nDDf9ERZpXxpAz3XchqImVmr2Mlj1On7p3+eVB0MEmytTW0SjLM0tGdDNH2/gzJZEEgTsvSw2Ce2a5NiL6uoDAccPUTKh4Mf0GrUdmBuRTd/YBH/NGqt42BGxaQX19RmI03tgR6mLRj2JeYU0qaYHh39FEYMok4ly6qxs15s2gtptbDZyYHBe4QSwFj7QcD8kLxi8GZlXbex3r61MfxL2r76MN7kCMoBtpSjNS6wfWXf4AK1TVEJ1UBesFC4jlBIRamqClweYr8Qz/RH3ynPy+Dbif8c+izpeMpeVv1cU9kG9i2rN4DP1yJIWveeiEVWii4+saJKcGH6OzajDqYUMN601Jh959g5ClMy03tIAYi5bmXmDzhh/nIe6mNEB0i1CgHTbIxaFt0GPq2k4RdSyogRBU0Q95f84/oOq9+ZeOz0//3R5fOuihyZCUF/ktExenebe0f4PpoWrXRxywTpvm5T9oBu+uMZuWXwfr2hGm2OJxgW6ta3JnDckbwjvfTYReH3+xoNWGT0f7DebPdq5V797mjb+yPgG9QtabE23FRGwBbLTdD/8lZTw3I5Pj/b71b8r0wxPnnKLWt6aZWr/BrpBEm3Qp+kOsjUspi1gpi/cZD8+MemP6xiOp40MiPF+BOATCgHchoB28Ohw1ymmdt8MD3yy+RxwgnF0T/aXkfYGPqGhtzc0RmjbCvbKHZzNBfjE3COKICvRjCh63bVnZygyT3Rf1Etjo20L7jCq/190CPoNAsuRThAYsB04z4QH3dL3Co9hSrE7/SbzxWCXNwqUjR1Ma2KFm3tdO+IKNIzdrx3p/8BGI9PqlNQgUPoTVtd/MV+vBFPUbQPuKEWOdDOD8qVtW7T9CP2livqz7qtVeFwtdEv2Sr8BJVNxlZZa31GUFHEU+b2VoUiSSNglHh+NqO0V6EBErdZ3p0I13zrV3pcHItaSoGwlfNZQGn93oVH4SHtUe1SwlFk5bW988+Xq6vrs5sZshGFKmnY//Yd2VF17mJ7Crn1EfWIqdqlXPBkwGN1eGszv+s0dCTHcwn1dMeZQof4SNu/97g5iyJFy8+OX1+8Quy3w1J3XbobR7c1cYUGYorLE4w6tnetEHEGWPjiG9N9PIDFL+TxDELobdEz7X6Zse63vHh91V/nEtgofJs4bt1ugb/jXuenrlnJ1b3y2dbWtFbg0XVRf3OJoMuCS/OrT8QW7vPj0r7BzrgkCet1Yz1IppP2xrVZ1BxMqcRR+jXKV4r8IWC3yUrAmTxpQmXA35+yZu67WQ2l0jZuAHCpvx6dIb/N7IVAU0BeqvJ4e+jsPs18tzTcY9w1T16St0tvvZcGuDwPbgdA2owOrqTcOBGmfwQWRXEdfi/uRyzgBP9rHf6ORTZyjNusTDdpqZXD/rtNb+wHUs4cEMn+c7myTBlp3A9G+Y7fLje/9tosrn1PUWPqDGxXemRSac9KgVj5byRXxRZMXma08GNWdmBfuEnBmW/8DW9HoWx/p/WAQsTqF+Hf/xU3lcyxLuJdQ8kz/pKsK3JJIeIc2Ez3+n8Zmt8uQJs0FlS9ACB05nfMXCr6efvLBAYiugQBM2IseeuW9euybkrc51d8sNZA+4exudzadgg9GlEyjyHY0TWbd+w9X7eNBDCIAAA=="
  },
  "scripts/compare_guidance.py": {
    "sha256": "e61a94bb5adcff4625d9e6838495851d1b64a008a1c2551144a3a2f55d237f8d",
    "gzip_base64": "H4sIAAAAAAAC/51YW2/bOBZ+z69Q80IZlWWn3W4AB3poB+1ggJm2mB0ssDACgZYom2NdvCKVxJvNf9/v8CLJlyTdyUMskYfk4Xdu39Hl5eVPTbXjrQgynm1EHvz05efgOriXehM0tTCvV8GnYC1q0XItmzoKdq1Qor2T9TqQ9a7TMyVEPlNa7FR8eXl5Iatd0+qAt2tsrIR//1M19UXRNlWw43pTylXgJr7j1QupTadl2b/tlX/sOplfXPz+7dsfCcmHaVrIUqTpJIY2TXknwklM96i1Wl7dXmBlTMfEsoauOpxHSrchLZ8x3TSlIn2rVNzxkk0mVi3e5bLxSmW8bmqZ8VL+Rzit2+ZO1LzORNAry999+Ht030ot0uF6bVcPuxCo6Z/Nyk5lTa3Fg04zA7rEkl6yFRy7GECtrHigiXRdSp1t0gIYr3i29fIlpNV4I7uo4ll637Tbomzu+603ItumVZOLMoVuWlYiwi+trXid90rTRCrqO9k2dQUg/fozU3aNErrb+S2PpXd7vfFaAXvBqxQIZkKpkWRaNuu1gGEvclGQY5EJQ0xETaeBxGRxEeAPXvWPmu/UptHwRegcSK0CunspNJx2xZUoZS1ugrwJagi1wjmscA7dtBjrFLSEYyrr1sZZafvhPOtbeB+5VWTGnD7DsFnZtHIta14mZP24bHiuQlo+Y4NhYppjtJLnKVk/nNjFMLBoJcF5uhxw8K7UKY7Dz2wQfXY3gEsRkXiVlsyNsFszL4te23gtdMiU5rpTbPImwWkZvCNnBNPoqGMxj6kVLEUdDsJLBidX7BZyV9ZoBlkugfk/edmJz23btCH7jBUZ2Wywnj29DUZB0eefT71tmb0mTklOT13O7SWLpsROyRkI2ay4nM/n6cf0EUuWDHlBtx3twW6fUspgdpyeMGIdw5+dDHEc2iMiPEbHUIkHkXWUI1Ml1zVgawVz1gH6frNAquAr7vYSSr9JpSi/DiBZNz4CA3DxSiV+lLTXGusIEWYnnfWFgz15ZJqrbar3O8EWrBTrhkUAoxBInZlITRZniw/ziK2RcbkZQxaE8HWMwbyzVYAt3s/x3l/B/jlUMytRXP7sg1BvRPDo/PEI+0C3lNboDjlMbkRtJnbJcnF5ckzGd/6IXxCO5JLPbs8z41e1sRM5CDIjQUvnqKZrkc43vK2aeh8FSFw0RQkm39e8khkq2snpeiPrLeTY4gsvlYgYqsIKtahu2orqhQPITRqHWox86xnQSD0Uo8UfbYdV5b7F4bDQ8pfR7C1sRTlS1mRlGIu3mi3IEONhUePA6dXJOUh/yPmoGR49p6EfrwRc42S05PW642vhJp68O4c93qaysFskCI70Dgea3l19mJJBTZ7o5ej6iuSWIzRuey3HomMTvklOApZkzWDvjpB6Px9vxet9aCPAxOZW7JGZ7ijE4ANtgPfIvsm6D44YdbxSSKiTl2Lzk49jH2xBLgvET2BKHblVIR9cgnPpDPfSTdaULmyt29ks5Woe+JMZjO/5HfMQW34R2hmoP6R2O5RaAYvHsbDF+FDuxWtZMge2AIPDMpPjqrFkXOHKqt/tTeLOtLQK4VrItZpZKWN/V6xezHTkPKAttSzgO/70ANkPgXmU7mwNjqttLtvQkb3ExIt4kCBDzTYxPuoLokU2GVMrV8cH3FlkMYreI4g+P/CMGIYRvUE2KvegDf/uyKvzwCdEp6QKdBNcsShrdvtU0Eqjy+TIer2B2cjCf8k+ylMg62/Ke5OhzDGpQcHjCtTMY+erONiuYQORebZaRB4NL5xmxfrazllmJIipoWg4FrBgO+SDmmDr6cXCPUTDLn0ZWpwrTYeJ6QiDxTlcxls7MYfuCxc4OqaX8GZ0FQ1EP8eAHk9c0cTZovhKNmcgn1RXvyE91OL+oHG66SPJJYBR3zRzORmAa6RxpCLXea32O4onZcsiyIAsJBatuDbdWu+SqxY/mzj43ZFxKmHQ3fAqZdYRm+B1cMfb/Y3hyDBbg+guAk7Rti6xSbeOmU3wQ0/TQ3zMaCPrHN5RwHZq5y+Ozle4SeiC33SCbeJ7wvhjuza5/LsZD3OhslYaCJI0zZsMjd1oXczzPOVuScimU6ROFhGHMYT9RVGo/6Oilr8hRLhhMAm8vmlFirLjMxCkVeJWmx9ar1wzAKXogirGw9AnUCrwo4FAdjpt3cJxerNb4PnMFhi1W9iMix0Vm4UDUxu2TNlb6phj+vc3tMYb8bBcXL27PQjrM/2Wy5ySAPliEkmwE9WKYqReL5gTi4qyU5tRunMrLEnNoYLtfG+cF1ei4lsux58LPswDYhZbvul9f493clYes5MDXK+ajPpWU3acE46aneg0w/ceXKyvfDsQOWNbuN3LwVlvkyV8ouIPU4T7VKFJqUFfIvZ+jn/TqQd9aulxxK6GRovCa7zvKK/3PW/ojolMez666gjP779+/Bp8+/rrvxagpjnfBjznLqMoierYdqdIDdFoXnW7Xxxta5rNvKt2KjzzaSA8bODBhiJZ59SkvpucP+z0SqK+S858NUAvfe6yz/TBZ6z2I33w600i/YFBgPI17T455xx/pVc0nuNrybhh7I8yPSN1foOi8JV+je8Licz1g+dbuuVxcwYuhpq1OKhFpxTCBuM9V9ZBM93xEgxnaOsPFDtXiN8kPt6eZS7PHD3c0rBMR2EsZz6ieabzFmDpoj/tlFkcN7gmOxcaNeYV7AZjWaqZPKIX8D0BdQP26BhvSOz/NXu6F9PF29mhp7AC/v3pwLT2hOnjsbnAI9osNV0ue3rB3pZSj2E/C4QXe80IH/NcUqYFAxloydDGOHVvxp2L5yMGIMQBNUrUbx14yutc7NwFftBz+nbLtwbo+fynEavkSBnSsuQrUUb08ZesGYYU2dfsJaobGRlQ+T5UZ71BXqLP4z7RhX//zTgkBfyhodHpLUtLtCmidqf65hWda7QGysk8vo5Qb9LRxDinE4BxtyOtQsvHx5/kooHIeoueD4WxoLHBgRhdtKfdVl36pkD6patGb0jJkzr1yfKDLfgloTq6JsiAGbsaj+2ojNk2BYkg4JqrMyVfPGRih24M9vpsHslZkbsE+cXiFVwKDmMBFCOcFOyRGGBo3iZxmtYclSl9WgSPZuhp5ELGA+3HROLq5ah8/j+k+OICgeEPShKWpkSH05QtLC2++B9WY/mhexkAAA=="
  },
  "scripts/context_comparison.py": {
    "sha256": "365715e1fe735b7e92d817d0229d146aaf439912693120052dd78c3a71e31e8b",
    "gzip_base64": "H4sIAAAAAAAC/41XS3PbNhC+61egOFEZirLjSQ92dHA77i1xpsnkotFwIHIpISYJFQBluxn/9+7iwYf8SH3wCMDu4ttvH1hyzr9oOAgNrFCt1aquoWSVBrNfHLQqwJglHcCDRYEGBaVRrUEJ1TB4kMbKdscK0apWFqJmeLhjoiulyjjnM9kclLbsB27PnMpB2H0ttywcfMFlFDL7zsq6Xz2a2ezv29tvbOWkkjyvZA15Ps8QnKqPkMwzwt1asz7fzFA+I+OZbA1om5ylzFidOAtLxq1StVkaC00OR1Hz+dzjcVAjmr+vv92kTIMo80PRpOxeSwv0M2DX6gitaAtgPWTx/sPvUdB5OZuVULECjeCObA+dTUqpobBKPyIk1ekCUlZ2Wlip2pTVYgt1itweHnN4EIVd/SVqA/PLGcO/XjVr7vB3Mve7QLQ7A8hOL0N++guye3HkTlRWI9PeJv15rjM6IlaTHtdg2V8FiGWkJppDDQYvjSQFzXkv0nOWjGylUXN9+f6dVl1bJpGBd0T6fBMckztUQvOe2OQZHEzFUtKSsu7E9fFZRrHwDAyxScYSKfvJTbGHRnwHbXCPX7LzlHEN/3R4K65+8ogRF/HnU9p7+uIf32LgDSqsN2isVPftZKPYK12G1S8M3QtNOJ0wv4mV5ulG+juDdXov7V51lonS+4UFuBOy5SGtNk/zUwomjB185ZeeLWSkhxSoyY8Tbl7gA3d9aHNcAx64GhrZcXhzH0/ScxFOp9EajkPcx4fzkbV+37j4XOP/ccY/je0ORr2gv3kUP17JByjzgNDxhm5m2Dh4qyzQFfy6pkR/HBqc/JfaY1fXC9fpXIFfsVYxA3Qz/tINSTlyls5oCIIG2+l2SNnQKUIM8qG5JrpDWjGsaBobw75r77CRlPCweh+6wkggdkfcGjXG1G96EX+9LLFTSvuIChTtrFaiNAnpUSbEU58JZAnLm5p+MvfqztsXVWMSLUnkVX3sQ0gqO2MfV2OX2EdWQ5uQ6pq7fcM386HhaCENsO+i7uBGa6UT/hVq5A+D4KSZNExsDYL3L5Ld44bDiuh4f3V0b82FMWBNzI0N+63vNfGlwDBWcmeWXnKxFQaCV2/B+qRKqFkjWllRBytlVWHpRFDAlJY7SfWJuK7YHcBhug0PB9CyIUcM8UnFFFqe83PFphytRyRuQoRcZ1ixk7hwXDjhNadWz700pXD+msry9BUJNYKZi+0L5VuKbFhVSrPwU7YBJV2EIIl6f7Tm9Dbjzmo1qdg+QiEI4TWhsATMfaTwmiA0wu4kJ+gGjbfC9afA/lZGzvwQUOxFu4PyioqdcrURd3g0HouGGg3AcSIgV5GRIcPCXuBZVlFozRvKEZ9zXOBkZeGwOJ5/cCnGyT8qhV7aAJRUDCR+/pYvN5g6riL6ZPoDDbqY0vvAVAuMjPHYh0xXWzdMGZdWGnWpmLNdrbZJxX+OUuvy7KJ8yq/zd0uvFiqhDxshHhv8NdrPQEgRUuFD4LVdFlFBmEl5B8i9Y5MGNL53fbZ53nOiWk5jH+o+0/DjI2Z+lMx2YBPupkQXQ3rdcOFTddRfo/unesJ2hjsO+A5aoCouXWQpnyZwMmkwYWt8Jo6QW+X69yjFJ7LOYNxZe3j/J8lvI22+hQSmsUHKljK5BksNKOZ95Nq9GWHcDPP16pvucDh0436u7sKEOntpkJwA7x8p7C39gc+ugdbTCeUFlXHupT0Tb2iGIooaYekV3LcNZsNkRB9U3bEHyYd5PTZQLRrX2JYnc86zvwrLEiduPyAwNaRKCMa4yJ74ZP4nssM4HIYsRNvINrmgbxrXX62ySMsJmn5Cdh9rrzoYBPKLM4P3jnrp6KvkrdGU37YQb0lp8vmh8JPrilVS48PXHRZWLS7OwjxkyHWYDq99H3KfUCscz4ZAdzSD0Xcb1YObl3uW8GC0OoX4PL+EtnkAEVWpdmjbEfdqFCcp4NG4nXnakxe2w2p0kL80Ip/ax4E5hOXZ4OuTc/odN6dH8+Q1fGYzvjqXMdnDFEtD7Ge4Z+FbHjBQdP2QjvQyLHHSoQCZjH368hUfvkfXsLbAttIu/NOGs2/Gn14vueFtjFXnAzwZfv3W7D9T7w/XdRAAAA=="
  },
  "scripts/cuda_precision.py": {
    "sha256": "57f060f17bb4248a87a14f47724b2c63a336c08ea1988009a5cc6195aa7ae7aa",
    "gzip_base64": "H4sIAAAAAAAC/4VUwW7bMAy96ys47VC7SJw2xS4BcsjaFBs6tEGbbRiCwFBkuhZiS54kp8vfj7KdpC0wzIc4oiny8fGRnPOFNRKdG5ZGihJuF1djqE2p5B5yY8EXCLXSGjOYXc+hEDor0YLRUFsczqoaLcL195tZwjln7CMsTxcs7pRT5Fqh9vR2IcTTcr5Ib5a/FnNQGoQGtJbyFEp72DQeNO4ovkWROVA+oYhftfOiLEEcs7tmI0vhHGyQMCLYRif1HlRVG+spi8TPQm5RZwncG2hq5yleBbkq0VFASicLlNvahKSFcAU6sgj9jANClLVFG4JhVYYghdbGAzopaqRPysGLsVu0Cbv99jBbXo3Tnw+Pd/NHmII9OztjHYwAqt4PwO0dy62poBa+KNWmRwkLOjInrao9XQyniFwTYZ93q8t1nFh0ptxhFLNgDpdXk4s1ua6omqi7SGZL1MaU5b3NrcZrGAHfEQvGjoj44ZPHeniZfOLx+oDRGyuLw0GQDMglObAsXDCl/ZGxjvLb0gh/Nf7SWaNXHgnxHnL0n+IJA3oyzCGlO1laCaXTymRYpoGQ9NSDyGGZD+Ccinf0Ot++hH99gPCoHDpb8ow+4hkJSyKP4cMUuGwywU+u4bFCOYTHhlRX4TzoK+KtsINQ++6RxH43iliGLtq0jRMf4wRISeb31PNpx1OSd6UfXWobsPPVwqJshb6Gh4Nq+hFoE4bq0U7gRi1HHv/40Y/ZfNoH4wPIy8YV06Vt8JTdom+sJqHTfEVx8l8C31PHTtwTY+lOYNrW0jPdl3xvNL5iuU/6tlj27w4TMW/VwA4Kpi8nMU/WrJ2FhH7TuhX6Ua0k3WDVosIpT7sKU+pCmCPGAv4eRypNVVGWqH/3sGnp3CHW7cQ6CkI7xaOl1US/A6A5sHvoxjzMNeFpwi5y7eqBnLpfdBuvn+iww14xseqTrS7WA+BDSc16O/LE+MGFylyzv7aMZFNPBQAA"
  },
  "scripts/cuda_reference.py": {
    "sha256": "8d3006a79e47f83efbad738143d934bf42fd607868f885af7c6c3dd3382e621e",
    "gzip_base64": "H4sIAAAAAAAC/51aW3fbuBF+969A3AeQDU0pSdNslfLBuW3Ts0l8ssn2bF0dHJqEJMQUyRKkHa3r/94ZDABeJDvZzUMskoPBYK7fADg+Pv4lLVSetnImv9ZV07KqlEx+TbOWvWCNXMlGlplkaZmzpitZu8GvSreqXLPTl69Zmqd1KxsYxl5+fnUKf4tdfHx8fKS2hl3arOu00dI9b1K9KdSFe1SV+/VFV+XRqqm2rE5bJGH2wxk8OiK96VpV+Kfuom6qTGrt3+z8z1Zu65Uq/MRdp3L3+zdFn44+fvjwKcEJAiHwjRBh3EhdFVcyCGMQXJatPn+0PALGMcoVq1LLpg3mkW6bAIfPeFtVhZ5pmFDIq7TgYUjrSLtcVW4VWVpWpcpA2b9Ju8ymupJlitq1NC8/vHv39tPPkd6kj5/+NbpuVCtFrxfUv+eWbWQuvlQXUV2k9ntWbVFisYaVDtnWjcT3RERWFutCtdlGrKTML9Ls0pEW4Ai6FcRJ+Zm3aSauq+ZyVVTXjhSkQboteIYXr1VbKWR5pZqq3ILqBrTTTzRGy7arhf08pa537cZJANqW6VZYcw9lKKr1WloRsi5PgUZmSqvKKwukTtsnj720R0e5XLGyauVFVV0KXXVNJkWu1rD0AEIhDRdHDP6BF79dl1Uj2T87kAWcXH6VGTggsNYtaMpEBUZEXmUdLkrmLG92JyZQQKxCRhBCbapKjBaah4HhskttQgQnQREStHEMUuaa5jdfMlkUOjm/4fhDtLta8gX+PB+8WEaG1P3jNAdfBJzHXypVBjTAvl6GTK0Y6KYE8cFBJl+jAuI6ZLLQkk3GQVCAn8EI/urjr+Lj5/csYZ+aTvJo8OJNCiN5eDsUaVU1hhdTpVkqCa/5cmmoQDtdU7qkEJPjB0YdebettZFQRxpd9lLudIKTRrLUHfh5qjOlEjNrGEOWqnII2jDeyK/WlqE19ZVs1GonLroyL2SQK3CQtmp2vZl/MQRg3raBxAdWxFyg2YVcofWNugoIDTR85b0ATIq2ryF8UvBAZ19dprXeVK23sJ+PEk0/fZ9pDB34plqB2ENv6KlnnMSP8SvHsWkuWpAYlmyGg2Udh3gt24Br8LRtKmDxGAw8fJA8WnjTNKkCK0Pu7+TrpqmagH8GpdYYL7AUk8lpPkZsOE2C5gSBqiZHg7r5zrnRF1/2/DFXJgPpadA5x/fkT8OlW/khJBklWS0aiQq/kqKtBipD9Q+pcN4gXIyCYH9pL2glSMy2SoP/r2fZJi3RasCw6lqtcrBzq3tj8V4ycIkuLZI7EoYRxVjjAnKEBnPgUviUmlybo9qsAk2cWY9HJv2EkKMlumFClGTOOxhGTrP2eTnSKIn+IHEcF3cqJ6sgfUHGtnqxy7fx6Qxt48nWEBtPkO2itMk2YKxB4iyzosuR7XarWvSpi0bla3yR93mTcAYVM4Abqqy79jm8pbEQc2wL5AW7lmq9afukaaejgHJzT3zqT2ytWkfpeGpWVFCDffJH28OcOmLKZHoDcuAJs3DEMngBOlGQYYzIeXVdYlwCmZFLx2YmaxAAFUmPSGKT5wXxD845CAPJ8uQl98AhjLiVDj+cQGht0zYBLvD4j9enr5wpG3llytnvZQ7jTgz08vwizBcmhYYxEKraqupatRuPl+JPEstm2uxeuVgIQpZqQ2De9+F2kWprBP+xdz8PHhMkm3H/zAck6EiJBSjGkzzVyI+JEJwc8ApfPkjm3wr5N6rRLeWxHsRSFGjjelWj1qoEV0CWbG6WUqhS8nAiXdzVCI+DrrYYhFw6sWDtnKemdpJ7u4/OZuPy3P9LNSAfbWM4sUmA0CQE4kqt9YxITlAum/N7yXpkGPjV4UiH2mhARAsYDEM7W+wb/1vVbzB7qip+gYnr7Yegd+TQWJyex6ouQV2QmW6gvIADtXpmMJeXIq534G4uvPRs/oOYEKh6V17ww4oZDvwb6LJIL/ZH3q2haCyUB4Io1AiVuGJDi4FCAqEFvKwC4jLdSsRCkMsXe3Lu+9pZB4kKfapPduR4XVnKZuZzjYUTlD2hBg1czco0wUFWHCwuwX1+MYQ8D77Xm75nZf8CzI84h7KwqwJQJFdgE80M5h4v3KGfydoMPkhuVhZRzm5Qxbd8MVwhvgoNwsBfWCj3zOHtBm8A8eUaPTrgsyHoRAZYTqnSOt9p1kV1EfA/T9cNDCdYwoh6jnnUVvYeh6D2wnCZ7JV8z9JjuJsp/Fo8ivgoR/CFTxJ8klv4YpJcDgQLH8OAhc+Qo9fLiOddY5ArXzyZR5RAF+N0GnHXMQoNxRHah2fx/OCcBPMW0JMYHLdAO0QOeSwm3ms6maFr3nrjRviNLIx4MzB8Y8hpW8RPy7Exvwtv+joxhJkDF7HJ45DX2yF34KvlXbiP/GQ84QB82dJu9xDi7SUgy8BuKNguBrdRBHR/piDfn6Qtt4h/5RGmeYA6BhJ4urdn4tXrNz+dfnr9yqRuggbj1e5pf6T2BQ2JTW1B9/ek41Ceko36kmjQujlrRapEFJU8Dh/y/5RjYGnXZXEloRrM3QH1/kM8qbF2m2wDFQ73KSwIc9sFzzHaO0hgL88+z96d/cxWgB/N3obNuwQiPYjzYBIRaQIPg32oCFBPtjnCpIk7SfgQo1iYJtKrVEFdwlwRMW4SfVf6l8ZNx2KZVbEfzz4joAX0icE4yxtYdsOP6mTAHXC+yCEpgJMB0KthbgXZZR4e1Y0q22FbfMPNML6g0cKlGSGgPJp6S7M7AvvZTAPhXnd8UccUvVcN5B2Tx/BdW0GrK7ZyC7APWKV1eqEK1e74wmTgw8L2ZCBseAtAxekW/LQr2iFyBcECyq5kYYCtGfp0LpcR8GmxsScXoyjxkBW6/qvkwEbSoP+l2WLyLWQ57Xc/0nBb3YztRsaytlqBMaFXAmd9GFieus1l0zjUzExach9AXPchHHn3oJE/TG39HrexfKnqEV1Eeohov0HanQ4fEh9KSTjjS3VBWcNj2he2dM4wwc9evvkRdydr/ZxdSlkzDA4Hd2mDpgeOveHGQhDK78F532xF5gtRTXswJ9BoS+MexHp4W8NxgWo4hM3YB3wn1jFdvuVC3fS04gJuGtfc+7ZKJm3FGB3NXFmftNJkkWSwetv25DOLc67TK2qNnHGGxO6dyFbrZz0pqGcEFmEdvbomlQy1YIkdtxG5n8INuE8HzsWQ6cihxqv2PGGl6VYfnE22iIahnGMZRSq+dGubDD+fIhVwgWcowR6dKq3mhPF8JHw6P0jp8RGQPJnft+TXdhOlj7Mn8xMNAKDMIwZBxp5F7Cm8gQmn7SRkPyy2Sa6yNuh1YF8DBKNMmpjkbcfYQnsfcqCMcDRopQF3tmnbQS7neDBQgiGiKVS800GiPRdY3OMdk63nCbgdh9v4Y/g7ES9v5H87CC2ErjecVAULNLqKuAk/qF2ky3P7vHRY17/HR3Sy+SHAC9TGTxZP52PIfDsRxTeVoq4KlUFh5G/OnjzG8y/4dHK6hbJt83LQ10X2d/ZDPA+fM7duBnk/hYrAKkA0zbXCXZqjvVYYmPss7/QfQQfE9DUk8tlWtikCtFlRlesTs3tmw0/H7JVRE8vxnIC2rz6+/9Geo2jIVjvbxj0HTKWAoMCDvwxcBrdX8WgKxlBkd+vYts+DfQfyzhmfdPf7Gw/21CUZnBeZfB1Zw0QHMiKPHPu1hBaajOHroP07Yv8wOecnJ9v06wlUQxuUGpzjydxsrjl7n5C9I/7MvLaA4ySXiBYa2j7zK6OPPKRs1A43vtTKngJMdkdcY7VJm/waVgLtwwDUjg+2gnAMrHtvSfbZUBgN8Nje3sHA2cCc78Gad3X4B1CQwz+5yk2DTQIYYABrHPB+7lwEnBHxuE5XkvmY2Jeq7WpAyv348O/BD9H8QBvm3GRyWBfYv+GBvo101Ack+GwDeQaVzi0bPhqGO+6L38cI9Tga8UdCwMzijyrdku4GtJHxmUlfaLc+Drod9Qf87KfT9+zD+59+XWCOMIb1ddANhNhaFZ3eTLgPUCtJPjqMaBQKNkJx+wE66ynvRHP4DwI06UnPOTxTWh50/QVEY3IgB8xWx/P5XJyKGxiFVR4C1py88uWtwARP72lf4/Z4sJVr+pD+2DygOSI8QUcbT3abkdqFEaIG4jqult9AWsTk3JUrIKeC9a2dawsucfCMUro5vlTUONreZBBo1N1G6aoFlU3QTeSEuBNjmdAjxJbcXMqd6QfwL57XGs4xnrsG4f/MBPbBQDP6inkJXgKQJAL3fDtUqJ3hBNBJkwlzL4LfDkH5HXCQED3q7tBC3NdvafRz6Y6/WO9KzKhB4qm+LZnPnd3pyB5PgUFCRRsP4+RmS8PBgLDfZvbvvaFwlWLzbI758Q81YymdlrvxNvHjoXl0Tif4+GRxVZpdyhLcIUk4VBXszgUwHRjXH57QgANJDse6dGlwQrCfnIeyGHwlDK7Atol2ImLPwZ7Q+qXhi7TcBQc3/PuV3MlvqBPPNPzmme9HBxmZgWd4ewVMPalv5pNZz8xBgPBob7vM7SYHsPZaCw/tI99FRYEFo5RVZrbbp2N4vBdkUL7BmFvTu4XTBQyvB5mNZ4eAzDbcQy5w+0WWNDhyABUalmgNvprM42cRgB8x+HDXORb1B4nLqlgQXFAdiLLI6kU4l7d/7zrYgnXk5jqZO9vyLFENvnWg1cAUAsUXF4CCcQ29zHTRK86qemf25Q+UG017oLbeOoLRW+Inv2aybtkLsNZr8xPDP9VMoqcsvqEmm28jQ5ys+A36aWCewlgItI4Qtwt2Y17dDo8P0Sfp1gSmuGIAH38viKBtoi2oyl12MAe7TeJu18WnzdrUwTPzPsglnYShKwiRV5kQ4WBcnOa5SO2QAIAwnUnxKDWZLgFrQW4XLd7wuXccwBceoUbM/lC0kUWd8JeAGguJofcuzZh1oMEmk4ksO+O93P3xuJ/hfmH8AfN3DgDdfy+pRU/3aAiodWJHmz843p0N4VUMeIpp1YshKkzMB/jR7515crxIZq6J7N3LC/ZOGoiPu+8AGqZNMbzPwGfBxL8Ef4j3IWP87y+BOaU5Xzx6vHzIY7yA0DO3yPJzjVWOOkV7L6it2A5QD8FMWDSUkwX2aHddDdnj+b6yO2ceo87c3jzETokdtZkPnChm9paKv8Zhb4SYetWV/fnj9GJHPDrPR3w7ske/ied6JmtBSVVk4FRIgfsQCm+JVHQTdrB27g0djDnbE6Dp/a3+wHF8O+3g4OGGENkZfv9RG9v9otGe83hWt/FsXXbQbVvTfaDTJZd1MUHBwl02BDwhBCYrIfiCktbR/wEwg3q/aSwAAA=="
  },
  "scripts/diagnose.py": {
    "sha256": "d18af5f78a60e548a54d4f9ac7ba5d1ac1111d2a1a6677637ccf896b277bc14e",
    "gzip_base64": "H4sIAAAAAAAC/51X247bNhB991cQepICrfaStA8JXCBtekmRJgtsEgRYLARaGtvsSqRCUva6Qf69Z0jJlrPZNq0fbJkazu3MHA6TJLmSSxKNqWQj1tLWW2lJWOqM9YV4TRuyorNKe4cfU5FzgvRGWaNb0l4YKypLNR6VbFyRJMlMtbxXxJ9GLYreq2Zc/dMZPT53jfRLY9vZ0ppWdNKvIT3sE5f4Owq69VSF6xeDJ/uVnZvN8FWwjkJpR9anZ7lw3qasJy3LpWqoLLPCkjPNhtIMshZeu+vzG3EqEm9M406dp7akjWySLIteQSuvhdeja4OOsDibzWpailrJlTaOUmuMz57OBD61crdiPjhf8L+yd3I1yASRmGYIfQp/+ZMYlzzdp6aI9tMsF4m01Vp5qnxvaSrSSqxrhJQflHQ7vzZ6KhVXSsDplNFB4bBEd1T1Xi4a1spZPCxMNAb/l5aoXKkFBK3pdZ3yasGryOH52cWTR48e5+J86slKeUgvkKp0yMR2rap1Gl5k7MZy2Xa0GoWmyU3Hd9lU43IJ+Bf00Ib4MmhGnQKYctX1nNPrm4kSb5DNEqXiZdNQjafylqymBoLHlVssla5L11GVxl1JJpQT2njx2ughRZ/Dt+w6VJlTjaqMBqoR3muG9EbM5yJ5Ie1W6URIXe9fHsEaxaRtv3+STCoEUlPVLHZs7EjW9asVoWzqsqaNqoJWkbSdS4RafuEkNY5gsPfm2B6kS7mRquEyQG7H7LCqX9DpFKS/VPd0n2Bvd4c//GmpNXbH/bBv36JaU3Vbmt53vU+vk9Pe2VO3UPoUVVj5JgGEJ5q/19sC+536C7HkwtOdn7+1PeFRtYT98/Oz7MjaGEav1VIhEdF6KF2OIBYvSC2NLwowherSLDsu471Kuquo8yJ9c/Wztcbm4r1sehqeJwFd7R/Du+zpf3GKq2nM6nGrxDo+ca1KJior07ZcSHNxPZXgpJ187MnuTlD4cy1byocovUG157VVYIGRCaI8U4T088ptcm3WJGuyeICjHqU7OwTg+sYfY2h7nQ6e5KKSHZfxAOmI0X24LiZwIdqoF9yMzboyNXEXnB0nDx6KBjQnlB7lna+hrHBdozy/cukXCedPiH+ovlzE2BFBCvr3I+xBOS+wctZU2KA0TXKk5yLL7mkdoZwyzE2BXiDU1aeEjYJIou1kY2VbtoE1DyXH/BS9wXJ8+DwpOHTY14tnsAje8r2LnX0AXyzRsVQ/gx0mLC/8msaYEdrO9BZw2FZpHHFH/V71tfy2hh93PMygNwfPhxMzyE4RDwsFGy3UhGgABVc0c+tXM3y/JMDNdBdqQmocrRO9kfvKClXs0dn3KwMqEdtkx4r8wJglqrvDEKFQU8HC/60ALBf3q4BXQy8OFCBOR9q5mFZFcsgD1cmkOobprO9q6YeYx3aex3jK8X9ZoisfwHb+IAr5vXAHEv/6qTAPB/FaOukxcUWtC1ndIhUujydPBPb4VYEXXxgeWm1/WLTyLk2RrutD+m4C7gxeYIKvoIDWwkwmQRHzs+yBk7EFy8TiTmTFS93J5vy7k7vmZCEdhYNycOKHubh4Up6dnY1n5UQ8yB5ZAGTxvE2TD694TpBa9B14hmQrPvYSrLIDTepaMXZ5KHUJU9KBN2uB6UCTLcRP7148F0NeVNjT9s6LBU/m4dSELMLn9g6RTGfyInkAvUNMMZLXRrx+//LFy+fi18t3SBkPIVTDeKM6DPsERpQeNSS2xt66Z7CGeIJnHHcg461qmtiuvRYrgu9xx45webgihpw3DXOuM8FjyPJBwGngiGSNUwN2k+xfZopvRvEAywPQPGcD4mqYgQ6Rv0Na2EOFkRLw1OIyjMjicXF+PiR6dH6r/FpEtpijjp+Jh7Iexs3L3VsufaGJIr0t+CAbqHNE8vceEzmYOjYV8uf5SAqGfkREucCAIBw05P9sjDttqKgR8AWhaYinMqVX4sOrQvxxeYX7EcNn6WOvcKTuQwN1YECpApITVKasPMzlN3z/O16P4/fNfdRkXysTL1KlrFj3MJfK6s0VZnlL2zElIqp/xgd9yMGYkd9MS0GQL3roLYQ0dgOPsBhh4HbT7IqxKXmgGDzATQ1BlCXTcVmGObssW4ljq0yit+Gam/INtah7YJrub3V8iSyqbZ3ytYIPBI0ZBlz1N6tLNLo+DwAA"
  },
  "scripts/diagnose_download.py": {
    "sha256": "60f5ae98fd232037825872cccf1f402b23dec0a23d0f8eb6b002bf4ca5b444f7",
    "gzip_base64": "H4sIAAAAAAAC/51Xy3LjNhbd6yswmA2VyJSdqs5CiRadTLtqFpPuSnqnUrEgEpQwJgEWAKrbcbkqX5MPmy+Zcy9IWpLtzkMLm8Tjvu85l1LKn7Wqrpxt7kVl1N66EE0ZRO28iActQqdLU5tS/KCCFp+02R+iiF7ZUGv/nbBOlKrEudo0Ogjltah0o6OucinlzLSd8xHL+075oMf3/wZnZ7V3rehUPDRmJ4aND3gdD4VDH00zvfW7zrtShzCuRNPq2ezn9+8/ijVfzIqCrCiKee51cM1RZ/McerWNYXOznc1mla5ZYyjgXuF16XyVeefiQqSX+Wom8EsurQVtiaWQue8tqbvijeWh3++N3deqpOed5Dtedy6Y6Pw9LsrWIQzh6kqKrwfRG0knClPJLczrGlzO5FIuhMSpOYvYNW6Hy0n78lQibKDNIHk5iavMXocot3z1EGNXXNyXtCj/oiCvY++teODLBYuqWLBcPSlZ8NHxl44i0NGo5vRY/snEQ2FVq7OnNXpFUGRubOnajmpFzl8S2LjyDtLOvFnmtBqeOZVd+sIaWMKl7Ihi1BGCx+Qeta2cX7798d3VL1F3Vzf5m7PoQJ6xKhpn5fZC1nl4ODJCPgXiIgZ/2v1Lz/+G049DtQerunBwMeOyH8ob3dE3EZXy8Mjv1Otk14KbQxibmiQ3UbchGy7RL/r71ZmpISqSQ8dzes7mZ9tJ0YZkb0md1J9NiAG+ffQ91Enu12B+1cXuPmraICkQxYvnYbn8we1wV6gGDqtIWbiQsOOoia/Em5tvhKnFQQUVo89oH6qnE3Iu0Kpa/OTsH2lEVwMLocueKmoJGrD0OF3Wn0vdRXEL735y8db1tnrnvfOrPxmdWwWDHk/7MZ0dISwBoQ4FF5frtGXk4yQP6YLDFvXNmUlys7mgPLsRWfNPB1MeMtkEV8uTJA8aNwkP/ineEzUMKsW//wWQtxU81GUf1a7RXDkjX5iAHVVGwfgnKL6JI1zb0jX0Xt8SHuezZ/UEFztnA+HuE9gT8mabZCNBpeW/t12J/wHJZI+3C/RKB6t14frY9XHN5fXFXA4FrT8Ph5lOcHt9c51qeMhh9v4XTt3i1KiP6ey7z53xuno1dlOe4NKwRGFqjNXUZKPDqKEK0vLQNSbS5lnPIZG0Ru3lY6CEZ7JDzVI4aWNzs9rmJgADTDy9d2ZArjoUSZU9yA4EtIL2mI2X549PTaubl9SVg7pJ3CtaNlc32418Kg1JZT2qOa3m6cZQ0MPgoZmL10TpCzJR+6NqKCFJHyrI1AA6CKUBAkinqpBlI46XztZmH5YKgmO42mFeyemcpHFAVQXlOpvPBzsIOSHI0uLwRrkZHpGdUduGQSoM9Hj+Q7BepAmxxggA8C7vOgc/YBPcJW453rxhw5bcH3lQNfDfBufDMAIw8A6I+vqUkqYFjW3ulnOMH0qPUlzL/+hWe3MXlGh1VJWKiic1EXSjWiUexiiv9o8Y3KJBu0ZTqTsct/uD6vqwpKd+pw6Jj/732+9ovbrpw4EbJ6mj5slDo3WXjRLThqrx9oqNRGaeaUgSZWKcYlp+kXcFygqTKOH8GV2Mp2nxkp4B2KMxRcA5W4VU+rwEkYQqhToq03C1grCda7KX0HHOvA7LcejhTt+vJvBBbTUw8qiL6DhJ8zkXEg69QqePz6xMmST9/ABdHDYs8P9n58uDsntEAkDqDYeBSWTi8UklFWiSObDMP9ZJZHp9Hq8LZiH8TjM1cQz5TvdWXyagkITPz8zJ0lCzOJ3tXogE2Cn5I2+5SpHV5cDwqIMlM61I7gfUayi92eGLA4OAQPWYo4mIOlGc1fGT83eC6oTM3XvCbfk6J8i3zFapO5gwMeELZ5PwykEf86l3RzgUBZiu7D19W4AdD7qpvizcCgrSiQpT4SoNE5ADbEXPJUNTXBejMleDVTFjqYmBoTjEvrz7I3267eITbzdwiK7S1GlLdK8hN4D1KG9a7+3UBzQlqHLUhbihyEr+oPuCwildwM1YHih2Y8Om5SmCw7ePYHgMfSuOACh8YHKGc7m9GHsIIsASMLXgKbooGFyLolXGFoVcDbCJr0vCmfFLM387TBofeCdLtdKRjnVRVK7EV+LJzVxVVTEOJxm+yEaUQMXG+06va5AN8BdkpTCHTUMCriS0ZiH8j8SEYQ4ehrBr8T1bBggYxNLCt9dPNDoI0DRpZBNqibZH1lDgO1SzRvlcMw1/ey1GODsFe+bEqm+7kE18OpHomfY5sSvV3/ob8OH/AVs95IcDEAAA"
  },
  "scripts/drum_kit_reference.py": {
    "sha256": "636b507f14605d8685b818fdf226bde160c541bf146c1e0961df21a0c964bab8",
    "gzip_base64": "H4sIAAAAAAAC/61a62/kthH/vn+F4n6QdNVq1740RddRgGviAwIUzfV6SYEaC4GWqDW7ekWUbK8d/++d4UMi9bAv1+4He0UOh8N5/obas7Ozn0rqJFXZNlWe09RJm65YH1nrNDSjDS0TnC1q0jBelZfOkdLaqRkpK4eUqVM3lNPW6crklpQHmoZnZ2crVtRV0zqkOcAyTvXzf4DBKmuqwqlJe5uzG0dNfIBHTcRvu5bl/VN3UzdVQjnXI4+szlhOV/0mCY1RevrQxsltVx65Q7gjv2kaPFIMfG4ozuHTarVKaeYkJLmlqVznZVWe0iaoc1IGrEzpg79bOfCBhQc4JY9Q/jCvSMo9Rbxx9WSIk64fNpSkMQrj+b5Y3XRlpKk18TVvG09uscclvMrvqCfpWeaUVYvLQsbjhuakZXc0bivNJHOFuPGTYLDbvk2fXX/XEMap8wvJO3rVNFXjud+Ls0lNCIU7lCekphx5u3Iz3tIikroK72jDspPSBZBIReCfa7kjd/fXYs+9Ohltu6YUPAJTNbB24w4qn9WMVD9pkls4nDaAegyqrp1aQZ4+gjnQwA3hNGcljaequJSEYXFMWaN02pD7SKmNbyZLYpSVh/fkzhXUBW1JSloCS3jLWtSiWtRQdCe+MRbL04l1vU8M/PzwvmEtjW9OLeX6fEITHsikPURwNb3LItTi+P61e6AlbcAhqlLJ4u4FC2VCuRkysmQx7RDIdZYF5xzADIZ5H1CclClrkhzJgXpoO94lGLG94RqnJAUNsi7PHVY6156UCnJMnFGa3sDSEMLaDd6TnFM/MOYhNrq8lbOfmo76e8lURYpgiWkIQ0Ztu8NswMqO9oT3DJxf5Y3w36x+D/9R0I0Qy713A0xwGJig2Kgn/PFD/MPV+7+9+3T1g49543HYWp9Lh6c4ljukSaVrt0+hYAJ6x+h9WNRvYVylW3vU4o6fnqxgD2MGcGgGXkE3EwPPsLGI+SKryTYvsYJUAs7T//+MzQfn3Wy32/idjLuYgwtsN8rQgst+N2EDthYW0wr3MTmimTx/9yj93poPesLViA26zJT/1Ji96UQg9ybTT7ZKJ6OgSjE2cxT8vCKyKk7CnajXVFWLObGG0HojgjHGdB79vSpxEeotVhlDjvGqaxL1vYUiTNtoG/Tnic4D+kCTrqWRiDfbZvShpkmLKVn5Hr8lklM/Mzi1nlORbggaYUkXkhv1LRCDkmCm7Hn90RzGHcE3ijz7gP3MXMG7rSoYIiUcgkHmLg8OcnTAtMSRbBwIarffUOkm6g80w/RjD4IwzQwRrpKmU3S8dQBXpCwThK1iL0+pqhCasWx5hBksEMLF1VFq3yoBT1BwSNtxdwe4guJO5QHcSsopE7O7k09mblET/YBtURdyIwWOCPN6kjWYN2UYi9Q8VQjKFRGyhocDkA8xGzhX/wicI0uOKG0OsKuAM6FydTF2Kqggwq9D93mhLEm1bKap0ixLbXOysrzlF5jp0QPGyXgoV7FGNFhMYxwaPMtOBypQFLlRXOWEWQTthcoiEOGBipNoEUpKWnv9YDtkMTwpP1jk1VMO7Cg40W6c5KT8Olhm3Pqnhh1YSXKF5IGwob92rAFvENWyvaXTkBmpTUSz/D6O5hcrrx3Rc6VVnSKnpfcYYpHOIWQANn4V4RD0G/awj17Iu8JjIW4Uc/ZIRUZnmMofQ1aCDiXpdxd/+ubN+fbi6zdvLuZT81RXP5Z3JGcpboIejtxTrRx1iJGClPykPJkifXe+3eq9F8SbTUBio4IWN7Rx2qpycnSomR21BSGNoLPIwHr+7WmCfMeodxDmWiWX3s/2Yv0iDDYg8ItM5rQjuhwlMlRz6PQmhv1cC/0MzwIHCflEidKwUboggG3VCP3r3S9cpHJ1iEtHQ1B0dAwEVvIuy1jChmRuflC1JlZ/lCjdULk/6wwqyUDJBBfUYfNVJLF1pcIxltMAG6aH/CAiVdYZ7og2GqOUE2igRNluWDEv8ky2GnVejxpfLGasz8haL/KcyVyDUkDlLZgPpI+rTCRr0Ixso1wMO6kkedXg7r+KcCb88PHqn1efbBeZau1KoRahK61m4Szq6sK1BJqDP2bVkSjAMuWgXRB6bv1rIn4vSaemxSjAmrSWmLxv5rcLMlvA7BWpLVuagltMXhN9QEefLfy5LbzS07ULpkDIhg2mC9CAgOn3UJJbzwA6pEtZ5frLMGBZuSTHMD05HadcwEHF8xLEErDR8o+u7AlS47pEbCGOEY164iHXXZpobUwm5/araURxioiMRwUrvbfbMPDkPthUNjSmZeru19YQIEVs//2NsutH6FMtzU54f3u+fRHgIluH30IoY6G5BeR8vnXUWkMBeZVgosLtI29OpF5OVkL4zwr6R++LzreenMnfXPSCiazAwYcAa5C2BQwlG5BZAmjVqpLBSaAwxzmAXTsiAg1UR22gPLahgmC2RE0+aSeRZTQ5QXAgrISerCAPcU/1zXZr2FJdhVZdmSKYQMDEs/F02RX1CafKup8S8QJ4mGeySM0fyvIajKuyhnqcsRI7VMHCD0meewJmwRxI6sE/csPVrP/tOV1//aJvQcRyEF02DEwCKmNjeekbJlV9Ek29UbGGjmHov4eFCDp4kwQpdGHyamnct89fweAl06ijn7mS8Uft+3AbHUJS8K7dLCtqii3auqx4mzK881kjN4oKEOP4h6HjNFr9IK8PgySDqbsqh7oXbcM/uy97krtOdgQW5OwGJMsBIyHjGzF2/peLo7UDaMPfB4DYkqPoOY3sJUp12NV4S+HJbjMS2AUaQTcYcIjAJwqOLIEUW+K+4rW00AsXK2Uw6l5fWGVXqmChj6IPCW3qdp7Fgt8H0/Q7Dc+F/bA/lhlgWDuX9qyst5g3tLotfmYheYHrLCedSXpm3hw3mXZf2Mdf3qioUpoPGctmYk+Cp0isFdlATqPReHKtHamZwXG/8CpBLNW3chI3qs3UvZrU15BlNK1O9ZIZwJMZC40jwEjy1qKxMWDdCNNEC+VGX9Sp/zZy0qIqmyFwssIWkXIPr5ZIXsVPehd1UyAs6whuU8yndptWFCX/Tr1wUG8ONFGVp0FJ76MZLBgYpxxGh4qubt0gJUdPR1ENjlgHsJMErgOS9H/DIdjEGELhLCIBN48Ahi06NfpsnsnYd/3kQkJX0DSYgNVncW2Au8DpMnZAowj2+vEVA2Tuz6XG5o7WgLn/znniCNlSzxj0n93Xcr4OA8j6JGk7yEviwBTAX2yqdYb5UqXqbWVnc8OCwzCYlsNusfCqWOYL4VWjgvUHB5t7+gAy5ifpZH2IckfekoJmpHNm4qpXJgRyk/cO6eB1/IAZ0F2FdTR2kr0KaBbftAxA1xz1Jhliozff6MIoY3YOsIwHY/1+wIBA/xdsMd1gdBn8e2DE4vunBXDx+3P0pfHaEK8KxmBlnC2wytet81fC6ZX4ClkVMa9Qye4Vj88IQEtwd0EcZe5Te6qpJ578MI7x3imOnyGexJAZQf9D8cG77NEhjXt/40z9eYCc85WVEOR7VHSx/LT7AmFWK8hZ+oRR5MZxAU1HHLvqBw6R/qVG+K45iHuYD/jUeCnlScOEWFEcp1USx1JwgP9pGhNF7bnrNVZTN0CViovhIKUZ6XL1IigW7+pg8XBPHKqXItfn6ucE6lZZci46TEn5CXFd3nH8BcShqbra6y85B1eRCycCIdYwBPIvF+jkXeFavGIeqBdOKXlYbKdEYA9TEzMiT9eoCimSMWobciV2wS0scf1LAmoRBsI1XF2y1w2DteIqMoXmj3v6hSEJBbQhof4hhXxfSEJ1E2e9MSShGgAF6DeHJFSwSOMPEmoEIn6TUbbRhe+v/gsD+2wqPCQAAA=="
  },
  "scripts/drum_kit_song.py": {
    "sha256": "37459879f0252b4b7c623176bcf30c9505d00296c67d17907177e80c3e20e4ea",
    "gzip_base64": "H4sIAAAAAAAC/51YXW/kNBe+n19hykUyqzSd3RcQzChI1apIe/EuBRaQqCrLTZwZ08QOttPtFPHfOcd2viZpWZiriXPO8fH5eM7jnJ2dvVXSCtlywiRhTaPVAy/IvbAkV3XDtDAK1nOtjCH2wAk87hOieWuE3A8K+aGV92STnp2drUTdKG0J03vQN7x7/h1UV6VWNWmYPVTijoQX1/DYCZlDa0XVPT2JphQVX/Umc05z8Jc/Wup2NIQZv7fpZArd1hS8uuP4Dp9Wq1XBS9JoDv7wWCtlE6NanfNEtbZpbfIqGQ5LS1UVXGfvleSJ5qatLGU6P4gH7tdWZPTjjw3PLS9ozmQhCmY5NQfmBft3mpdcc5kP79ZbZ2XuSobBcC6uU9hcVQ88Xidu0cvNlr3eaNlZFiWJZ2ciwhC3e5bF05P1b7aaCcPJL6xq+ZXWSsfR24NSsDSkeqiLYFZp4s2R395dR70D3rOUPwpjTbxk+nsnQbzEjrS4DZH8Y2fYKsya4Ro8xOLrfcDQalFzacN2n5N30mC4Sc0tgzwwcsdLpXlIuytWeXTGm4rJHSkUkQpLKueNJSXnxR3L78n/r/9n0u4EixFELYzVti8EeOXzNpM/TQv+BqEMWyKtFCtMHIP8RTS8S/FdhPqsoFjw8XowAUeCUl9Q74rwYuiCfzBUZk6zL9H0I3uIdsbyOlswadzrXh1ChNEA5VQYiq0ar7EacBFNDKsLyb/skvnr5S+G1MIgovgicN4DBlFfVingQMgzr8wo8B+FPXQgkf4mmu9wr2llrxEFnraTptX8j1ZoXmR/zgKenEQieSGmyWJw/pps1UfI7wgBMe2d4TZ+SiWreQWFD/lYiM7Py2FInLm+Wod+G21YcTk1/1mGS7NdXapMW8cixQBSI544dB4UORGSPKVCQhF70W/ffPnVq9ebN1+8evVmGksXz5n37+QDq0SBG0CKNVou5hjxckc8uZKN5z2x3s3rvxd+vgPW865O9xCTyFhmWxNBmKI9l1yDdhGh66eCFkYaD5MHxTce+dCXFywN8ZrH6SqMCEgD4JAxZVv5cXa+IX0hjvwYwNVvexM5fLTR7WcZHji9/vHqp6sPU+efEXrJr743Ud5jMExWUYJHJloI5E2ktNgLySrqxxROOqgZ3NKP59QvdFNsiNxNJCQMgeh2bONl5Zc8/z7YII1gUnU+E8c7FkbYcJjlST4GfBggRe80wtvIz2Xthb5+D8Ymo6xLd6+44NKEQMxcGqdhkOSPOdeNXfJxYu5TfOwV+ml7SeD8EAmmj+Cm1Sy3AghBASCXWwWLMPyNKPzY7vJKdCvRew0jWkiwC2AT7LG2AMqJlIKJqtU8Je8BqAqY5vUd5k/J6riD6Q1g0u0HBPNOWOcBQIojlcPkntOb+dgOhBEP4lhmP849b0GyKG1a38OhYv9gsg+6RWIHqEjVvXsaD9McwY4HMtDZRQv4P4Y2KsVjFnVhpT2+RwnskU32HQ3p/zzlyqx3aXnCj14vTPdgJP2oBZTz3dHyAWen1gCS3bhfklyakSMsnvQ3WAXs/LR6dhg3RQdwAdQXO/STwK7f7KJ3GskJbMPkHpB8EfhGDT+FhEU/xrg3nA2rXyECPgd8GJiXjvC2d+jCWx9ND4xTf5wDMwfkWjWz+SEcCPlwt193SZrej8KdK/tmk9QwA4XMvkpKVvAs/fLrREhjIbNQs5mvoig5OVmG7vdb3XRj8jaLenJO+6aAfoh2wRtfTzi+wzXnIkILgX7hX2/WiWfeuFeFYG5u3btSaGOzTt29pZvN5oJZbEqL/6OdE1podW/eX0rTXDVHx2Yxo4lTGffNesHrkdCEOfokjZw3KfShKI+eXHg9d8DELXhJN7yRJQMOgPssDzfFYIbn99lYZDGXFwP8uLlLnV6UQE6AUbnNwkSmbim6vQgO/nj54SqZcb/Jr2iB8sAcyOKJIS6L6PZ8yfZ6bHwECuASlMkpk5gRhhOJlzrkfbj4BV7gTDzHDvy3jP6KqHlH5gjMKAMVIG11jCbZW6pUrfaA0iak/M9oE22fqb+//snWuDs6e/1hu37ahnYCpEoGjbAjxU82yEbdBBt0Z3IhmNtF+BopwpI+5RJOcY5cY61Jxra+ZWe8b6QAF3Zaq4JXwKyqylC8SXHjeDVeaqY9vz5/PVZVQKa20U9AJEZw6KZT90XLK+4A0KWBmQ4CR7i4iqoCqsELQyq3CO0EfESRqx8QTIUseAMlDVVA7gVcwiBSFcs5ImDapXIGdPAMjOdfIxtQpVZLZ261WkFvUIo3OEqzLKK0BhpFabhhNFn3xS291HsHyNf4pOOCm1yLxnUmpYXKKQ1epqwogEN46Tg6P0c8iRJ7bLgjMUnBSwY8wzMa6i7zoDx81Qhsxdy8vn3OpE/52Gh//3YQu5urQEiel191n1y8Xt3aFirjiAShag2QIbrXqm3iZa3ZXqO7wLDlerck6jnXOX6NGIk+c2z+yPMWCjDx1DiDYlCaUxiVwKJ3DLx3uUIdE74OdYDNUvdhkKUBuVmq3BA+/UTJ0mHt9FMlS8MCeNsDK0uDVwNShnoEZh6+qA67q2lV9vMNGgTIAwWVeC60+htMdYtYUxYAAA=="
  },
  "scripts/drum_probe.py": {
    "sha256": "e1197fddd057474b7cc0b9258bb0851a15996fd9e2d96281a99bd4fdf65dc50c",
    "gzip_base64": "H4sIAAAAAAAC/7Va3XPbNhJ/11+B5B5AthRlx9NmTj4+uI1z07lem0vSduY0Gg5MghJOFMkjQNlOxv/77eKD30rSds4PsQkuFov9/O0yz58/f1U3R1LV5R2XpKxJ1uT5UpbFjkhR7HK+rJiUpMEHovacVKIoeEpuvr9dvlO8It//8uqG1E2hxJGHz58/X4hjVdaKsHpXsVpy9/wf4LnI6hLOYmqfiztiX7yBR0ck940SefvU3IFcCZfSrXwQVSZyvnDPPOeJqkUSo9SxlppJK/hi8ebt7bvb9yQiHxcEfqhI6ZqypGykEskyhXvL5emSBvRYpjzX77iEO8HiN8s7Jjm8SvlJJBzeJU3KaGAYVTVPhBRlAetZXjJ19QJIRSEV8OSFgmXNHRYl56mk683F1u7dNSJlRcJjmbAcGL8ML3BrxmuuV+F8oP8GFssMeaexKuOkauj6fd1wyyRhlTLH/yDLnCkwiLsXwZPJQYAFkqQ8VqwQKFJIwEpwu5oJMF9ACqaamuVkV5fliZN7ofawKTkERBas5oQVKUnyUgLnvVjumQpIVuZ5ed/5AStKUvE6K+sjXoiAB8DLgFT7mml3QR7pY8GOIpEh+T7nrDbS7YWSAVEMLguGI2DSXGpqWYEJyB1X95wXhhGHreii4JxF/giCl+RUguq0sx55XqZw5071MgQjPS0Wi5Rn4GnJge24VzaqauAGsknQm/y11iIIxiNtqDgDK90BcQgORvVLrRDrbuG/RfUafls+K9wY0HsaoH5BqegJUUv7w5v41e3rH2/e377y0RtZnezFiZsz8QcUBqYAq8EqSE42RgYdgiGGCbgNsIXo4elKFHCiW93xgtcMLb/iD6B6Y1nzsmU/+elvk0baGYYXFxfxTazdNkanvVjBzZr8C9ijMzY1X9nfjrdhBRc5CX4fHqsrDDTxMFjZrgd8RUacjp2C/FDIGBXr+WuryfC+FoqPCYN2x8KystbujviUTY0N7JXRC/6Egb/UyFpD4T07Gc3ov0YawZ8vvbbxeWtS7kFgq0CWTZ3wwAbAV4FUrFbRRZA2xuzR1UXAH3jSKB69hqBCfi4VsSYVZfRTWXAbMFOO0Qazt1f5IWipzE9gJX3zCq88I4G/1YxMgg6ZUizZazJjNFMeakhIhc4obVVgL775NtD3j3UdaYlt4Yl5cRJ1WWA8uF0zr7p9kC440+ZAF+ltifNyt+NpR3lkSXxf1gfIxfd9OnANSHs9Qp1sax5D4ZCQs4tWfJusNRdeg4L/2wigy6qrF73dUF7itq64ra9//Pnm/dWL+Lef3/7j9u3g2h33ZM/T+D/lXVDlzKgmKYtM7CJU7IqaB7kaFj7jgy5UDE0XajoZI0WIFUh69j3oDMoRf1Ce7z+LTHnt/BVKi+TkV5Y3/Lauy9qjFlpwyRVJRQaOJa/Ncw1RgVWkzFPSpTJ9bANMGCn4vd25OsE2LHbGR4wjhcdDKmoPFQ45P9KlkT8IqeLyYBzZUEMaBSVFHyk4vmqgsNrMCg54rmKbU+naXG+Y+DRSAFrjwDFCIwcfYqhJuYzRuggiAGpcjvYWpcK9P4NngD6VKBowiCmI+AxqvcbqplFMsm+KA9ZQCCYwmoIMUOxC8uvNLTjkI5TaHLOK4nUBpz6G5KeSoN/BfbuUjns1sIA6ApUSiFffvfknVHUmwP70ydS5NqiGmbCfqIwSjUJV/dhZHJiylCkW2YC2VcsbxPyqLWbUph/9b5eC3B/+olcHRmmICAm6UQSz0TBDjvOVTkijxV56Guz9C3ldc/6Bd0zI3aPiUtuhQFyTiw9atoDc/isAMoNxULM1+CrGbV1W4TDz1+UHXkT28h5tedOvR2KFsskyyPuQWXgNITVgU9ZiJ8C64FFZvGdyH5k0OLnbYJcB0GFSVo86kkfEgRHOH1ddy9q+fRZNDl9Pg/ttq7NkzwrImQQMicrBw6n/SSOZg0Yk6GQbOiKlWwhe7BjoGmLVSQi4WktM1xNJnxZDjmFTpVgOTQKIKCbJAn1RI6vIuXAn7++NCKtCx2hj04Hzarr9GxRZndlmfHrqz7auRL0aYwqpzUfToHIrXeTTtqbb3/7cEV9HG7pcHtnDEorHUnJIQimms6sL+Ge5dI3K0jQqAX2ply3KW6ZAn/IaY7puFWVfUlvne+qxgkzRjU0dRmnVo9qD3jtEMKeZjf0Deyq6TAClDQqsNyyaVm+btmfa+sFXjsPlejuUtEMAnqUJAD9EM1jCCBlA8U0Ouv50AvP83IXbK6DSzt7c1zrViSzWBTmOexnMRyuYq56D5UCAjIxtjJywBEbqW8v/v9xcjm7cOWU046fX3d+2og/N3kGDCOPWmzo/9uwOStMAXOIcA5cHwN8RLMXW36NvLwAN6ya/q92R7faBd6zNEzknMpOC7SfaIZO8B627gSfDxj96GSCvaAAlhrL3ElGvT6OTzq9bGCQkLJfOCaFwNXVhM+JiRrs9rOdN7TTpNv0BFFwM4ryl3HHlOdwFNcXxA6th/cx54XXEGwo2kRCbz6LLUX2fFJ5b2JXgyKPUSOpY5RyfNJACLj0twlM0PQOc5DoD4MnrGZec9sG0V1CwO4w6xO0ZNgGCb90nDREMUrskryGD6S61XuxcCfWifQ3fAxvzsodLkcA4t6XYUIOuYlsBt8+irvSMXn1OkxqfG0nMoMH2/AQgt8hEYmBkxiAFpT21Quixowxcl6G3wNlcAabdoYKpoYA4mXtp9tHtwA/RoFDp28HWJGubURpdXwTTEdr8AG0mRmlbldfn63VAFZOHWD1WCNdzvisHrQLL+6O4EX9Dltjh3N+tx+tWR1sR8DNLDojVtf/qFwYM2BZgPZvTKTTEd5AlBpiUrk3HThU0B9gstAvQQgF2UHGrTbM+x9iRojrkhAHk3F3Ddty+ePpDCN2Zdw7b6QJ1DtDCIax49Iwz6cg4QDic9IThEJxwxuB4h5AqjxJy0QxQvUlUw3KTH5wf2naUDg6zXa7JW3puC8dBtsDI67+7Y9CPxVJ80PF7OX6NyjPdgU4g1Heq0Tr8XEz+Urg7OYl5utIntsL3pLaAazZ/z87lhpn7uj+M8CzhQCe2MzCJxe/nmhZ5u2yje1VDPq7TK7NfT7j8P5GwfsAkNe43tJp6oOJL+0gzcZ5vqp5FZ/uRDf1Sac92SLPCWjgI1odzzWG2vqzaLApOrgEhJKl+b2KHiJ1aT5Ak0jiDuOFYVL/kjHYC2XJuFz7PFyNS1kmQShXsGMQlTjy9wYBzZhwcvvQDrztmMh0OLv2RjrsvQiEAU29Ds+xYcRzlLItSqlTgWHWJvDiaQK/jP2LQpoCkCIhZRoOMnsoc0nr0EcV+op8GdtBtrAEW0lzcgXw5aAGZ3+m1y7++OAxOAV3421mUPN+YdvAocIE+U1sD23fFLvLt73OCa5cxDhuNgrOzjh+g6uepnHn8cyf0Sq+F1H2cM3l7nlHF2SHWc1CW52WCqoj1OGbA8CwV9f9QD389+kg0NtUYOPOHhFeKfAcV/Fb/iRCJSaIdbv0ZG1soFWjiKKMfEWR4+skP4xg/LMXx05p81EtP9M9NJXBWN7pdbzTau0x7kTW2IcOBrhk6Cz1nXP8eKRYLSL/uTlFE4/gIIRbH1CipitwX4vCm3mlk9Qafai/lMqmFlieK47RMoP01W0KWpjGz1F7b5aIS9dwPgiNjCNHP9c6hHRhvLrfnWJqK1GMKHjIh0q10d6ytomkvznd12VSR2XlsEIDkjzGoO2+kOPFYv/Z6pJMjWjxqztGfmdv7XV18amv7CR87WK1GqBwlVHmAppyeu7gbKcxsCfY879nr3S9vEJ6/O2sVV/eWpmT2dHlO6TgRnr3pxZz6bVv7iesxUL0WFvdJr/0myEJzzX6gWmH1l65R/MY4eYxYCPbunGjq80PA0W7sfbqcGXX2MVt/02xnPWBuEuEYmPjT2Zr9RIRjUShZ09dQG87jnOtOMzqckHpjRrHbcwP13uih3Y1acB8pY1V6I236CButhOEYjRll6PQOlOGeP6RixyWo5llk5JnHYucx2W/a/L2BP15oheeTo5BHxNlzmEz7mxswshDzTneFYHSlwHxCskOjoGsdXSs/+cjaPpujh2M07bYmLbUj4yrk5j4uYeEbl4b6rTrgTeVpT0ubYyW99vOwuUTgOJs72A80sDj8RKPhspaj+482KCQsOJp25Mycuia3hCNHMBt0k6JeXoCr/w/cELTpiyQAAA=="
  },
  "scripts/electric_full_song.py": {
    "sha256": "1141f65172b7916649d87ac6baa82cfd9115a9cf7afe853d3c2b3bd8bbde1b16",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwWlB3I+tZWK+SgxY4s2Dayjozg68Qo2DHzHW86+Cq11j0CyBZh0oYTa6Pz69auRjqNg+idXkZWL4YNiVoITj5zFwCZjOlKUkdsqb6u6ELsn4O1XOcIIOugHp7McMlF5OoaUXDu1SGGs+nufHv0XArS9M+l08mVbBh9vpvWpHQ/c54Sy97DOYHkZrPqdUR3rBGdho/I0oVMYn7TbRfPsodO+UfbJrMyS67w79999/sU/PTl9R5oQvv/17FP9/eH6/S8np387u3yrv06vLs8vfn178dGxzPq08PRmt7g0BkCtzbntoDnQW2cYuXf1Aa7Oz99fXJ5NnWPY59PHk8ub86uPH84+3tg9Wg02YOjuCXbjkQPMwzQitB5gV+YlGgN2FLZvvla8JiGf5uHSthypwcOcD8a/K+/iuSD46ee3J2Kd1N4I5vgoIK4xRzrx9Prz6MP1jZgHcTwLwnuncRadSPHAuItioFgMAAOb6097JM6FqxXm4JqpNd9ZAeOKxHAxHnWgvRPMhZ9z2v4d7+Bau3knoUSQ6S7r9ijDfMrFr0A++iyakcj9OinkFjKegzQL8gWI68PD+wf81c2JcTszt64wIMbmIoNJL6TtmhUtb7TvVAjEgtOMaOBJvAirLYCAGbm6WKExLMvx2M7ttalquBP2Rm/VpxF65qPfTs6mdbHDPF4XS8svaUXDi3VGmdlnwdUGVBPYAB5/E0ifbqFBy9DS8t7s173ebowCNJrYb0ZSNIXBKENrOP8X/m1RkKkV0j1ue4E24fgYhvD9FxFIBT4zaSeYGkdnx4BSzoB6/vIRdI3hFIpkF2L/KfAPd9xWlSYoDJn2rLbOLCDwlrLuT+737dSaSDuSZGu2d6N3mhRa0NgPujag8Hb0pWsuOgQAA1qIp5F/Eacg2NCT0UaOCz7cJlXgVZpiq+/sEi/2HKjC64iNO87GeFoBorjy7MwR7NtJJBlbhCz+hntqGtlcWegqgLQZlKmb28N0pKcvtgM2kcyVVs36nFkcJK7ZUisp24AbYu1Te7rRmJgQ+T2dGRMXUyKGx/3KPJma2ivkahIkU0OEXL+1w2cD3mhUT+mgVbMAa4opNlhl6hQk93yUe07HybKgCMCzLqIt4LpFG1FAbACjVoGbazT9dL+m16Q31cCsJSZbvdVtdwNEcPUPCN9EzQGK1YZ7qoO0bNXQPmgBUUMPKGhaA2NYUeC0JoE+pFartaGPxW8oStY5CHqK/ORTywvkHgQJFTJQIda0NgDLfFvLE02QhgeZ89uIwahSVmKJpzyjn4hSNBJQy1prdUgVnKhyXUydOVg6wFtDnjGdO4+knugLZTYaiL7/BN4wNTUKhVCts+GKMiq2zv6NpGZcKzR3WDJ28sthukYmpITysZ1WelGi6P8jD1nlFAMQrKspOrnsX08f6fBPw1aycX/I1Vr3DycY1bydWgSrjA6xz9O6YnqNwJenMhLkxXwjdTFwEJaVWuLVeUXHoA8kw32wkNq5/sMoQieoiXRyh9yq2tWPsToyocjNsC6C9eGcGyUfvFX2GlHoOiv1hcpJzEj8bgyyTIT/A8Ko2H5vprqB2VjN4GAxwJRWmnHr8T+9um+vuRvnILcAriit+RcV/xRu08PURSUaslXNDSipYh2X3leVOQjBCrbo23cA3sRDA9qDPbe195tLGaF26O5oI6oXdxbyKPahq6m9f1PZOZY7VaDi+zpfKPDIgWTMVFbjL679t2fn74Hx3nK1DljfatMy7OZU3xqDHNuwe36o4YzWL5lwEzORw8b1/mbasPplOKWJoIkdbXJt/V3VuxqlNGqEqyzjZ9SRr41b2CMLhkPfIuPx2D8hnPtY8D0eabIwQ7UaG+m/dTzTVEiRQqkw2xQDjuP8TcpMBKJYgcYQGLlGbUy+NoNDBHPMjVxdfRjhUrB+/bRgFRT0GAArfcJgDbq+imK1kxo7aKxRH4cRB7ILC/c5asKxfwoRmfhOH+7bVaANEraKL3eSGpM6jdbY0FiVOkDZrtY43FWwMTS2v7aXNXH2LIKZpyBzQWjNsFpwoqubrUJm4Ou+iYM/Fp87vfrw4eLTzXBvtcg3BOhYM5EZ62sVYE0B32SxkFFjhypiiEXtEdqGNlXpvIKFLNawSEaXaXkOujIyxTk6fY3T5tgOVhZPN2YW28G0klU8aozexvsP5kKrEN1MtErXMZ1p1U7YgzBr5mNsRNcWcjR4n5nQeF9kYjiRms+xcp8fx4DBUNAhwaUzSZtaEWkdG3G488+u85+06+dNONFsO+jiyL4eFdCbtHHjHZR+E6GfMHEIPs2331ZfoIUe5QzJJMfcFMt52Ms86qGEleXQsS7AIPeEHjx5EUhV/OEy3jrq1zGumo/ZeIXzNEPdkleGWcOWj9VJGj3rc3V35TgZPukqagdY5+2wChjvhc/BsNpTJet0TTG6+lkYJSAjmYERjaxcPwlDajCvwcRvJ2dUfFIqitiC+iB3xBPvgjx6ABSAg1IS8Zs3OLBBQdZLuMVnZaCJEom1AlxmV0pCtOfsSRj1qGdGZY8nZ6X4+svmGmGOoWgVn9rlubhD24XDkEVCE8ksmDbTV996A4tztCqY9IRa5871+5PLCb3fezQb337XTP19d/dUGJxSAP3X68+ieiH2fACWj3XQ3vnq8kxcn9zcTOyd6z0n3uv5ky6+Hj5zOB3jZwqtXx9WNLbjjFbhD1rn3ep/XU5OKtEnYeP7lmYkh+KIU4hk8R/hLKfOfvAAwFnD/h/UQUoiFQqYseCT+dwndwxDrjXdVIqsLsYBPTjdmY7quhU1+d06fGATEKGgZs/upopNa89OwM1SRG5fIK9jyw4aSsomUmskJSG0JOUkhDGBIgcTkfgKpR5+64D2LrDkDkYev7iGPiWRhT49VdRbGQYmJJVs0pCjkfU5YScMM3b2vh3fWU+j6MlTf2SzY5c7FmbQQKe4qrFJqpdV8HtIoeoGyPQMk+MldUxG/o4sDo6AlW5bFQV3OKJirXbncwC1czP6QLwzOzhoM6i5Ykhqx8ICKAe3qZwE9BE5vLgEnEKWJRW33N06PAj1W18vT7WC/tKgGBVy/SKVtaqVJdBvYbGuYNj3CHbXE9j+ahy7RqKCZat8Arcswf+h0D7qzVgu0uZL3SBuvKjtbAMGNfhZPj85VV/NlmSN4Oog6dC9sprAxQKGLv0aEtzTv74ZjXcoelbBRzzrYCFN15NNkPiIgpFFJHgvt0R99P6BzEVoGepPMBgNojyFbwPdwbMPYE5Ytzcel9B+wMN5oQ3XHruV7ayRKd5qMBGTj5Y59DSazoyPxPANnNU7wzIFv1BfNVMdtwcgjHIQzOmKeBtG6SQTQeq5y31OKsK14q240Ih2FhXRW8qLvfedkrjfy++XwWThoRrEH4Qt+kGvHHk2XZKGAc3egsBF8MGnAQ9lCODW+LQNNCimQDFbarEmesSN1RpSXvewkG8M03zUkWjtJzV9k4jXeE4jK+uYcnm6UXslJOVQC9K9K4Q9F2GbBy9CYGQYwiaDlz/xqlUTlv6wThrpbIS5ZpOQ3WatfVOot4vS7jqPkdrVblWcyiqyG+zVFC9fslF+3Vq0URx39yzALkjJ4FO7Bb+QQe/O1uZWVnFgq4g6PVG9Ua60CzqpaOihwK6P3xdJrc/dKLW727HV/sV6ehvx2V2b1W/znrshxmA7m7Vi7M9fqd+FqW20YSWqpl2FjUk9omJfU/FU/70r2mzpYJ29s62cTq+zM2ydyeCen5ADL6JtB0bWbAuOYmPBnaNesrBOs+xftzlo97JEhswI0w4vdekHDH7E5K4Jzdj+rj2Z3rCY03ruCTPLHKS7aXszNOTFtDF9nmS+1XttJaIOdnuVL8yi9tPu/yKDqgPcu+z9tgNkS2w9tRHDarpn/EnemaXgzbxeNf6tkMZox67AvyVVCMgVgLHUF2Ss+Iv4CCSg+NVOlVmseHPEkp8tGmkme38wC/2i6+g3/HAWE5GnYhf0x0zhi3eSL8jWvqYeN5IcgcS4ue9HaQj+vTUT/1kIP9BT3Nq/RwqZXlNOAbYMgNGnu2IE+h/QKG6P7/auXP2zGdXae4dzTME6hwmLWj7/rp2oDltPJnOmvsV470zr9Zw9Gf/NqalztVJl6+mgiZKtgi9qtV7hyx9jwe7YQYeqqLqGkAIKOcU8Llxq/8wqCtMzUR+xIoKbz9foEd40imP0uvQXrmxyrujbYHkkb2DXZDQqyGkMkoedOKFGQFX7XxWRDZ/A7FBUr+H4QZg+kWRrx5AI9htkO23FDPeoUkbViYb28vWhTO6IO5sxS26si/FNNoma9cegFdFzbkiiTiqOvDWWMfq+/yG4LgG7zQmQZRWWI7KMJefB5wpNH31p4uSD/wF4HN39Ik0AAA=="
  },
  "scripts/export_glitch_feedback.py": {
    "sha256": "b0157370725c3f0b8079858b087c8fe118443b2b64453522e64ccd90ed2995ea",
    "gzip_base64": "H4sIAAAAAAAC/41YWW8jNxJ+16/g8qmVcFqevaFEASYTDzbAJmPMOHlYwyDobrbEVR8CyZblDOa/78erD1k21g8Wz2KdX1U1pfT6dOi0JVa2RNRq28qSGFl0bWlIV5FCFDu30vW6kKtSVqKv7er9zW9E9KXqSNVpsq2VLXZEy6OSj4w8KrvrektUW0kt20LmlNKFavwzQm8PQhuZ5v81XZvGjbC7RaW7hhwwqtUDiRs3biOOzZNJw75XZRr/oQ6VquVi8enjx1uy8Vcyzt0a58tcS9PVR5ktc7wuW2vu3t4vQCp3L+WqNVLb7IoRY3XmKawItV1Xm5WxsuHyKGq6XAbmguDx3U/vbq8ZtNR2rSqgvz9klEB3R9kKSJ9Omp3489/+DvVoZSX3ci8WN+8+vfvl+vb602fwnFErzJ7bp4OkjFBwZXVfWNW1blqIQxqGnQZygC3My16LcS9qnYPzg6FsQWZ/dAutCb8Pfv1DRsrS/dqdaveq3bpxbyQvOssnr6alRoLN6UIt2m0vtpIuIRFchNTCSoO7XQNtK0iaLdeeDTjC+13XGUkEcbu1tM65RCPfuDutJSVcr5SajHcZaeURK1ar7Ra/ULXd4XcrWxnE9v7lyMMMJYTD49CmgdJlmQ3m1H1r6DLf1t1DRuMzExb5N6txkjvz0OW57sa/vXza1KJ5KIX31bX/nxsrLFwMojdWNZK5kJBw9s2t7uXSE3Px4s4iOibsroeHVOVDIq87UZrMU9VSlNzKE0iDfWkz6t7pIQzZbEgUpaTrGbNa2l63ga3g8n5bCwXl/y7qXl5r3emM/tpNLPFc+2C4b8ucfJa1LCx58wZqJCtHdmW71QU1nk6nwQ+khxZewb8eRLHPcJcRQMOhty7UhLabvzCSvHfz9mp0k9+lVtUTKChj4ZIx6KCwSJR0bf1E4AgAK8cZmFetlRqB6gEobMGxSKVO2NwKNfrJhI8EFViaoAQLi+HIZH0RTZSJ9ilTxsWhC6Xs6BTKHgAYSwIDw0PJs90M/LEKdrXL4VDj0cdUqgUmhHPLmRmdu/hl5y+ZVxlL+gpk/Br5nlwlmleYpCPk+w0ZtHrZ/r8ZpyL3fiT1wwYknKIHIu7pK/b26p4G5rT0FthMPTXzfgGUOo+hmfvGKD3AN+UL98PmKv6+SOQAJuDW5WUqaXeVBi/SgTGDOGeR9adJZDnNJn6en4o70wC8EGUIgVmc+Tzq1exyaYq74HCGDrwFk3w72uIH0qg2S2LdjdB/zxKPd/RRHCU8p+GVRgQYeg+dDJsICnDBcU3S+1dd4/OzyJKnAoIYH1yQpgFDITAHNiLKIe06DP5CQ+GQgyO6Js/MM9llwQx38QYP6RIcnoEwjbrjUWdnpJP/pFM+4MIDgwZmey8/VBz6lx7BzoRw4hzL9P6OBiOOdL/OgJ85AEtqDYrKfYSbbDlLA+G+TwHe0dK1M5x/ZrX3oWILdil2yM2y/A5OXvXGAantEoIKuJKpERBueYxbehYY3mMkD/l5kMpxNCj0bO81n/KVYztGwCTTlF0AzhK0HPYPCB7IR76CdvNmXyqdxWLO51fmcwXv9psPojYRR4OoHibu7gc7tKDKvGYZX45mAAY2MytMy7pwPCaNVSARUtgclTdpwFzK2Vzl/xgRPXGTi8NBtmX2hbqH6doTo1F962j42VPLr1F6FEGqRSRuZrCVljkyFbRK2Zd43gMADqcDA365tOz2olah/y0PYLOJl1a+tgX2hGX4wySl+SPzlCahdPJr18pAUFqXt83zt+PGyKOrJ1EFiYAXcN5GcFc2OThZv2WRW9XyH3lsTKAjr/MB/caNQfvPSjc6yjihM6wx6uzFRV37pgF7MB2boeXaF/vPCcfzybrsUsCsXwumCzRjbNH1F5Sa62n2wXzpvRgDn5pTOmK4VNfutwFm1cCuoyrkOA19xcWSljb1iR8FUDdCvrsVl+K9SZDPprYcJ0hNk4lAIfj1gmgIWtGYKFlyh+Acccu5xkVZx3bpEmGHv4HqDJFf0NqgniRgsevbfUqZjKJ/07U4jAu208XuskjALZgqQW/IYB7m7GN3ntxZALWUU1e+MkXgkFJVDqOBhJ3rdrAMxeTkdqcMQo/UQDfplggYL12HxzxeCpeDbYdSUhWgIbZtZ5TJaUg7Y7c5AgptRKsqdGihKmIpAEM4Ckipjh40hgupgs/RZocS2tfYsenO/6MOH/CbxauMnpBtAexAIyfPZjj38w3/6frDvxFGPy2JMOQBvUUtR8BN4Owt9I2PK3bG7XKe/wKF3Is5h0wGbtzvxuPnYtIURTZjk9JA0ZP+9AZiopdNHz5CGnXNxnekP7gq09tCVtZlUpei0AXrobPw3zY0dJe+c+Tv9NZ36jd+JyulKbTyPfWG87IrOF9ObuaiLLmIVzLq2y3KnINuXEfy6lFI//8e9YgXD3v8ZjEfox/byfqwoR+rCqHpzQxb/BhczYn7uOtq6fMFffWJ8ZvEhVfQk0Rf8xkikvA/joiJnZYrt7wqTe6GSEHj2HiOXL4JmefCN4dJvZDI+O9S1WR8TiZUeNng8ZwiA7oPTbn791d09zt5ulv/8/48WC71urHV9a+FJOOHQwMXOxnlNHYj9tC3d7c1ZZHu7MTPRq3JobNQPWLe9M2D1GxAl6jacQGVFiY4OcQPnZG7VaXYEwR++lBnFCOIph4lIyOAgx6z7VZgSrqALyYn/xJG1VB2I4A1B6mt29+LNvdNP1TLuYs3zv23Cc5ddHEee6MQaov/AaMGdER1FAAA"
  },
  "scripts/guitar_articulation.py": {
    "sha256": "ec679e50b1a6e042beb973377d3ee79956e3ceaf8648cabdf84d0d16f8741c3d",
    "gzip_base64": "H4sIAAAAAAAC/51YSW/bRhS++1dMfSHZyLRjIwmqgocsThGgaAM07aGCQIzIR2kiksPMDCW7qf9735sZLtrsJjpYmu2t39t8fn7+aStZxU22gpzdXF1oyGSds2UrDFfMKMFLPWVSiaWoecm0UW1ViXrJNqB0q1kjsjW+rKUBHZ+fn5+JqpHKMK6WDVcauvVnLeuzQsmKNdysSrFg/uAjLrtL/4imECWcdWsnRdoouQDGtV+fnZ3lULBMVsgBQiWlmTAtW5XBhMnWNC2uf8QtvGuSV6/i5zdXL69fTBjcQdYaSN6jThBNzxh+jr1mCZuRWGETxQq0LDcQRqyQijVM1Owox2huyTkJY40WKiHmxvBsZe9H9tgZQMkN1LzOoLOBXvHrFy8nbKuEgdSaim47ynG1zoUKSdfa6OSTaoFUEdqkcu11caqAJZawrwGqblodTFnQ4C5XKE4wYYETOXXs8NT9CN12NLFUhk9gDZg6RBAxu0Yyeau4EbIeHd1cHbyuZA5lmvGy1KmCLy1oAzlevT64SeAhWf/gFYww2AheSzQ4eYQTJqEGx5hpMAZ10j8zWZf3LOON3c5WvF4iDtlbhw1UEC0CasAvX3JRa8PMCtBenyFDmdhPPUvV1mwB6GmEm0HwL1pig+gWGpiRiLnawJ1hJdRLs4rZb5IVbVleoMeWSG+QMA4erJaDR0OPrUsWOOQK3IzpBD3jXOfcaNT9tLeQC0Dy6UO/R0CsyVSExaBTjfzrgjGIpjsWdjRm9GSOlDxEvazH4welpOtdDDnH77nt0U8HkQSB0ced/56wDWrPa5MQj2iH7P+0Vy4yEzqjTWwiKAEdmTpNkxJjI3S/o2igzydsgfp7cwyGm0/6PW/Aef9GFIzPAovBYM5+SNhiWKEbhrNZHxXu3s3VrhMUJwj9xcsWbpWSKgze9loxCg+BmcbCUpNrO51G4eAYRXuiIRBF3Yd0J+L+Loq6G+t0z9lvtpcV5t8gt2ND1IcCYYn5SMxH4mIiwqhNc1EUgJksA0K1pvDMw7XF9JoAjSq5m8E8XsO9xsT7L2m0v+m073eXYMJ15LXf290x2REx8NEs8CmEokjkTxvhd0o75C1fKP3zS0eeiRxr6j1zXEZGcCaP2yanyLMe9XZPxlDa88jkiNTJ4daOnphTu7jb02VHBFcokqApeV3v+MvdNa2q/ZM94PkSYxEXdMkkDwgMi5OnT5n1jTSr3d6DVS2m60LUQq8OwN+VAbIZxkghln0EHDt6ivsvjvGozriX+tCRDSaThrLJnhQYGbzS5LLF8YOewlOB0PAB/81iBHs8GNCOJ0dA3qPZwgCJjZlRLsAnY+CwC+wZRhGgVZbyNhcyeHjKZn/WcNe4MjpmsgCzBcBwzkyLWcG7tDfIqYjo4DggZuJJpNaAmBHVThSMfp+qT11qSrH/qEbRNlraELOpf+fSYv/SCQ4IUYF9TvoF5RTmPgnaWsFGwHYnoL6jFeiqPTXFrh5Tt0EOnYWBb46xMousLV1DVgDkC56tY+ykkZZrDifs+GVMIG1p/FVqK6O9CNkKjEbfk8d/i+Y9fg+yD0IF28BVYaSoqej3bz58TN/dvv/19afbdxE18FxlK7HZy0i2NruD2BrpUfscbEUHxL61QRqwSJbZ2ICcBePho2fu2mnIL22u7rf7u9btcdXc0G4l7na25ke5YrySV6cn26tBrGeDXPGWbzwP+/M4bbJE95ys0ZM6ze2EJ6w9L3sCE7fxjAWXAf7ttsdxvV864C6DxrA3XMOt/UkJFjEBlEimTySEgiOeKBvY20kRfDX3DYR2FcVpStKk6cOUfbVbD+MMQznLTV8Eg3LUXX9jSOJcWrCOF0uwsqVphfNEmvrCZideRVXBT7/xa7VsK5zbPtqTMAedKWFVT9I0l1maRqOXMc9zjFH3JAwuLqg3RwFI14QmUux8oeAYtXYVpimFGdIYxtTYz4mz5/NHKbsWY5e2b0NzO2M++hqt9b1P7SzRPS5Kyc2gVD+rP0rBtzWuNJAlMUvjxIbdfwve73idgtgTsF9EAiuoI4wDsQnJu3HeVo0Ou/8l0J3YDUT2ZzcV2YWk/yu4gcgdunG4m27snl9gzhV1jvIm11iZ/wPdRaJAZhEAAA=="
  },
  "scripts/guitar_layers.py": {
    "sha256": "35cda223969c932cb799137539b8a0025090f70547cc9799cfcbc30db6776165",
    "gzip_base64": "H4sIAAAAAAAC/61a3XPbuBF/91/B+AVUTNOOc3dtlOHDfSRtZ64zmblrH6rRcCASklBRJI8gZTup//fuLgASpEjJziUPMYmPxWK/9ydeXl7+fl94m0bWvPIy/igq5RW5V2+Fp/heeKXkeeHJvBbVgWcez9Nubl0Vn0XuqfoxE14l1qISeSLCy8vLC7kvi6r2eLUpeaWEff+vKvIL2Lb3Sl5vM7nyzMQneLWL1LapZda+NauyKhKhlB2pxb5cy6wl+lnqV/ueN/vy0ePKy8uWSNHkKS7CYbVul+p7x3DAiqb0+8XFRSrWcCOe+shnADeF+6rZ/MKDf7xJZRF4Fa+FFwG10FnIs3v+qOK7NPq9akTgpfVjKSK2zgpev71jMyIg13rzq8h7e3d7e+sVlSYaqi0vadzXJwbe3Qxn8wKuVYZSrWUua+HT6lnIs8w3TOG/ikslvH/zrBEfqqqofPYrKlTT9vaNqj24piYBWgN1FcDA7u+fPV6TVsUDT2qj8kzkm3prOK5E3VS5JmSkIx5QgvFePvhFU5dNHQDJqtkHsD/ZBZqKYS4p9iuZixTEFX73vfdar/SuzBtuoHVADJfgmK+5uGr3akZKwXewZM8ffJKpD1LBF/jDV8pvF89moLSRBXACzGlaGy5zpCVz/00YeOE774boz1BBdJDIQJ5vQlq9RjWATgIjTti78Jk2GBWT54iUBR2/geczukRsF8HhsABZWHZKA/u5r1CnWorAg4/HwM1ZeM8PbGYPfE0MB9pkAvQLbVuffv5nfPedURRRbD0mrJrcX7D1el+KDQvYdV6oOpU5Ph7gP4FWQuP4n2QB6GWSj1nQnjD8x66TOQcS4ND78m0G25DeisbevLvbTRKG1UB4CVLbimRHPtMzOLyxsbeNyAV6jV8VBdpa0VQJKMPaXht+YuOer9EeeVVHICvxIJKmFtFHDgo1NvksMmAeCwxNfjkDL1dFdhD+jEyhRAN4Hi+zpTY3soJQyXyTiZDXNU+2REDfWEfFqjiInMNWrw2G/O77HwKPbCSm+Imr9VHhfpfKCkIPnFYrE3LEg1R1XOzMbY04LUdbrrYYtYisP+TUcoJhPfI6hTGK8XG7nIH2BntD1azX8iHMintR+cbFdCgPk6J8xOjrH2lJn9WGRcOWGcVA2Od8PhnjBkkIDIrnG4g4aVOBwD3koI1lJNjI+wK34nWj2JyVMMhxIfgn0+qMNS9sbnjSozOcR7OKlUiKPIXN9Dr0DQbn8loWebfuLRgiGwigPaR/zSNq3TSKkc21gEIdj5xZwzqOuzS1gmj1CcpFtef18ba+XuG4fZGKLE4g+SiwiD8aAakkZfO7I9oUFON7ITfbGgSA8oawz+a3EPYDhlFfPz8d7YR8BxdgWJrk4j57bL0/hcBcEafAoLJ5y6lUykJJlHvo/bxt8p13SyVLQs932slSuaYbYlkAplHLPVKqBGVZ8L8q4zINvd+2MJY6JmXuqnQ14ckUSMj68T0UJMk2l3+g88GefZE/0qF/NDyDeTBNmWW0W8IhGXinyMHUQvZE1+48u4uQWnQqxFEWGJPV5ltXj13yMAUb2PJTO9YlqgOYNIdr6lSlhR/ov9e0E5KLr/Wg1WFHnfTUnbJAoks4ywSykwEZ7qB50DFY+4hnnSICX5hMJsf/bPg2f9uLRebvUcSN3LBCJVvgrYBzcxEjCkg7doBksGyXQyTiCyZzuAhbYhRadW9Yq7Vvi9bP9bq3t2YeiicoAqx7WxqD0b6UjyPbT0W9tSqm6q2BBa29Q6FRZgIqube31zrItMU6KgAtUJ/nlAZ4sfzRR+XV/jAYseDL00zPGA5HIjCZV0XJjwer2ezP3YGi9nXnYqvHWqg+v/3wqxlCb1gM4jRp5mz+OMPtb0QSKYF0wZGn0orDouWGpN4xE/WsZMCqez8MOsawB9zZa+okhSRZmfEcaks2WEi1kl5/pGqHAIiDtYGUfZUm+40iKRT7CbUdGBmcKuoaopzC64NxruWmdYOxqXOn6yxvuAd/8/RGZaK5c3wJzl6utALckyDB871Cr1+NT7QUbILADAOlEkhVpP6O5LVDeZU83IlHBZXg/+Ak+4xNAyf32ZHlwYx+6UmGJbykeEGKB2LuYWjBsMUZmnnXUKfYPRC8q8Q665H7Pp2T4b9yaNpEglkU+jzITlqPnhGGy4ojT92JDgz6AKktjfWUIzmnA8TSmRrjXkLA3GbyBzYVbWfdZi2yxzZXUVyedfR1gIu8PmG9/EYXcSK9KSoJcY9n/SNaIqbve14DO/DzsClTzHnapyLHnQL0BglVUWzyfsSavBIHKe5p1hynmYlMez+RBLV2YjJMiPBV7Ggmcp4Db5MVK1i5BXlAGt7HeLeI+sQJ0nTFGEqQvYlGEbpJ94r+gffvLVkNl0wQN11wDHK3W01E7nQ16JhNb9kSfGE1ZAseLIt1wbFuoNhym3N9korXQqQrnuzCz7JkgW6OguEiaPKarNZLqBsdBKZ7CUHQAE7hf2T5EbualtGOC3avoQCwSaWw4mn3/ONT/MuHj7/++PuHX2YIOfEq2crDIPaTEeiJITrQF0bvbXZEo+9VR2DFCD4xm4/qdoKZQScf9F9HKUEMRBXNz5PU6Ef/dfSGVZGNxo3xq7gyWTAXAbQybcfIe+kuAQPh9EaWwJS/cIPZ0t5N40aL5QQDJ+RJN7GlM73AzW+w28ah49tPCFPXP4HTJbpxbFgtiIdElLX3E1fiAz1iegXLJHRofiYArjlYNZgSLY7W7AuCUj69zcKYOtE4fpp7X2joyS2cMD9pwAGDdea0NS8IAn1wKEZpvAAhGgOGLjUk/u5WXUONUYsHCxN7WH2p9x7GCDAfaBTlWgrTXCoKDXAWciY45dP7otoRGk6qMog4nG/IxmYjiFo/fWNcytAwJMZBqbYgIDikD3xopsIj/GMKw+ivH573dOEUvAa+IqhKuRD2JK7l1iPfDJ56MURlrtC/qdNtWHEupoS0HEOwPoLRXhMW8Rz8qodhpTKpfXsqddvklS6e1ceOoh50hFAO+DCcH+P51jLf3bKTKd5iOlEL6TiIDgE6J4uh7h5GkEf+fsNs0gbWTuV9jJHBt8I5DDemjvR7/nNDJxn5RO9ugz2voMiMfgjWHGQYfv/Xk4iGzIkdMOfIpAwWmDQzBDPOYRnDvjBqu8L3LxGnK039HOEEOAhPlT+9cUY/d8UoButGttXUCUI3Pa+ig26YggPqxZpoCJztwePpVy1ng73NLHKN99RPW53TUFNy07mOLpCpm9KkdMMoFay0oMl7nFKiOmgwomps9aSlHVmlG1kP8NelG8uMQ7VhOJRYRmZw6EHEdWEkSRcejRuvomdEjVOC0PFw3crjNGZBSSwyeOE3daE4QNpBHJlroiXFOOQ7LnRD1jqsV7F3xmEyhb5fMDQlw12hc/CC2Q6vFZArxBFcqCOuCNV1UsUx6PUchSBRX/PSZnNetmD/MtBzOkgcDWPI6AbhQB8Cyg8BBJGzKIyTK7A8RHrW4F0MBJW8wBXLCJ8vHCF3+CfwTppC8ZkZDYR2E6cM728OFkQoeiZzYUChPa+Tbe8Hl0Xvd4O6qEF7ch03ecKTLVjnMoKqyof+yS+7452ahhgMD3i+soHHUu711mwZDa+pT+uhFV8bKgeI3XE1jT/uj8e1DivopApdhUgjfxBusNewdyqyFPxs6dR5Q/SQaDw76tDqm/E2/FVkOTju5LXNj9I66iKH1PA+LqVzRv6zidKpRRQMejICv5bQ0fONKfzbUrdfKBwr6Xldo7ksJAdTquu6+mw0S4tcRL1VqpY1WvmNEnxvehhXjbgDNUgV6CAG4BY3NdPaoxw8WZ2O82FiLBm5URaeM8B5jvvXU7oigU4qCnvitkUgNmL8OuGkMA1YNuHmFx3iiJqLXLjx3H0DF3ecsgR9All3FP7ltUveQI461fQoXnRuTR2a/VoJexp89qHwgEYkIqfgNQ7VMQug3zEm43ydEjkYJZEMuusGDj55Fpt8EUx4FH0iHWHGfj+eQP2eH3KmGDhHbCLmzP5UcD8XTPqBREMPY3tsqzJ3jOF8iL84AjOnsUzs2J4FaNLKU6jmCUTzpmUE4cxnoZmf+yHjcw/gGhP/SINx8c0RTMuGth1CGK9cyPJqCrC0+No4gQ6gvBqDJ18GTfYv2kVC2FEVGxS8Ec98IFSKdKQlfLpiN+xqHCxsSynzi9Z4qMQGD8OFz16z2QSCiZVkl67op2a/NAiL5t/yruVKJTjFj5C+BFBodCASyDixniVLRzp6WWS7ZCdDTYOpViCllUAb9tkVfnhWhseV0dGtj/NoT/NnGsJgagIRSqBlkVAvijwWx3sIvXFs6kD6QrbCH9jM17Lhj9WGsIJPNOOnQiWVJFw2iuO0SOJ45uwMeZrG3Gzx2fU1ViksoK8DMQUFEKY4+L/ORzExBhScjtUAbYs3y5N0dbZzKZuPWlIHopviycrl2vx8+TVEQAlft5Gszmylz0Jbmdye3GfKfBbwhKQP9lJUYEVwHju5se3GT2yF9fizqqFAf5CG8jtUBd7CFp1zoXiM51Ef9abFVKDSk0Gv6Bnkph+G4KbFv2nSvOjTqVwbnPNVR+gPgPR6+gpo+syykiA9KnXTZl9CuU3nBzLHT62iO3DR/wO4WyrXsC4AAA=="
  },
  "scripts/guitar_probe.py": {
    "sha256": "af580070d1f10f3c21a3624cee0615968aba54ad2037b0806e78c363a74d4d0f",
    "gzip_base64": "H4sIAAAAAAAC/61aX3PbxhF/16e4uA8AHQii7EkzVooH1ZYznqaOJrKbaTUczBE4kghBAD0AlGiPZ/od+g37Sbq7t4d/BCTHsR5s8u52b2//3W/3+OTJk58zJaJUyexUpSqqdBKdruukklpEeVap+0q45/P//ee/L+blTOSZkKJIskzF4vLl1elNpQrx8v2rS6HrrEp2yn/y5MlJsityXQmp14XUpbLfN7LcpMnSfjX/wYBfV0lqR38r88x+3slqc7LS+U4U8AlWMo24xgn+XG665GW9LHQeqbK0Ix+SYpWk6sR+t6cMV3WahrDbWshSlEm2xkXXv1zdXL0Tgfh4IuDPSWLnQjij+jndnzuecHZ5rFJcJGFXUAcMf3e6lKVyPMMiVvskUsSmjiWSFFpFSZnkGQ6u0lxWz5/heJKVla53KqtwwuxiuZRKxSUM384XHs3FMotUWEYyRd7f+3NisFJa0ThIgsu/mzN9vsKN4rDKw6ioYeadrhXPRbKoWJo3ZZ7KCqxLR26UJdglZBTlu0JmCQrpi0txJ/XOo8XaE3UWJ2UFSgb6IWWRygMMZ7KqtUzTg7hLqo1YA5tUgUdFW9xzk+tY7PMkAmuUQmaxKMmDhN4cqs0OuJGCSl+8ztM0vxPVBolllotC6VWud6gUw6f0BDgkMPJEsdESDUwc40MmgRNMg9B7HIU9gAioO+x2Ks3jgy9u6qUVUMiqktHWs2cQWV4pEatIHoixuge7lmWyV+D5VRLVoEjQqi+uBqrIs/QAXHIBXlLyoCdiPJjH24Ms+xxMW/pg/0/glm9e/u3qVTjwzqdPzYD3sJ8a7Z7unzl/1N6/krVjfXjY1mRatlwC6/ZJXJPNi7TuGLrKM2WszG4gdaHW6yQHNcAU2pqMBplEUG4Rb1nzxhKGlqygQQQIOSDsWKGQNQ2xH03a+MiXPst/fHFtvBYFKFuj9l3VMKi02uVpDqozOgPlNwbvOcGo9W/e/fL+7+FPl/+8+uULPYBEOcUIhKz1/He4gTnMqDeAZUllkxkAzSvyFS3SihNTa0O2uXGFRlfGdzpGtOoeWBHUBHw1S/aoDYnamtEXP46dBy1oFI7R9of0jRH3Jer+mnq+MfHXJtxekjVO24k9CgYQAe7KtDyygg0lmzJ1Dpcy2aHvIl/bEKTsG1S/k+tknWQSr9rGBuZIONLNj0bf/Od0fB8WHscSs2lWHBkf0u9JrFaCrk+VgRDKhfNXswvaxyAHn1LSxkzQ+J/s4VDVyzyHdBVJrQ+kmqjWGo9q2JEm8zQGf46ACRipzGsNnr7USbxWpc8MfwIJRGIUZNwiKQXDnR8sAmNaT9ypZL2pjJGBOc4DzqkkkoKsleGKyAr0i4jKDUNESmE489H+IQSechmGhEDtFwfHnC1Z4amI1k9KonJZHUOepBBxJpwqz9PyDHDJLlR7mZ4xQ6PDQkVA0EeEPo6GCP+MWCmkRAygRiQ4+VKxRQA+4a5GPNbqEUOAajUwIpbI3MV/Zo0EPlpYaV/dw75mrWtYmTVaQShlzJ2dAhLSVq6Vm9dVUVce4M8IrcHK0Ar3L0GS257QPqJcRoIQpio+SzIgb4YhJpWmw55BECptrl8z23VucvDO4hLDFSDHCJ/5fB5ehiwEgsn5GQR3nU6yxXRVa3XG/5tlC1qG7orOAVdrHSc5uuJtY5UV8F6CVnwA3iBD5+CAg9WdvyueE2xO7ntDi9mRCM1f64RG4j5n/07umSN9XMwWrSuip4KkIgjECBMLIKzZLnoiYP2TZLVqBikrcj3h/yspXqPfG8uDhxuNOHcgC2YzbWwRNOvfXIevrl7/dPnu6tUMSw6pow3k1v6eqFoAM2CzPWKnxoO+Naq+ONIRHLAVwVLOxqKy+8d7+3c6qdQxvddyOhls9oA2j7fC01AygJOYTfx1mi9dp72nnjoTIiarYX4ZSI2zJup9FGrGEck+b3K012RDG6BPPcyBugqgXoprExzB8zkCRxXVlQpeA+6C9Xup4fqqgvbe8drbNSRjBG/h5mXpOSPujMSrJEMJLf8ZIjqcPp+LvwTNtvj5xbw9vZYJXNj/kGmtrrTOtev82CvES7Gry0osleCCXJQKpmLI/XBwCWYQlF0E3ArP522q5qOQBGAIvlQf2vd9ts3yu8ziEWbAHNGzVYX53TC65WmTG8a0jvmPLoNi5gNxnu7BnMY5UKBRS80W/VR+dPuaTETNAZ3vVUaVX9MTkM+++zNcgegoIbUUmtV8TQKvfaLzzEAsQzYy1dIBjlByF/J12yGBi2m9VnG7krCMViEi+5DQmV2s/l0nMLEqnj874QyzStblAN+Mg8r2zmDQ82C1N5XVe2jo8Wqht+kjZO0SQ/Wpc0Q4IRoNcQCf2YHP/LHvQOCwZryNfErTyJNu6NLlebBHHGJguLOZ+CZgx/yMeGIPjpMVxDNED2BbALKZuuOZsz0MI1Q3TsaZa7eNE+2iZbOqDKh9AkkDYHiYb03WmHUufbIpZJqqxjYMX/QAFaf7PLQIdocx82FoPWo1IYWBnGGB/kUR3/ShQqga0zJER8NmFELj8yM2JopCEySwwnxwKULNnN8gwaNb2UE4i0L8zP5qYT4kogSzGoJ8QvDirelwnGGhC1ppwUiD+yHks9jjkmRFlYOtEbhcyABNiBSUjMTC4IUfsMxOrjcQWWcvc60u30BZJJOdzy7XRn17s43CL2MpY7VKH1rP2alKxrKSYESG9wzU3EFiO2shnGMvFvq3c7k0t0AXlAyuEkzZmJ7xRulfh8OFFlb3hzt5tUcdJ1A+oC+yjYdkfeyh8w8qg8XN4ToXtQMQZEDtl/VqBaALrKY0xGCPmWnL+lFeHCiIB7Qe73aEL1hQnsWwNme4mA7nhjU4kswgFaPu0Xlwc2f2iDrNToNF6Be3zmCxs6CgRsSBUVNpKyVEXxNMRtpPJ31efl3EiElMQgicIpXYQwefIeAfWIdrhf0iJ+4gEYYzfWfqsmqi0Zlxautrqi084NScFlBsd9T1sfvteJ1kdjtfTPGzytjJ+/C3fBkyigleICSjZnmbwAJumk/WBoxngYYSYGBS5y235RcQqb3Gd9C2vaf/+t314HtPILdgJO/2jzhuswdquY5Shm5Kpaax7UlbkOx2mBsBTqH3cWoyib84VJAQDTSaeeSdg5q+yRDor6end7neKj2uWZhGRo7hQzyJBk7FYwzSaNTCwwY0L74o09nTfYvVMghg6U5N8LEsgwTWbtUiMZc5gXqzfTAC6uyJoo2KtuQRrfZ7bt/BHJ9n0VkPlHTV0Fm5VpVroQFlOMsRIgkrhVRlbrv81oEYKaGqxZXng8vhKCVeAV2EXUGEnQhEU4XfGMgDp47LwjcM7aOdIHRP2gqO2lKdO+HhpoLTSXxYGLa5w3S2MOBdw9RDATxyhMHVSITgJ2+ps6l5xOiNX9OM3ig34ArgZKY70IpVaxISL7p1zAVuwc8C19jkezT5mLL5/jEimRYON1UEIMhklZiGlVhJCMG4o3nImhJ7/y04JioQQFUVvn2BERyzCFPY2KwhdRY9xyXL4x3V9pptPrQj+HqIloKp+ehD4nhyHH9aFI4FN/C9VaPJSM3UYoJlJcttWB0KgpOpWud9bEyt3ul0zSsj21P/kWOI+qvs75WW0RZBKAUEzZgkxBX1xcTFcnTzX4wmn0fSm1BQE9CniV2gaF1Cbs7w3TJNPlhFmg7EhM42SbbFMsIuEw5UL2GUV2Fr84cY2NVorXKEC9zw61qulZ361A1NmR1c45YUbVsKsT1V8lvI/ljMWzf04SrclZAEHwuiy6iq8TmIKwJ2cS7OnF5m4LrP5E96DycBIF9heHdnl7KKNmGZfOBMcT5cgMfVcEdgJxijYWYNR4d+TOb3WRNtjdwqPqNdmyN0ZOce6uSNMtprnbhLuk0Elwl6WmIEbXLZrJ/gbJOhzX+gFyY4xnVnhgd1VWd/KFO+wdw4hOesui4S/UzgYB4Dx0salHMSwN86vzO3T9YWo1IzIgPngO3NnnzZUXOVszg+gSAoI3Q6wPbczG4VvYfMEIcriDmFIODztmp64d0N2rHHuWNElzoCPF5CKbuWENjdJr9tuY+29v3vAVq5ne2Oev2eOJ8NDND+bscHwObeOqvVrlDULTnN8rKKE2pDnRI/hSYyU/Rv0gOlcFYQffYwyHdO5QqoVs4+T+G2CT7iGT8Rt+iCfqSTJkuQNQXN0OjSjJ6/eLYd7gY6mi3G0eR49dfCPa/JFsHxJY/VUARXqQ45zAP+f+po5FnGwYOjoO5YDuyDJplaaS03qcIOFuDSrYvQjmanGRVKbkMEcaFM6XkPAOLyUKk+w8lVzuwr1MvDF7yhAYeFmLqPVFGJvwKouKKPiPBkKcgpLx6xPCNBz6wOVs5HxD8ufZv5IT25huGnC/GRhj45X+GAvc7W2Ik7vcvOAZvDDYnLst9hNY1vvFXSzka/W9yTE8j5VgP0whSGOwjKMOSHJfo9A5Yi9neF/qVeE1K8phk3VmWkE5I5CMM4j6Dq7VD6Mo5DySRuW+KiBYJrekziZ5RgqnL2uft7e754kLO5Hbu8H1xuyuqOHHzHx51sMrUTNhstMf2UsD3F/EHKBp6PEz9/mJrbS0AsCYEHDv4cRYWAyG1XZIKy7RZEmxzqpjKoaqhSXX5NmrUitE8iD5vxqFHweVq3DZCxE3hio9IiaDzt5v01yndjGMIwvtkwX/oPOZdu8+SG33yzQTclWCwREIoZ5IqQfzZBtOASnaey0dDpY6WGw9TTL+LxPgDtEnWhpsm6Q+x01KlFhsdPt91DUhjhslvTNF1Mtam7WhuBfW0fwPL+Jhhd3FEZv7i2D80JPouYR+2wyt1jLRMN/0bZH4JLoyC6dVBDG3VvGr0u4U1zxnFoOQ0xfyX/sDC4PRzq6gwFgVu63GFN0QfGU2rqoeOWHSvwMcDL0vToYH/63SM9a3bqauM5Y8CXskJoPN/oGLNsq2RvxL09blIE9kGZlthe4mPN2rZhELTdZ9NzCrqPxiPCzI6f9Jvv/EuiXjFodc+/zhpVLCcFZZRqrwJca5N6tw8EpUTlUkjG9a4o3ebHC420XnfLVnv2sclM0ovTI3qi5yD7JkVk9lv72wca5i/tTyC65vicXYY6HfPXGb58xGi0ZwDi/g8fRLQKhzAAAA=="
  },
  "scripts/local_clip.py": {
    "sha256": "1c7fbc0c5788363fb2df16b10082809ef5741834f67d3faa5d781a3f1fab9108",
    "gzip_base64": "H4sIAAAAAAAC/51US2/cNhC+61ewPEnAWl63PRnQIUBSoIcmRmrkEhjEWJxds5BEdUh6sw3y3zukRK3Uxj5kD1rO4xt+8+BIKe8IRyAUIDrbQieOwXig69HAYAVha0mb4bgTdvTGDtB1ZxEcawTBSfxph+MfMIrWDtpEOxtqKWVh+tGSF0BHDu4wy385OxQHsr0YwT915lHMhjsWs5M7u3z02I8H0y34EIwuio8fPtyLJmFKpaJdqaomdLZ7xrKqYz6Dd59vHgqOVcerajM4JF/ud8J5KlOEayG9tZ2T8eT4KoXP0MmqmhhC0MZmfi1XYzBcH/MPzgnMdZsdZrEoCo2HqZSq7cxYOhuoRa5f8GPw8Xog3zAPHQhiyZobFtYFbN7bAavbQvDPHDamSTkbUiTxUyP2F3X8ERiH4hN0Ad8RWSplbtPEpT7BszBOQEcI+iw8mb5Hfct9xUxPnIx/Evgl5oZ6w0FWy22EPtCQc19SXXvnxCdQipqbWt9jrB3Q+a3hQfOWzmUlwAmdxUteiyr3fVFc2Kx7tHC5AGOTl/xlbkT6rpqRDztxBMO9uYTv0YMGD8zgq3TtE/bwCcmxr7wVNzshCf8O6DxLXzf9kEazLo5dHN86fn4tq2oX+QzH39++ZIzMki0y3EbMLNl8ISx5xj2Fnmef9fIRnJMxDmK8f//tPzHSfP/GfYjOm8pIB/3Y4Ufw0fbLz/s9T6h8RPCO5c8PLGh7GlaKbeT2iZfG4noCioOQZPneLjvDm57V18lZAG+Ws+OhdGEcO4O6Fm8ivys78MLZrpeHb//riYpvnBuz6fYGFTeP/D6uPpHxqDx+8WV0q3XoR1dmn+rFeX95tjbxL0+gKPjZKjVAzxtLNI2QSvU8ZkrJadTTsiROJC/O+g0dU0fvkqXU6FoyaRU3Smnb8uZbIWvQWsEMKeXV1cSKGfnziM1dYhPnlJnr5p4Cvopm3j8KnaZ3Bh86C/GR4QFCx9vlVeQy298H37yO3myqFfcJxJ6Oqztj019Eu3Iyr9Z21NZ5iSTBxu09qaedkc6X55fE9fVV8S+N1xg/XwcAAA=="
  },
  "scripts/mac_workflow.py": {
    "sha256": "c9ba747bed5932c5c75b3db30ff6cea14354a6e6c287651777d21ce187bb20f0",
    "gzip_base64": "H4sIAAAAAAAC/3VWS2/bRhC+61dM92KypagqSS8OdEhdpwjQ2IbtFAUEgViRQ2krcpfYh2LD8H/v7C5JyYqqC8mdmW9mvnmsGGNXqu0atAhfeQnvPsCfv8Pd7RUoXW7RWM2tUBK22HSozUdoVYUN4BOWLgiM5c8GhIS7Z7ul7/f5fJ4zxiai7ZS28K9RclJr1ULH7bYRa+gFd/Q5KBm37rQq0Zjx5NlMopl20ooWC5R7oZVsUdoB4oxoMrm/vX2ERYBPiqIWDRZFmms0qtljkuYd16RnlvPV5NO3P77cFtf/PF7fPHy5vXkgsxeWf+d7lgHL2+59fH7g4Vk3vAwvfHiKenjGF7XZsNfJZFJhDeVWKYOFUU6XmFRCY2mVfs6Iuq4RpbCLGyUxvZwA/UbxEPh4kAa5qEczEAaksuCto7H/RTeD9aCb5vTGZeUM6uSIg9GMcD1WtM6FCXQl6QHX/zQXBuEzSW6U/aycrK61Vjqp2T3ueMslWFHxHSVhsXU7Li/hJSK+soMrjdZp2bsKpyVFRoYWDcVtqJ5YJb5HoFY6NIvvqpGInNA1fR1Ff/qjbLzZIQ8gD/HIuLoWT3mjvnsmPPBp7SNsp7FGrbGikA5FmQETsnM2tMZQkIMqoR1yuTzNeNQ74Z8gGpTJwTKFxQLmP9gfFJa/rn7E8PU76/1/qvaGvZr9hZbvqGZguHXkMhZ0IyzXs05wqYgGeBmpeM2g4Y2De0cMNg19bEQO7A0o+yJpKTTciBl5dVvCW2PjWkJqXcNJ/zH0S4st33EBVBhKFMEIadFSe9NUux1sUKImkPyAHpOOmf1NYWBMiX2qOLlYk3rHxxwobqKEz3K4E43YwsPtt/ura2oHUibScK3UDkjBwcZJ7imQZAdjoXNq3mGUsdwVYfEV/cpJurDt+kkpSUQNc+CWHS2/jFZZZv02/XhEFOs0pZt4hbxybWeSl4uIeXEJjTA2IbN8TyuXlmwhZK2Wl+9WaQYXAYqUwjMvikGnyI7hLwiSlLac6mp1EpXXvNyhrEwWpHE83kpo6Rk/P3zPRcPXfohe0zTi9uyjcY31Ezvu7JxISZZ0VQysUCRTvyI9Lyt68M4XuFDOEreLR+0wA4tPwyut78WZVZ74TU5Ygf6gOkQQuF0EdmmkeWWSGFVubEVO6KFFl6TjjESLJYvhsRX8tIDl+wzm8xVdc2GEBhXKn61Op+g+BjeuvfjZX4Wxt43gHe29CPNKPYeaxuToToQK5Yaa7OvdA/CdFXXOhnTCmEfLvuOIjaJULbVxlWilbEZbRPmd5l/Q32BVBpHOrL+LcfGZN2a4ULzRcBv49367xXCoU4Oc1tqeiq707NPV9fTBYjed57/NqOvkfrYWctbzlcb7J3gBJCcR1zdoPPWNkvaTEIImF2/7wX8MPq1SjZkZuiwK3PNmRsnm3TNLs9PFzqbTIVsWIYbP0GHTvmn91ctLZOfsBa0i7Xw3sZHDJTs6XQUkg1gZ0vi55V1CsiPdKFqdjY68TkMTHIPHg4hb4V74yA7S/iSKqYJ9YrGW6epwufT6qq59hxdWFWXnjluzpzrnXUckJB4u6rKj/wyhZj/YwC9UHzLo5SwE4zoKBHnL3lbrTIecKdXIiDFozZSciJr+PJ5glUrWYmNmvdaaG8z9ELM+8eHGi1FO/gOjuo8fnAoAAA=="
  },
  "scripts/model_assets.py": {
    "sha256": "10aff4ff4942bf29f39454c19a2c04b001f2b6bc2beb049b31157affe52229c1",
    "gzip_base64": "H4sIAAAAAAAC/71YzXLcNhK+z1NgkYNJe0xJ3iQHeecg23KUWstSxXLWVV4VC0OCQ2RIgguAkiZaVeVp9sH2SbYbAP9mqHHsw+rgMclGo9H99dfdoJS+kbdVIVlK+B1LDCllygui+I3QQlaasColN1yJbEPqZlkInXNFmDIiQ+mcwbMmS55JxUmjeUQpnYmylsqA1KpmSvP2GYULsWwff9Oyav8v9SxTsiQ1MyhC/OtLeGxFdN4YUcycnGoqI0oe8+pGKFmVvDLtmolPbg3Tmpu4VnKluNat+FtR8Ev/bk60+J3Hht+Z2eyXi4srsrAmBHGcgVgchxGIyeKGB2EERwPV+vPR9Ww2S3lGUrHi2gR4hDlhxUoqYfJyTtodF+9lxcPjGYE/kfUSpJJgfEXuqc7Zix9+pHNCV8I8XxZy+RxeHdEHtwj/FBOak19Z0fBTpaQK6MdKNzUehad9XJwt/R40tBpucB0cykcichsG4diexYK0phBewHYD8aMg3D0ALhhb3Btst4yaOmWGBxlFEXKPPoq0YQb8qE2MXn/45yGNeJUA/ILQ7QGOS8BvcK4FObRvbmE7i5FI1rwKqFrSEOJKtFGclf2mgEaS5E21Rr8Kw1VQsHKZsmMvCWFkaXB0+OJ78pTgTzgnS0rDXsOO6VZdOPre2/dsQQqwZ0IG/NQjTttQIwzGG3llVirotDpFiptGVd6WnN95lIUedC41PegUT6RKH0Ucbm6dJ7TFM8Qd/DQRC/KXhdf1meIzvd4DwIyeC61FtUJlt5Bxq+dWR4vFYxfuB9oBZ5Qo7T49VK/7A4RDS9yyL9hy0mZAkvNkrZuSlEKXzCR5a0dE3kjrCmArYnIISmtpRFuveqrA1FdSmjlJYWdRMQOU6P2JHwGX9js5GAn0JDHlecULELsBmpF2cS8d7kvz7mCDnQjXCauBf00O5Fs7bIN7wWNGqg0dIQj398fzEAtKVokM1M27xXBSXw4Wbxmk/pwYxSqN9LKguTF1myPtUnAB8niES3qFNr8sj7a5bKRhRbzcGLB2QSAuwRhgNmXdK8zZVtFnikDV9NppSWRZFxyYrtN06JlCVCbInpzb4nXfr7bVjF4/QPAxQXfVPhD839zZR+47/g8GBocPT+YkKxqdL65Uw3sce1f1YYOXnb96Wrd+A1K/42ZI5dNx7teXDbh3CewLyzG3cHlPLsPKt5gqegEWsF7+O/KhKxNnbwdmaiBVzA52I0VKPnHzBGjKItQJZdxxhI2QzZfLi9fRQO+5txNSpm0EXG1FTsib1Qp+Abk8zpslOsTqyAD1eedBAMTSg7LXLHXkj/OZnr2Nzz6+it/8/OHk1bvT+NPpFUAGqs4RHfscK5F1tytc9JBOqIvAUkgD1hQm6DRf/OP9u4uTN/HVz+enFx+vMF4/Hg78vX/56dXJT8OlL4ZLPTrpVWsmoLEz+eElKblhUGMYwRDKxizuB0d/MrHDk+sH6FXoVg3J6NnV1SXB1NurafuoqI3uAtyDBoLqMJEwoNTdACP1YGCTQgDoXkIirXl1YJQFhahSkMH6zDYQemgdoRinj4YYwhq/Pnl9dmqDO8Dy9te+0GNjt4Mx325m+BS3GBsc6u+c14BpOADk/CeELXQl+qCQUDHATIM0i32vSwnFG+1PKiAVwL8FSDJMJMsi/XHarWLnK988bp8DY+DPEULl8HjdB+MvqpkmoE7L8VbbgnCkr62NFjLQINVCs/ylPZAfAYAvV2tWw0cD/6aiZmsm5h4G6DQjUraGDzmrGx09iqB3nAGVQKUrLYN0KW8VHRhe1s6N2Mfd8qLAX1EBYooCPIyMD00MNNrRIOgcnesGgigVeh03mq140FaxMEKRTr70/cm31J2dPq3zM9L7oE/oK2jXr/SFGkLUN12jaNmz/K0z8Sl5QZ6RH3xX+vTpX6fKxS+O7H3BeA928Eo2K6BTcAXJIf/R28dgiOYKafxWJLzbwtpMXP2sIWKw20/ilacr9AYkLb+bD5zCq6bkCpvgHffMydGga2nL8Od7q+Ph4LGie03uJ730QP49rMHjUE2WYdtbwKDJsQp+RThGGWPX9wEa+/w7cnontOW6tlHElpG7DpNDu11x6MJhdITDmmJDLPcBdMGVEj7cAucZXkXj0cK17W7nvnEfDqMB/RWFxJpp4fJyw8AG3aQsJ1Av6HyrRw/Hk8d2r/Rsu6efIIWMEnKC8DAInHb342FMttSGEORHmqZHGMHZBgiuGj6Mwm435bzf3U+AgRJoGcYoWVj2FasKapFnkjnycwWa6w2RFYSBZXAIO7o6b4vExj6a4kJCzoHtmipt8gMg2BUMi6IgKVPC893Z25f9HNe1RYaVNcj9Bl5Cj8HMJth///jPnoP7qcES+lZ96tCueC1jkWJyta8w/BUruXvnrmYWvbx7AR8fYauJP3usGEaFBeRJMC5c4RaSJmfMR6FqVXwJnEO/b1gBJGMRPoIdWXPgoyIFR9uKtN+zPondrUxUruFogb+isdJzwjGNY7meWOxLCYLHUoA7qlPZS35t0mo8GADWDSF7PPLnU/X/k6Ytj7uzQIY9Ojz1jIiVY2Lw2qJsN3/CPA6awTu/9yUldj4f8LYfaXBG9pRMKbWtWwtXvPFCytdoMVI0J7dIAt0orDdVkkPTBBtBUwFyrrW53JhcIlek7r7yywOttW53qh3auB9zdprua+yejqPnPy0blfBvrGtTRXFo7p9U8A0JBVzuLI90k2Xizg3BkWYZ4LHSUuG0QaPa2B+YEbaHYqM2u/djMCpANq0Dp3o3Oe1MfJfw2pCLD7Y12tWBlu00jSOcRb4faw8wuhY7nqTXqZ7sNauwPcyZStFo6GvFKjfuMh16W3RLguOS69h0DZOLhQVrEWXrGA13Ntwmqse8gWPD8exrV/qboqFHcECxjU4tgRM0hQwGL8YxVqM4tiNGHJdMVHFM24sxKIQKgeev/6MTtWpwbLm0XwLAW6JEjXhbxFD+kjgOBysjlqYx80sC+vx5myEAF7Op+eLS16F/NQKq/5C2pte3KfOt61u2gfXAdGg11QYajxgaPU73Lu1mMVib5BLacL0IRvdBId4t2ruE9n7N6gMleLvl1dofVKyD0c14gK+i/hbPPvYMYR9b6/1jZ1E4+x/6kcsY/BkAAA=="
  },
  "scripts/run_song.py": {
    "sha256": "e69f02b9d047310ab368c807d23ce4bded8756466c24fe131345968e64bd8e14",
    "gzip_base64": "H4sIAAAAAAAC/61YWW/cRhJ+n1/Ry4cViYwoKV4HCxl8kA1lYSS2jNgJFhAGRItszrSH17KbOqDov+9XfQyPGSlZZ/Ugid3VVdV1ftVBEPxL1KLjWjDOir4s2Q1X6iTv+orpjmdbJmt20/R1LnJ2dnry6vRYiaypc4ZfWtxrtWR6I2qWdY1SBc9FHATBQlZt02nGu3XLOyX894arTSlv/OdX1dSLomsq1nJNG8xtfMKnJ1IPyv/b9zK39PqhFcpTf8bfUnzklVAtz8Ri8cvV1ReWGC5hmhayFGkaxZ1QTXkrwiiGTqLW6vpstQD3mITHslai0+HpkindhYbDCQt005TqRGlRpeKWl0EUWfm8z2Xj5ZcSBLWs12kl7919uuZW1LzOhCd6d/Xhw/svn8F+w79//cOS3XVSi3QwQdfXnjbj2Ubk6dfmZsnEvch6LSwNiNfpRHbbCbpNSjukutTZxn1AX5nDsWm26ettqoTW0FEtFotcFJaVzGEHqR9C1fRdJpakN5lryRAEgnzrz5LLVXIG8+BiXcnbYS06XzD8PCMunHDYOx6Zs7NF+E7WOpyTMlmwomz4/k4sVYoTYi26MGKiVGLO0ojphO67mj0GCvateAoSJZs6OGdnSxZYG6TWP1iz/zjTRCBwxsGWN5PhOv4JrN32mNhlYtK3iC/BqzRrqkpqkLjIuA4Qu8HqAM+JBXFgZtFgdlVQzFaexj5vS17v/N30uu31X/K7jyE47Vti6pmIUHJdc3hLgK2rGrGzJaVMnPdVq0IvC5GPZEi34kElX7peRLGosyZHrkfxRtznci2UDl2smSvH1TaXXegqgTlEuYZMTput5WGooVIh11DCHqOa4IXGpEhgbVA4wtiwUAhCjhJpNEXA5sgCuw3P5ymVTWjG/pbsjHe+83rHJaL3N1724rLrmi4MfkFhgK4i0033wG5ECSsrphssFoUg/W05Oqlw4/LEJ94b2LlpFNX1Wtw5/QN7q6H2OMWWO02c+SEDlx5Xl1nMkCXcdh78OZ+inlFqXz8GtCi1Tb3gAscDlF+NngMdhvS6Hq+uKEMFZI226VsFq+vTeda4XKb8QzEPD2iMD6PydWBSejVqD9GcWVYKXqf/V5bzWuMOum+6a96jKVsLea4ddbhgBTnkD1Bz6nspNe9g3wKaIyPMmRH/0SIJMTU7dYx3VJPV1dOEcdF0loxwgVXDfIJwNauxGvlLXAOqOLWJksB2M1wr3SU4WdT/DwoKEizRn+mVxhaxgncLdJW/XCRn0upGk2rBeyCfVtSUHZanYndSbwa8g0SreiUzXhpIJOueiqFUDBzYuucdR2MSeRw8LY3evhYTpiDkoUJfhMV9KzpJ0e5KK9GYjLGmJdtLaHK/JPOQA0SN5CD/h8PZa2vCVTTUlKF6jOpYcfRomJ2fvsqf0ov0EcemGfeUUoLZdZN6q6ejoU4Jgz+SEVgJd3KMguMLXR/0/CrasUMFdRxhuY9NLc4n7nBRRRu7dWOdmLfknUE0bmYZUbTDxC4Xx7FpDjo3VFzWoTOVgaodruRha3zRrY0tPpmdEM7OOtnSLZI0zZsMsHJ0MuZ5nnJ3JAyOj13JWBq4mhAcXUKH//TQNR/1mGdOw1MvHF2yjSjbJPgVFR6FQIycjNaAygM+hMztptVk1xuCFwV7nDMRDmPxvtSJR8a2b6iTimfHNDccU2zblvgic4dnwZxnxpBwU4MWg6gTL5802Xfs89cpB9RHzaeRmVBJSDjl1Wk0aHt2+rKJbQmYMzUgc8TE8sBBSkbHyvwhZsrhCmc1UIzaPu3HbmfS/M2RvlZ9S7GK2SoxEEbkIXwUuhMR+zsKacZNxAUGgYpMKvexxjREE0aqUHuEbaIGD9CSFq0KniKPTkaShsRyNxEWZvzonUiTCEZCJHZNJQwJVj5QJCHQIB/qsUKKMlcojuw7RnLjrw2yaCQj2gOx5gfUMaOAFSVCtZNZSpFj0EXcPpj6ZiLWbMMml46MvQV0ZIR7jt3U6TSJg90Nd6DAoCB0SaArQtRkiePbs9fHmGtFwCCiFHU4hxAGjJ39sWk+Xb1jvcLoCY2EwXjiHnEMA6EyMVMkXbAYrAuvzvPFbhhtRgBSdyMIOJTNpQdiA3I3EeWhmPloDgJ4szVrgpb8EDQT95lo9Qh2gg8zt3/GJgSCzH+Rj36a2IrgZ77u2aPpz0e+Px+tzuPviycEzCPZftQVjqhVHa2iJ9tal8w4D6Ws6jkmf7EFEmE3suSM59zhDk1Dm+KIu6Ls1WZcR40OwTvfmRlkrDe4ktrymurgTV+vef0GA0VXNbU8WXcNrAE3AhbZ8sKgWtlDgRpC5QEZiDSTFGRJV8oGC41gtfeMQYjwm/X1pMcvps3NsrdTfZHVujQLZZNt4f8Jv3htn2ykKTPZNojiBk0wDPh3Lvom4WSYxQVRhvRr6VZ+vnr3U3r5b/b7+Pvj20lAvKUD6Bjvry6nwUDrcVZivgij2eyCeUXjhm56uYC5Nqgmg84Uq8hLxZq7GnPMBg3fl5037I5LbQqBwTrwdddDDb3heneskJ3Shy7q4dIIWg0JcsjwsnBnDmIO3yzBcfbCFPohftcNb4FBmu7k4t3l8WcqOGfxazj7hmdbbCRmsqc2JbJti1KpE5K2P+mDLDUZkMyrmc/rtOK1LFDUEp/nubhF7xsO2O9D7whNQX0tH0jdQqqbNGt7EgJLJeOaYiebZBLMu5lvT4AFHbtZeg/vJX+IB/d5VvyeoOXu1WG/rE17jcvJ0DvvsOO/KVqI0IJDb4zw4KBKbc40tviO3wbD+UreT48HWBkIhzrbwYVNr2bEg0qjvjGK4uHpAakDzrvvOZJ2uH2EUrzE6fOEaapuY8fbvK09TmdCEIaWLSoTCrCb/CKWoAW7vKf5mNql0VQqA8vdKwnpulvZzwk65F7icNYwnQqD+9zcHI2pwXaPmAzuaWd2GTexD6LiWy7p4UmWrKSWhvrab9gDR50qG5Q9m8yIdzQ1UuHpQK/YH2Cmnth76flR1pgi3YRmTc5ueu2HSTSxvs420EHQ8zuVJEJJlHalfmMhUYd+JrWxQz97+xkmhFFQQnWYh3cP6YHwtsYllEcv7zH9+kdoXtSuz/9p4jzWVTsL84HjXsBbB/yP/EZv2qF/5UbmLmeqDwcmz/Hhodp1YvGTETTns5xe4DlLIVVqtINwKnpy1JNMuIyyuG/puTy02ZKMEsXeMCGIZdjbbxe4yTgZlmyI6GQU99FiHtKfgakVlz5aGSYHNik/++HrIQCQ6qX5l3r3Pix8/ka7/k3JjyKByQVlkkYWCx2X7CfxcNPwLn/vKV2BCQqOakCWMITJHG2+ALbmZfIZyEW5Zx9VKOfKPwuXfv0YPQ+BvkmjxQKWSVOKlDQ1FTNN6VEiTQOrk32hWPwX9w0l7KobAAA="
  },
  "scripts/runtime_environment.py": {
    "sha256": "ba615414d3e8acaca8e00d6dc268708e1ee6c110fcc1106acb46ab760dec2955",
    "gzip_base64": "H4sIAAAAAAAC/31UwW7bMAy9+ysI7eIAmYtdC+SQpumStU2CLR02FIUh20wsxJYMiU5bDPv3UbLbukUwHQxRenyPFEkLIa4RG3i0imRWIdhWk6oRHElCUBqoRMhLzA+mJXgsFWMaiw7tUek9yJbv2SOXpIwGqQsg2zpKhBCRqhtjCYyLdtbU0EgqK5VBf7xhM4q+r9dbmAQjTtMd06fpKGEBUx0xHiWNtMzv7r88RFFU4O4lwBT1UVmja76NrTE08UxjjrhEzgWLycpoHJ1HwMvfv4j4/UAg3OeSM2RAwJ2BSHqRz+FCBIyP3jHmT7D8Enc/09l0tpinl8vv4rxnYff2eNY5jt9hN7+3i/UqXa5+bKc3NyecmmcqjT7tdbFcnfDI1Dv4Zrk5HVGjmiFucZUu7i466BBWtvs9V3Unczwr2+yDy6/59v8uT0hDl9vNzWy9ulp+/RBMLampDHEr9Oi/4TsoKD9zoXKKjUv6U1C7t9KCcuCrC1g5fDvuarkzFrSsccwUFnMy9tm3cShfwrDaxX1X+PWKSeoD7+O+3SZb2zIDPilHqTkEc/TqNIj03ks9cLyObPxK1kE/wa0psALXZo01OTqHzrcv8CQ5VWAYLW0IM2MOYXQK420o5RHh7ZVSpSulMelJ7zhn6edTF9IWUKIsKuaGTOYHZJJHxV3Es5qXUvvSBJkuL/jWco+hhQNajVXy8d3vfc0uprPr+epS+KzEdL8Xve5lF1ygRVisb+dj8I0UNmRYG7gdC1Z0Y/5FmKdncEjU2VyU2bT/NXQjidRaPRSP/gG0RpBAjAQAAA=="
  },
  "scripts/seam_diagnostics.py": {
    "sha256": "dd5a2b55cca766a433cf3ef47bc18c1d9f88d0e66bc2c041d702d1924f6e43c3",
    "gzip_base64": "H4sIAAAAAAAC/5VY3Y/bNhJ/91/B471IjlbedXpNYZwLBLkUfWqLNrgXwyBoibLZ1VdJatd7C//vNzOkZNnyJncGkpVEzvfMb4bknH8+to1xLJPZQeXMKlmx1qgnrZ4tkzV80ftalqxS0nZGVap2lj1rd2g6x/aqVkY6Xe9ZY5hRbSkzfJFdrpuUcz7TFXGXZt9KY9X53ciX/uVP29T9cyXdYVaYBpSAp1LvWFj4DRfCsz10TpfD24vtH7tO57PZ77/++oWtiSISotClEiJOjbJN+aSiOAVN0IrNw3YGtCkKSnVtlXHRfcKsMxFxWDDumqa0C+tUJdSTLHkce93Ivl6z3z9++ZyA8TIXbVYl7Nlop/Ax2GGaJ1XLOlNsUF8u//F9v5Gsn81yVQRXC+uks5GVVVsqG69mDH5Z09UOjCpVPazQQqvkI3yv5DGK5M5GoGanYlZAPOiR6Zr1BAkDKbIr3fo+BvPeLz98/wMxMZUlHuAI+xe4wXZVVJSNdIHdfL68zRG4kGYDO6aLoKsqrWL3nr1ynanZK0dlRb4rLF+x5f2cBJbN/uE+wpUYickeov2lqVVC9P2Pg563yWGBqNGQt4iLrgTfZhLSoTAyc7qpgQ+aOvLbj2uy48PXzb228koSBEPI/E+ZQZqJ7CDrvQJJQ4gCt43e3g2Pd8utD5pGeQZJomXifXszbqeQM5C4rcqcMF0dwb/E67V+H/IGPvWlAI+jKqBVSL09LGMKgiNlbiPchZkPEIBlki9wS4obOBJDjjt1dFHs6cENuL7h3g4BWKD4lv1tTUWxGrxipAY3/Rvd+dmYxkSIOqA1AE4m66bWEBUwiz3+/B+vkxcvKUZeFMBN22EFeAUjjkglci33dWOdzqzg7B0BQIr/fQd1flDHzeqH7Zg+rR5zbYL1RmWNyTHzN1v6gP7PDl39SDEnw+jV8u1mRX7dnm3Sda6OQEs7Npxe+XZYhuQEQEXmFmpe5ejYdF82u6jgr7R5df8+P4mPYr6AoEBwg5fjs4SCyj1witGtD6uLTMOM90ZcfsffDqL1ePF1EoWC/6IgBBJyBsOH4fCqjDzhdT3xs1phy0XaBB0399tpkgyKkpF75SB2gHCd5WQSD01E5Zz6zU2DLo1BQAb5Y6ke0yExLsUAcCP+84TRS/osn8DDV1XwfygIbqkbRwqk2gqjSsjRJyVcQ+WF6x7dI9xC5J7txivjF/l29a3AfBo7n2kL6dYHCUqcICVHaZDWVueKuQPsARVGcQrYgiUTepNX6jq/BnADZZds3ic0YGSl7DdV5Z/C4IAxIZqAjLkuCmUsoyZYyVoXyrqReormDqBc97W9YIWvNzEukJ2CZFTQFDUEmwJ4tpDmgDRr2heMMpmXDIxHGSufKQ1Gpd6XOyz5Yqcapbj6VPK1yuchZaZ1J5/TGq3V2NiG5IJEq/SRHk/TmsyaGoalTs0uP7Yv33ICWvCKwRpMiU+XvrjlD9icEPv4KoaBRyrbVtV5hFvImHGRUxX2O14Dvq087GE5ATzipGKcsLC3zrEph9TxnykZoBss/IQ0ccbQLvPOI/2I0UVeAtov58gDGiEPsyns6eNMmn+FPZpbNbkqRT/XAvHgA0IGmLdwFLg1fp28T/7OftLGOuaeG7aD/M6l0cquCDkhXyCBOpMpnB4Bw2QJ0wfbSesn6ELTAK2PTMLkkENyUb+jRuc0FFnquw9tu9mNwVRlNM7et1oxpuPbO9VRW7An9qPK62nWIyjZ3w8HJNyDFN8SR/pyxlEeGAQ5aJzAieqchUDTYyOxRoCEcacpOwdIu7pA7152UHv4dA3NE57nKf4G/H5t4PiJvEsohQM/UzAJtoAI7gDBU6UfRs4A6mBCfES8gNqm0GLOTEYjXOjLHj2CSdSr6pHgTAYvZ4edhnmjlDtVJoz8AVDi5aZwMKjs2GnBEb1fqbyvQGkCLojvyJ2t10G9czqOGmToSbE/7fnuRbHAhjDKi/+1eQVXU/r3nUoWTpnhuNh8o02R9K+YcakAwTjBFY3Ob81D/bzmN0wHtsGN1HnX1/MfbZ/uJkhpOisAJTGfb1HdPWyvMfEde3MjHDp0LvruS6XoVR8dqMa/GkDAA/EbSr+biu95vntg//RYe0EXSt1/dI0DUOz1mYgvVeHC+fMW+icXHoon5EbvD0Sv65v04Knw+dIvycjuKddwmTFKqaG79uHv+8h2Sm3VHgEU3RmOZ8s5mrlazkndqRMa+B7Q24sGighJ7m7ZFAOjiDjdXp7y75Xu6h1EK1c5QN8ey2gbEOpGL0bxC3834TcA4WiZ5C/e7sw8DNfCGyT8TKKOcHAuX4B+1zRlNHhqPbjAH8Mpqj/65BjO46dRCwBdYVKjmvkw91piCiyX4WU8ACwWy3OQQtwuB6ZXAoiT8I7wNooP0BbEK8ggK1f7q4FpuKaJAstkFG7UxSuJWQvxgj/b/tRIlzjod2wXK7owwu4Dfg4FtOpR6Nq1fDSBbFo/PyJ8taMZNBxUL8fP7YQTdAPsSfwPfz2HQ4umczDLG2oV4WisWNVZmjf2paaIpuxTGFo81C2ZH7GZH7Hx/g7v96gz+pmELgV3L6wECaqGDSnEE1SG+V7hvZa/+5PasGdJp5A0NLjzDde5ALl3oB9PkuDOeHxP5Lf2S7PZDDJKCHSWENQEhKikroUIjYBuFg1EpL9lTD+afYeJ+RutRDk0e6NbbDxrIfImEyIeUaYyz4UMJBG/u8O4Jsy9tGqNwxFq8lenoeWvv5gudCbYjz0rcKA/yMNG4+uG5Jwt43sa3JdiyngljAaxNPXlXdXayNMkBNG1Wy/j8T7+r+HKY7inXYG2XmA8+y8qG/g/0hUAAA=="
  },
  "scripts/sequence_diagnostics.py": {
    "sha256": "69fbc6e8e86f395722526fce18ce95abe91a553be117e7037a9db5c9a05736a4",
    "gzip_base64": "H4sIAAAAAAAC/51YbW/bOBL+7l/B036ovGsryRbdD+npQ6/oAQds26DtHbAwDIKRKFuNLKkk5cQI/N/3Gb5IsuK0uxegTURyhvP6zAyjKPok20ocmNlKpsVOskxkW5mzsm47o/GLKbmXSkvWqFwqZhqWl9qU9aYr9ZZlTW1kbVihmh1Iq2rpjsmikJnRSRRFs3LXNsowoTatAKPwvRV6W5W34fOrbuqZZdMKQxvMb9zgMxzS286UVf910I7CHFqpw/nP+F3JD9BFtyKTs9mnjx+/sNTyiTkvykpyPk+U1E21l/E8gVRQQa+u1jNwTOj6pKy1VCa+XDBtVGw5XLDINE2lL7SROy73oormc3e/6PKyCfd/evPl3QJWEzlvs51XSTV7WYs6k+HU24/v3//ny2fw34pfX/22YPeqNJIPVlBdHc46l/Cvze2CyQeZdUa6M1qKHc9LsakbuCTrTaDLTS0qro0wejab5bKABJL05Fp+6yTkiMF/wZrOwM3z6xnDD3z1P1GVuTCSCfbycqkl3JsziLRxMjBBn7Vo9baBWNuuvmNXFDq1/7hk96XZgisTlZEKUYKwKTclpHGxQPeMbg5uwdLIIwu36IUb1i11mcNZpTmAlIyVVI3IdUwcyENhN6G9iGjhBiMfTDx35FaZs6TeQvkFHXmWHqf2ZdPp8zzkQwutd5DheQHIN6ZTEhx8CiQuBmLLMO92rY6DHogPOJTfyYNOv6hOzhM4r8lhjHmylQ95uZHaxEG0hoIbfAP1KvJr0drZrmDxsGddxp2TdbRm/0jhdPjL2uiZbcuGfnBs4NS1yBKKxazZ7UrjDvsIX0VIQn//E0KhtTSaO/0dmbdFSDncXpQbfeFOLm+Flt60Y47BK8lGmjhyOVI2Ne+NHc0t7/B5SmtNtIp2sGvlhCCZkebtcn/1yt4Z0clK1nF/WktJZrF8r3wG2egWJcASidTJd0o1Ko7eISgyA0wNqcD+BY6jBKPgobxhO2GyLSWNFeUCGhPQaq8rvOc8A5wGwnGFRHXiEuQEAUfOs+L9k/06yO6stIoAJUH2MzTfU+YD1GbCgExow8x9EwrGoI7n44RWWFO5XjC6EqG5Wi/wz24VFAp1Lh+oysRXC3Y5utkyAUxUFvdAeCrjyhKC11Qnv9Hz0U2nMsqJaZpH+LDcVpHLkCnQhISpAXWOS1JqrmQlTLmX3DRPOM5t8rj4dRTWwv6WEOQ9cx9/k22i8Fpj0XIZ8mM4XigqcCEtf6YImDLuueQdQgXpMM7x4Qo41G1MQ3t1uX6WJeqjUR0B3YR2vDFy5/lgejvqNfpoh4ai3sj8NSKrJuPvxB0VJGo1VFNVIFC2ZxlBAGRCHVRws/NJ8eLRxsH15cv8yN/wx7PyHTlp+nhG7+OLIRCdjOmoCsf+uj48FwOynMRO6KU0+9DU8gfWKHxuQVFK7xPAcNXV6XQc6U2tCFUir/6FvxGqUI8yQn4vECV76ExCiNoY/c2FUECK4Qz4nJz4kUs/BpEdd9sssV0HrLhtgHC3Eh2MyEx1oDgMxWVQyMNFItpW1nn8GAUTcFeOrAWia4cbC+YTBAvUpnmFaBm3hkXSYHEi9fgHRwUKbJDkuk9Hu2yzDBF+4bu6CBDRVYaOWUMfT0KwFzsvMxMPsWHFSk8kzICfKE9nds4pnNr/fQPxE3u7ldkdOqwqdOm3ElgqKVGoeIj6YGNIqBKgiXYdRjWNOrxmNbXyDK2ost0mTlIEY2k4ntg7XOuV7O5AHPv+2PYf1ICi/efNXfpv9HQ+3AnJ20aXBDIL70MCdVkj06hMxd6vI0SwwC3vEb2+FUTWRj7HHt3x1YtzxnixtlkdTTl5aUfJYYeFJGvaA2WClyGAahSKB2S4CIGU3It99Bc4UHhN6LE0oR46+nh00EWQ62GCrVYhrtajTBBkm1XrjEvWpE5Q5rFrmE8kCTNMsqmATtHPTpA5ZXyb1DTVEY6CxeMgJ4J5Vz7YP48DRtBlcNqtBLjRIDTxorifYPrUQkQztksRgYY/OpbHiYF6JfxMwkMMIePSPqCeHvdFkaYbe3I87gzQ9Yxzn17vg85C5t9h2nt8YPkT+yQzm523B4ORVBSYgvqJybjszFkH0AyL2trcVbSFawPrBllmlGAbAfPbXdvygTwZo7lvNZ5RlECbQGkVDLk+20+cYdIrRiymEeprS2Dxl0p8KOnsfksDytgep0V8JGzStTSJxiOMfEbRM2j6Xd/D1jQkpzYbULQJ9SOYv67RCSEtbkV2BxSnVRpdsGJby2sroK0BFsuwEJrbSXUZiha6EV+GaMJdsH4euw4dE6WheKBCMapBLy+fsEQCE1X0mbLZwX7fMS3CG03uH2lWV4vLNaUuug6A/j1dhoFGJ+hDmOhMg0GjzNyzxfJbh6Ef87R/Ryhx6v3NZxSTA9sLdUii42yCZj1gR6O60QMaGTcYGh1R7Zf8OwTsMLxBOEY9Uf8M8QlNHI0WdmawKREUsYMR5uY6r6R6TUCjpdpL5qJT2+wqRFnRfL2TRiCERP/wMAzI7sLpeNx3oOn0ESkOE24a5tI9IqRRF2/evlt+pkHxKnkF7X3opD5wLBS0TVmblBrAwanY5laVdDp9LpifiXeiLgsMoekPJ+EFyxHtiPyel/sGs2HOLQp6qRiO+AWMMTxrO7oXzkiDQ8JI0y9MYjR9eelcbNRhQAD/NhWP43KIBvmQydbY2fed/RNZDmWZJKAYjZyWpM9+m6BpRD612WlPp0X0SG9+sf2aJ5xTkeP8eM0e7dK4R7aY5NoUyslqJPDfjGkXwTuAcjzE6o0zFfXtBHXLpq5O2q9bZJYsBKLzNTKykIoiHyy/dejLNFsuvdWGKKVHUhplwoNp8kZt7MRyY3fiXOpMldZ+Ked5k3E+H1EmIs+58CRxtFwSCC3sG2l6Y8uzvzx3L0rfI4VV/l9SrxbI0e6TqABbNKkc05f03sFxyjXPwP4iFjqe4PSTt0s6lNhnRPsXpBz60KH7DEke8Ho9bj6Rk5i4/qs6gya47zefdCIv1r9cHf1QgGCtSvaD3vSIXOkqMTp3MmCged0cNYxSVJ3ejqyIcm518WYb5cMYMIO647fSkxxDSz7VMrr5/c0H9vHD73/4qYUe8AH59WuPpqbMxR0Wd50wDHjZ4e+vAvXwTtQTSWczyBmyjaUpizinfOA8cte65Jj9CV6i/hhYGAAA"
  },
  "scripts/setup_runtime.py": {
    "sha256": "ba2a61d143bdc7442c613a6f04412659e51379f413103856783b09396cf20d6a",
    "gzip_base64": "H4sIAAAAAAAC/61WS2/jNhC+61ew7CEyYMv2uptNUuiQ9bptFnUS5LGXIBBoahwRkUmBpJR1f32HpCQ7WbfdFvXFojjv+eYbUUrnGpgFwkglpIR8SIRRJb7JSV0Zi5cbomtpxQbIi7CFqi3hBZNPQj4RWwD5XFdbC5o8g5ZQJpTSSGwqpS1h+qli2kC01mpDKmaLUqxIe3mNx07Q1KtKKw7G9G+2/WMDsomCiTaODN8IreQGpO3MHbiKopurqzuSeldxlq1FCVk2SDRggg3EgwSjQznzMH2M5lfL5YUTppxN4eT9Gk5/mk2mH05PZlM4XvHjFXw4nfDZ7N30dMJmJ8fHNLq/vr27WZwvUct7GhOKweZKU/d4Pl+Mbi1Uo2nynkZRlMO6jxIrVigZD84igj8NttaS9OZQOXFZOzMxveVaVNaMg04CX4ESsXYVSirs01rpDUkx7hchZ+8ogdIAoSshWwU6aH3zAvgzdq/zijaksr1XNCyMNd2t+/VXoU7J5jkXOvZymXpO73QNg15418MEs4wf6JOwdIjlLJUE9zAaSTXqgnAvCmsrczYeo2BRrxKuNmPGYWSwZuP94iXBEmIx7iIaPA5DPt8ZxGj+1oCLbC+W0SgHy3iBzwEJBxwUwHJs9Z4PL5GhiQrL+vfONDQjPwxO4rfF+SeKHix8tcFBgvKiioMjY5mtzX92FdRDVjgbHEomZDgi/DTjz5CP3DCYVKrXYXTA8Kn+kLa1IEq3Me2woZlAnN0EPC+0VjqmC4cMRws9ceRivQZtSCAAzy+El8AkYr4RRij5M6lwHkE3QLaq1uRF6WfCJJKPcZxkAIvm6AmRB9wqvaWDQzPTYnyllEXXrMrqJt6jghbUyE0XEjMpyy6YuiFCWoWeVohxYpUqPeGg+47fAq8dGQJdepWohn52QJpaAx496zkPfQB7nJD0L4NMmEsU2An/P3O+N9WtEWE86+0PtcsuWcjmYy3KHHTsOD3DFNJfGFocJNyvg7iPLZj90SVNEE8hdlNXjnUNscjx4ArRl9qQl0LwguTKB1Iw7KzTFaHwkCetwRvAFhtfYl97X/YS1pastogA1xfQuq5wE43XDLPId/UmzFrYVLhi0B2UZbD5dvpd6bCMvLZsVYIbgI0bA9RvhyMULkxQOAyGfaH2f7SN3unVTZpOkukkmbjZwaDTvey/ZY0WqId7XTf/os9103N5G07X1x21++P+ekwPbcbYQfMb2Qd6/yW7vrn6vJjfZYvLLxc3V5fLxeUdfXRUtEcyuwUVbOAQpX89eodbg+bqxrPVVvLQDiSJP0C+aQ2dJdOpKzR/ydMugO8o+yGHb7evcz/yztuPCOQXXjhGQuzFR/50djQMr5Msa5DKkLLwG6KXmd9/OiesQXg6iO2EeZ0zN339FXrrlZbXt691CmYQ0DoOuivH0DI3Q3K0qczRwNPh66sEL96a/0c0RhHCLMskwxJkHmJZtsHVkGU0wMjvJ4297D7cknP9VDtD1/4mzsF44GIR0izLFcdS7GkmLM8z1qrE2MXd1DDulXA3KQ2ZxXjwZQFllb5l5OXvJIcKkwTJBWARKmXRnECRLa4D7AErya8fW+ihN7coW//+z0Vg4p4N3SlpA9mxYD9AYQhwxnZ3oUu7mRpEfwJ2k+TPJgsAAA=="
  },
  "scripts/song_audio.py": {
    "sha256": "637781e09561622637fa3ae542b9fa662df5cf302d35b9aa787902ae07670bd6",
    "gzip_base64": "H4sIAAAAAAAC/51YXW/bNhR996/g9CSltmKnaAe4dYBhy/a2Fm23F8MQGImyuUqURlJx0iL/fffyQxJlJymWh9imyMv7cXjPoaIo+kzrtmILWvG9YAUpu6paqEbsSStZSyXVvBGEioI0d0xWtCVKc50fOMwoG0lum04UsC5vhGb3WqVRFM143TZSEyolffA//gGj/ntN9WFWyqYmLXyr+C1xDz7iA/ddHTrNq/5Xd9vKJmdK9SMP/dcjvWOz2acPH76QjbERZ1nJK5ZlSSqZaqo7FicpRMOEVtvVbgZrU9w65UIxqePlHMKSsbFwSSLdNJW6VJrVGbujVZQk1lvaFbzxvn765cvNnEhGi6zN6zk5Sq4ZfnWRSUiYoCJnpA+IXr156yeafJiZ4AzuZDb1cyX7t+OSmcHZbFawkuSHTnzNlKZSq7iUtGZZDsnXc/fEDKnNakkunG+uYv4JDibrGYE/XpKRBfJ+Q5YEiikaDV/eTxbCwHgHawH/JOWKkb9p1bEbKRsZR38ygIJoBKtb/eDShdihYoBPTauKSaIPMKgPzNqOEmPVRgdF3C53ZuB4gDK64e1itSOvAl/AtVEcg2d2QUrblokirrmInzaxmIQ7D1KzCCYn1k3JdCeF28WVxx4XluHZiVXTyZxBBTrddn2FFINTUmCJhtr0Y64y4UoPZzs6AvPcjttZ1ikAKi8oIMtvpjUcUhUHe59snIx2zQ5UHWBLC1S/qQeM3Stl91wBAJMh2TUVvGQKnUVQp1VDCxU73/A4YUpSfBRhBHBesFPELpfOeuytbKNG8j0XtMqsH9FuToaHQTTho0lk0S4hP21IPIptUojTZAxBncf3DQbPJ83xllUQoCK6IQUvS4ZdxmL/0hfhHenAEiWCHWGOZLlu5EM0JAA7KYw2siBcjELCJgaBhG5BtlyF+hTbtdsIe5qL2w/5JIY2zof30WK4cCc3hyO6Z8U7sFV2CsOGEAGCXc1GvrvD4J024w4r9VcINnZ9d/NFdg5NWj4M7nBRNiF0hl6f5geWf82suXg77opxVJYw6xZcmZNocRfBf4Zh4JeFOjTHDHaVHPI3D0KPINlAQJuis/Uz85sSPwxGLRU48APAEKzW9yFmvxb8LsFnHWMQW2c52m2j3vguQDk22Nrwjiq5ABqI/cRk3H9789CY3y6X6fLNS8D8vaftjx9+BbJskUXU0IotGhCKpGuxjEAT0BY7DekZPMwrRjGm8dHFNKRAsNHQXIfyyE6cVqVu2d4WRTRKw6bRmQLZQR5me36C0b5oC2oqdNdUgL7NMv3ZrKfSGTDcZkZyfHBlnuZrip9AyZlava2Ym2uixMoacI1gaYIzcggpyDO7mz+cVUsbG1IxoBU7PSGXl+RqXGnPT45Z3c9rLKfj55dK+huUDJVVXzrCFbHVBIPsPgeuVU/UccxvEAlqtJAHnA/JhDFh8qnMUKHCmDt7k97pLY4SxW0it99tX1oHcILCuNa09u3MJvpxF8ZhTOyCVslBct7Prc/YMJkAUMCRYY7mJ31cgNt4Uh15fDfL18vXxWMIbJMHtq+xf288ErZXF8bo+uqi1xETCeEUg0p2gSlDyFkIF2t9AhcHmWD+U5pr4ifQsUaJY6R2av7H0QGSCwIqgZLEVxfxRO2Mt0lG5TJyy2vYgVoweXO/XTi74HtL+1M6wjXhVIMFr8d6PFjbAw6swcdwqQXBsNZUDyY7EESmIjYehBL+wkYxihKGxz+f7jKGHfySsMKGin/E5ZEa+h4paDE1zeCkKCSDNVnB2qm+WZNAokQWeBniGZ4ZKX/WY7imaLDSO9y72pOPH4OS2BvBRD6tJ3ro/DZTXbWeyiaTnXtWZC6OPeW4NzRpv2O/lc+k8Rc/H2ch9pCCz0rHQek9Lz2wMbaa3JgPSMLoUmAulKmstWQskM59A3Za/n9K6f5uFXNQS4BEuPxN19wCSRquD9sxMj80shivBq+Xg1fY7AZbJ3nvrZn1z0wEgQKnwmiVJLB+qkimUZ27GHq334dhjA0/SxKwcPXjk6+fpbDkuQvpX/B7tVzgHq+XC7vS08qR6wNochv23PX7heLfWPHybdUCxb4LsXc+j0LkJVZDzqET7ZkeUGEIAB8ZfY6/JtcavLI8F8wNOPVgPSB1By3mQO/wVrFnwrBfYXaG+whwJBve0zhh0N8on+YK+De4FHQXJ2PxjALfgLIuwNBy5gnZ+GQDRxh/4+2Z2FxiRiGeEVs4JZDMgcqCvKGTxuLWt+rdS0rqM7pl09Yra3tTU8S8gcHCtv7uwwWkKZooI5BA5nXAxu89Zp3d3I8GrLMbh2F1w/U4gy+5/QfgD5JpHde8ZhUX44uXx+gGinaPr7DG1VnYHcMrpnEMbUq82MUmpIlWOjK+P2CcVvtBD/bHEkyuEvMiwP2+JivCKnB6mb5Jes07dAn7cBXqAAMWKgSrTL8Dr1fJqbox3A5OgHrxgssYT4zyMstP1lh8b83SXa96g9GLeLWwAaKhQeBZx3vTuws3KeCmAfeY7tGIF6KvLEb6FybjRT9tnjxZz534z+YM9xfyomGWKnJMsYEtiDKOh73pz/mg4WwD8u+Sktl/bzaIL+4VAAA="
  },
  "scripts/stream_process.py": {
    "sha256": "d48ca0b293dd949600f01918b00fae195963bb253d09ce4447c9fe63c2111741",
    "gzip_base64": "H4sIAAAAAAAC/41XbW/bNhD+nl9B6MukQdWSFRkGFxrQLE7rLknd2BkwFIFLS2eJtUSyfKmTBQH2a/bD9kt21IttRWoaATEo6ni8e+65h4zneTOjgJYkyVmRkkJkmjBuBHln5Z0B9YMm0zuTC06ENdIaQnlK1gCSfLEMDJE51aDJV6bZsoDI87wDVkqhDBG6HX2xYKF90SzjtNi+2aVUIgG9NTY5hpMynm0nWAkHBwcprIiyfIERZpD6iShLDCUkP4Yk2aTxpeAQEuBfm1GSQ7KO58riOAeqzBKoWWhIBE91fHQcjA4IPriAKcFL4IbEJGWJ8YWOmlnCVs6AME2cUwKFBjcRPF4aWZlSA/70r/nb95fXlyfXZ2fjq/Fp7B15IXl7tnh7fbI4ncxen5yPF9Or92+uxrPZ4uT11Sz2Dr2wcjfwzD+cXiwuJpeTy/n46s/X57H3s9ffWwrpe5VpswNu6cKtLbWhGWBqHG6N73KT1OTREovGaQm+NsqnKgsCshKK4AiLTxpsPx6NbhwErQ3CkuoNM7nvRfLOC4KQDPprlx/eBME2CGUgxTBcMaNScGEEZ4lffy6oNouGXnFrXH2RCrnoexdQ2oIyfBVItlctFVNmaClZsaYcffBMW57hJFnSDc3/++dfBGJVWJ1XNAgahxXZ3DZb5kVTIYHvGOXohH81m/aQDjG0FLeO95dOpuNv1q99cBkotb9sNj99fz0PicGqNCQFnghH+9izZvXiV4wd1wilY0+BLGgC3nf3WdqVZn9DfBTWIC44bJDyWjPBYyyVqxCJY+JJodmtF+xBvPLmNKcyvq/48vCKTCen8X0brmTpwwCYBePgoKz6O/rgfpuKVs2KbdxU1W+6zT1G3Y06iTjiOU+Oee2GNdCjXsLVjpHz6EbB9vuKoaYUjxzvjHft4IIC5YjYykw0r0Y+ApaBifeiDklKAbm6l3G9PKrQbVLt5ANihb7PKArFdm5DFYd0Uctl7yOqLhBsh2opIrGFXBSFH7TS002sB2GbLbqvk8ZMfNdqjq0lvfV7ChiSXw4D19yd6CqB69kGnb3gNgHZaHo0LqUZiAX9PieP9mkp+HFmqLH6hjQs7JPwVSMBpKSa5WQJ6jPFzsdGEoYW5P6xvLxoxGQUHa4eUDi8J1po5RnKJW3FZUmVHXC4p1W10+hJp96Eox5Zp1CoMZldW4XDJZ6sxPLU5pRHA401gOfjQMiLjmz+FpOXh4fV4ezItF/V0TfDa9T14xRUYVE4JSi21vSGzFlK14SmWzA0FLSkuMsxQSVkJiLX3Ni1m2/zCJ/EocKiqpWD4pOsrhQviU4Uk0b/lDKacSzsIhUbXgia4gHzyUn5HFTpehs7z4mWEeou+v5G7/AwwG3wN6NyR2gMd0kzWhXEMGwjyvUK1aAmU0k/2+cUY6CnnXHPFnvHMP7oA1ay1rpv9UOtID2HrjP7xnUBnUN3fqSx953wuwft4Fn8RJZd5VJgrHJnllOdtkc3lLW62OjECd4MxtUQD6FdBs9UCDTrn1t9HNBmzYpCZv6eWITNVTOaTd7g/ekieBag9XJT087AI1QGlbeTfKu6x4OquXcFmNeG41vJFArUkIY+J/Vnpf/H5Py8X95hCPYzcl4HeDFQ7YoRlDX86B3Hzbn5WTA+hFD32I+SAqXA715PprXuo+CApmyEgDIT3+9I+BAiXdfGxk+cAUco1wMdgkhX/y1U6rnzOOomtl+73zE7SKf129hd0/zdurC9P7dXBvehs1qUsgCzdbC7ee6cBAf/A9yqA1eaDQAA"
  },
  "scripts/summarize.py": {
    "sha256": "8def13b7efdaf2935aa3d6f7544d80f683b9719bc9c627e320f14d01dc91d937",
    "gzip_base64": "H4sIAAAAAAAC/21TsY7bMAzd8xWElrOLQLe38FCgH5CtQ1EIikQnusqSQcpJ06L/Xkp2muvhvMgSyUfyPVIpdaDsF4dggScb4x74bAntMSLQkoBwzlTgGso5LwXs4kPew5Q9RrhiOJ0LQyZwhB5TCTayVkrtwtTCLJ1mS4z3u+PL/feFc9qNlCeYbTnHcITNcJDrbrfzOAIv02Qp/MLOB0JXMt36jzuQbwwYPcMA39RLPqo9KC62LFz/QuJCyyTl1JvLyYcScmpOiH51GZEwOTSM1UHiGuzjUx4vwWF1ntH+MG7x1gg/2dmC3hxvBfl/IyEjXR62t4DTzOJIkra8B1TNnsIF6T0rEmVS3xvmKklrfn0YRYAxiGAhAYsJ/YMvfYr52KkPz1LeEouutKt+Y3FFc5m8gFWLjtl67iqYliHwpuDP0vX9K++WW9t5xlTTuNKtCHsQIYYWKZJLkzrZCbfQOj7wKAqeRYqm7U3LSKheZ4Hr1FVaTXiNIeGgVA+WQaREOz3KvVIoSFKuxOkvkv5re+hWv/06GDUzD+uM7EFaIMvW1SEYVDilTKj6N4i6HWfpWcDeN1K+crcRsHrMFFLpxqffUaq/W/4ISTZWNu5sfYI52pTQ10eWpUBIucC/IYRVG37qN33LQvfNY1mFMIIxtSdjYBhAGTPZkIxRKy1twyol923Tn+nU5v/QLJ1HdhTm1r8xPjtj+leR2npv7BbSKdl6kaHcZhzqKq6ej03cYtpRo7jrtYT0u79c8rqYTAQAAA=="
  },
  "tools/stem_eval/accelerator.py": {
    "sha256": "38363c026216828b1de42812cb0964a9e640e411a2275ebeaf5691afe87c25f8",
    "gzip_base64": "H4sIAAAAAAAC/6VTsW7bMBDd/RVXdrEAR0XQLYAHI+3SooBRJ7Nwps4VYYpUScqJG/jfexRtWYqddogGQeQ9vveO9ySEWJEmGQDBEWpAKXntMFgHaErebKwLsEa5JVPe+Iak2igJNQWnpIcnFSrbBrh//LIA9L6tm6Cs8bkQYjKZlLQB3wkUJe2UpCkTy2rGvL9b8oHKucA2WJHdTYAf2ZYIc+hAeVzkyhe4Q6VxrWmadaC68Yyp0GMILhHmR4N+BoLLIuvMj0s5F67Rqc3ZDcznkAwlP/EZFEFETyIe6ZyS9pQE41b0lXaMNSSukyeC6C5+DVVC68yxfv1opxNP8sflwVjtNh0q9vCzNUHV9NU566Y9eCMWgwG/9Pwf3AGCKnELgZynUiGUCpZ7Hq6B2pakcxA9i1giN79oGk2wUlpJBjU8DvMLXJI9Hf2c395CiYbXD3EY8GO5Ak1PGGIsQBkfUKNXI/bv5AwXv7XNnt3A2mqq2FfAhmNX4xZ7ZxqVyeGhM77mprSq2DfWN/EFW4L75WOemLNTGvdGVs4a9afPYkrmMYF862l9ntb5rge5HPKkHMXRv4bGyI2RyYUjT6HgbqzbF3wHwb/LTKJrCLdjzpPc+4SOCXsRnUAsFqi1lcjBKdY8Iy/uhm5qfD7Z6HHTbNbz9YM+88UG3O6/dCfYNDt0bB9HuaLnxnry/GM5RybMwNgAUWMGRx/WfSqd2nGodqg5+VCRo3wyapJHVhwZ3mwzjvUSk0y+7rQjTKr/5LuAnOgOk7/ngI9npgUAAA=="
  },
  "tools/stem_eval/audio.py": {
    "sha256": "a0be4f7be92c39ff3a3ce97ad867f48419ae1ab66ff880570a38f4239c085566",
    "gzip_base64": "H4sIAAAAAAAC/5VX32/bNhB+919x05OU2o7soGmR1gW6Itse1m7ouu6hKAxaomwiEqmRlJ206P++O5KiFNcJOqNIJf667767745KkuQNk0qKgtXAulIoaDVvmWZWKAlMllDUnMmZUZ0uONTCWC6F3EIjbrl5AVLB29+h5C2XJZeF4GaeJMlENK3SFpjW7K5/aZjd9c+m27RaFdyYOHIXHw9szyeTSquGRi1v1lap2kCY1vzfTmjuBieT968/XMMKLpbrPM8nk0nJK1zBynVbNGmLJrOrCeDvIOzOnTxXiDU1VvvZKSR6k2TAjPffr6afqPzIfMutYU17EKXdpRn8tIIlKD1MymLHpOS1OTVZadZwpJP7SYI7mKCfZsJw+Mjqjl9rrXSaXN+2vLC8BPRdc4W+wc1vX+DPN28Xl/DP649JFg8gXDU3SICjeu7+pskumQYERIWDYNIBbxjIMn8QOoo8zzd3litdck04k1pYW/Pk6tiUW2YOrE39Zs1tp2U/GwJw0MLyGIFpPxtCUTLLTiHuV/0PUHTUMaJHIn04HWnPjOGWEr8xabqcAv6jWE0hx33v/nh3jQATqSwUqkGJGMPLJMuOjnB+B3YdNKscOKQ6MFP0ahNfeOpFNQXLNIYFCcAHu0KDZecFuFrgy5YJucrnz6aooNv1aCrQiXr7BcVYuoVTkHyPbLVczwolS+F0LJVuyKTb+QKqqmn5Fjilm0HBq5ZtMUOdcgP3wgiJcGTB07HVKWxQdRllOHHx4KpUSPSnqhWzWVxNBWAuTCUksnRvQ1ySw8t7XsJLlHY+ROqkWGpRCBs5wxrV4GvTGQsbDkJCiiRe5J+TmFhM3qUj6PuHvdr/kCv7bKzoCtfsnV0XzyGaPpT+EDeFzuZjz8de32NhtIaOoPnFI6RUyd84wMDDC7ZerXA/FfR4aqDm69jUt56moULPdSfTT4O1UflNE59KCYlrJpWxpZAklNme/roEc69+UJDIUY4+77NpPDOZGdPPEVh3nA0jMUlokFU4WiV7VXcNX30lNr4l44OYDttIvX5LQcaXDkFxxeh/rExrs7jEUuLXegEGQJ+nUOx4cbP6oDvuyWCF7bBBrqDGghK7S9iVwTmkSzhz9WLIsQ1WXL9tBkOev4IFLr/fBU7E7y/fboUBpBnMDvvelWOeG2oMX2O4DKYXL1RJg97cN5Pcq8x+NJSfPVYBLEwYPUE1ak82zZChoaIg/GMdhIXU/13+koSGTjkstA9ryf6QluygJZyPEnH5GzO3ciOYvx6X2+HO8+/UKwxSxssUi3oAn2WPlZGfObPgSImVwwNyCSIKW9+hvQJjb+j24y9Gop1pXiOkPb/PeYDlOWdluca1xY35ruBv0CrRrw4yPLKi4BKBD7X9Pd2sdJ+CDqLFRolrqdGpziJQax0o7E2ObDvDS1OrYKtFGYv6cFWICRx06KZ91wop3ndiOD+HpZt2DqxxbovNdYVUWCcwzPp8ni/CERQVxEeBce4MhKuqwkiQbdXJMnWrxoLxWbTuiaDbQSDCUU3ZRoJy+2aOr8wVwaAlZ5pGyXRkc9Sd0R1hu5Ljwfl8sfQdbjCHdzdOjjyPOyonNlnc4Y7FZZ6f3LFc4pVzXPYdiYRBM7nFFidkOiZu2rM8C4SMczKyHFm6WD67fIY0DejPvFgwBV3BcS+twKcB7lkAcX7E7ve/sJ/ftunscrRvjDi7fwD5GC67ffdYHLngIokZewt0Ez6DNIT+iT8/w4dwwnfbQtZ9cts/437iz5Hgrj7pjJ6fT4/XPfGIA9Th6hlvVf210ssxfsKs8RMmKpI+M3pdDtp7XR/YnYEOY+0+g8AvfwHINooxnz91/djADect2IOCqqvrmcH7HQerGWo+ZkMUotJiKySr8TrApfs0KE+IcjoaQWyxr5A8+xPcFwUNxIMerXG/RnP+Sy+W01JUFdf4NYeu0KcHYJfYygb1RyMWcWC+QouFDGlLHuI59VmbMozIJnOV4WnmUoZhpSMevog2PeE9fYf8BzMi8gCJDgAA"
  },
  "tools/stem_eval/backends.py": {
    "sha256": "f3509ac5232a70faafd713ef75c10c07f4a9c319dfa45b572a8ed68468d417fd",
    "gzip_base64": "H4sIAAAAAAAC/61Y3W/jNhJ/91/B4z2cfJDVuEVaID09ZIu97QLXYm8/+hIEAi1RNhuK1JGUE2+Q//1mSMqiHG+2PZxfJPFjhvObmd8MTSm9bljvuLGEbZlQ1hG34wReel473pCht85w1pHrd29tQd5zJmG25YarmhPFeWPJm3efCFMNuediu3O2oJQuWqM70jO3k2JDRNdr48g7+FzE9y55twe7iBuM3nPFUPQ4t2PfXn6/WCwa3pI9k6JhjlfbAZ6wKoORgS+vFgR+oPa3uMAbYYceRYARr5jlZNxDDFNbTja81YYTqVkj1JZ0uuGSNLznqgHTBA9moFzREmERmkljTjZayyXRhijtzkxnQrmctCDdLY/L0OZC2FYo4cajj3Nr8o+S+CF8WV8Gk/BnmIDT/4ZTr43RJqOf4DsIIdc/vT4aVtmaSfQduVnn68tbuvQyDHeDUeEsUWmEM/irEgowd9ocstHZYB6r7wCJnNQ7Xt/12pvjMaoU63hJWc2t4/1qv75cbQBeGp1gNBhTTmHzDcnoJMNSRDMKJ2VJUA4lXIJFNIZPPPaZdRMmfyWfooK/WYBVqFXwn9dEmLSa8AeMYEs+DmajfXj+6xfSatlArOeEg8nkXrgdjjbCso3kTXGU36JXwE4EM5vMzsnMboeiKQzuGcfHv++5+m71utvwBmNqdVF8/4omey5//ryS3Wpd/PBqhGv8gbUYBZmH7xuvewmhUjXCZCdLz8ZES38R1mIgSw1RQGoN6aPAreQRZT39SMygFDek0dx6VY2+Vxj8ZA47/uzQtuIB1pXkkRaWtdxxZbWxaEyxEco/e+cfv1sdvj1M/s09OPo0YSmkF2V9Lma9B7dHZNHYwmyl3mT074n+kx9g0yMWKChbelfSombgaxqyT8F8zww4G+f6Ipwfx0dLln/Asy858by3mDpkqbaXwXqaDJ+7eULgj3n6Vx2dHD2HEk/9HdHExJpkwrGnVATSIr/CFiSgaXS05i+QcjNj5id7zkkfgGzlwfOuBgm1gPN9+Hj95jVJxaT6ffatVtNIEgFjzNxEoChQMviqrr779m73OYTeOAUZZTsm5WosP352Enu7SFwWBIPJ6DscSCML3VMAj7tsCY/Kis8cuedicpzfvnyJm9/z/wzCQM0JhARRKVqGPMSg2HQxR0Ec73p3+DLV+Uh+FmPlqVf+zMkgbKBarD4AE5HUJWMUtXpQzbxm3DxSLOH0igDZZvhaGG613ANaSwjrUJxx2r/4FcsY5/B6PNctFJxaMmvJdc1fBVPDUbEMVRXWsqrKLJdtTqYiNPmwxEj9SgHKMfCQ0Sqnq7ofyn9CHYBNDd+LGjbUQ8Ng0bxalj8UFwlqeILipJ6WZ9qO+ZKEOQ8Qfr7KK8uNyy5yj9xo0jKJcGx3og3FDrwtgZxjzwMgoZd+DqPnt0w9WNz0hgO9Mye0+kmrVmzzZOQdM6yDqrcNI7zqBivqudW9X1O5Qw+Y+ZHay0lH5vsBmecqnh/jJaWj4eWJzdkEVMetZVs8wVCD7b6SJFsLjB0gG8jVCiBHV2czqoKW8ndoBCqki3LmDAiwYCF6rExrQQyZ8Mhn4k5ibP45XzpYXrUQ9buKOUg2xGSMSaRrSIzKKx0HcX0nH6Dmuzh0FLdcnJSeCMY5Vn4/gKouZn1Eb9oPXDQwWQXTEEtAZIZnmAEC7KVwGb2iy5uL21T9XAIUivD21aO0nhsxehvyGPY8TdlOIrHA3EwBdi68HTxtWoAMCkcLZI98CVx15JAxyCKHWD2YGh05hFjM8UbjzNBx7GMt3FqALQbXDy5J/pABY3wl+TAPKMfsnR8uqeRb7P+sqSs2NEL7+Aq6l/NYCOprHwMtjUkSrimP09GeiDNgGEE+a4hWft5LxlB1/MFd0bncGi5vQeZbgI95bFN5rMZQY0p0Hjktpb5HKP39yB+U7JjptDrkBFyFU1h5mgMkgqjhDnTODJTFZPnR4E1HHgwsLOnN22Tulk7Yl0cnzCOE99C3O9BYQck1rrwogC2TUagR5Wp9qj+yXoUkaMvLi2eEfobCg8dL7/a5K3dC3YGuNANr7aoR1ZPhjoPvTwcl3CYHSLHTlPXtl8J7RaW06ZChWCL0TGoHNhrjL2HfbMNcvfP9SLkOmuEO2+iu8nbFA+G7LW/wcZuHqKla1OxKes/2SXcFqTZINyqa8/OMDHIS6m4fyb2OlG7ZnuPVxMd7zKNnFBW0FCNtQ08A2ZvFUX88u0T6WH+VOeIejh8oZ5QMndpgq0hxOIE9DjKFHLMmrExM921J6f+JmB8FSO4mNDu3zywJFX1sE796XHpssyCawx8QIwjY8wJekGPxgChy5hnfd6HCnGDzhbhDfxVeCqgyjaihPYXeK/gC5sLLNPd0bLY+OMDlf2m30o7o5Y4GG3CgP3ra2VhUXWBpnNoaXyh9xZs3AaFTLpP5LP3XYda//Z8pfzwZdsDKk5+dPs5ZNDghbQiZmVl+JCaGfz9uPoor8AwV33NzQN7ZZuFM99rccWM9m864AOl+RMWLHCsLuDqbgBsL9nHriEszJnhYdoRrFr6qsqzroUVHYoElwCigGnhkghEaKLirxA48nquMzzndWXcAAg4LN5yNexpuayMCo97g0K0Xuq02zIQicoYMY6IGj/ko8/gjkU3Rc0Q7Oxqd+51fzKjEREidyUiKzsCLjPcJTY4Mg96ML/w7QacoA8HS/2labQ5Vcq29SiLxafFffIcBi3MVAAA="
  },
  "tools/stem_eval/decoder_capture.py": {
    "sha256": "bc98dc466948f59d88b2efdc4f15dfc0a0f1f3250a9684de20e7e2ed106b3b26",
    "gzip_base64": "H4sIAAAAAAAC/6VXS2/jNhC++1dM1UOlQNE22W0OKQQ0TbOnthug2ewhMARaomJuJFIgqdhGkP/e4UNPOzGK6mDZ5Mxw+M03DwdB8KXRp4xDThrdSgqiBL2mQLck19BIenp/dQMV0ZRrBYQXIFaKymdaQEFzUVCJmlWloBQSBKfwXaySIAgWpRQ1lC3PtRC4zepGSA0bSRrl9hqi1xVbdTu3+HPhv/O2bnZAFPBm4YWleKac8Jx28mpNzn+5iNEi0zT7rgRfLBYFLVEts/ohuqyEjC4XgA+6dC2MUXDLoAVc336FshJEfzyHDdNr0WrI14Q/Mv5oQajxftVPyqvYaxlbkiJSvFstqCb5OoySvGnx0xrEt/MBV4V5ed8YVxrRyjzYIR5WVFTGUDBJESm5G9z9hlBZRL0Q+uEQh5VoeUHk7lcoBHDhnUZkMEKKVmgIo7Mi+RM14ZKItCQ11VSq/gr9eZBa5MPBgel+Uj/h9xBNGAKkd7KlMZKDKZ2Jp/QzqRR1GkIyxI1UaNA7nGTO3+yRciqRQVndKpZnSKoi84zyeNqIpvASIDy6VcElBBvCNMYhiCGwBMO1hyX+sDFBy88sp7imtOxQTNxiFFujw9Op6F2zp2HWImO12mbPhGYdfCi3Qt72gq1C752MTYJuvVtjygbib4zXgfO98bEHj8gaPXiCLmRTsdgZi14XLiDIHp95Icdoxi63mOCeMeb5zaZX2O/0G0ZbS5LTIjwh8lHFcHLytDHfRtrmoZhlLhKeP+iuO24UGtlybkLzOlFlpZWENIWg2SF38rW5TnA5gwMGPuJBnG51jzKKJwNZwyjaU7X+JW1TIH1CF+/UwNlr9SQAC+N808Z7atax78GzbJmQpsF7h/akqaSWu/3LSKraykDWob6H8OFLPHSALlE3cCUFeXfAvi027phprLY5bTTc2BcebAomlVLIy/dhc+emQUlYhSfGTiktgxeDTmh/RUmWmWhm2eslvNil12D/JpIwRRczVx3RHG3HkfXVK+15PKFJfEA2mtgYEmTPkN8ZGZkJR84dnyBdrYr61PIFuQjHxQlj+GayINuHmgkfIPA6CW92QZTYCokMnobCwgX3pGrpjYE0DP7oe6jrvlhHKlYzU8CxQZnq74snxneEf9eP06HbjR2Pxm72jhWshh9S+GhaQreIXbShDz8vzcaZ2TBVjDcJUyXj6EbYWUwwO45f52bbuO7jtHs/7TEFPJxh2TK5iJiansVppZaja+HBimA83wQ2hj406I/YZA3Lnyo67kNDSs/5bpngWoqzklm30gpDFU4A2SvhU9wzN3uk7vUODY6YqRlP3bjQaeEKVr3ev5ps5wJkawSOuKeLmRquhK4eIsR26+JTNKqDwxA1vY2nZWJ2EDaH66C2VxD7Ythl2JF8mihvMPQ4RdZjVjuDSNBo3ms66Smv+9U9Ys92zuzO+SHKd5IHOf9WGpfEXLwbiQtWWLO+IvpsUNiCqADSFkzMiumP8A+eaodObwPHPRsowKm0wcF0ZXxyEkhoM/8V2AFMPD7cXv91dmHn6WSKkp+UzbhYYq03DUKVExFVJjb2oemTk9AX7kqZdSJBTJDOPYgIbHL3Bgu7CqxI3VQ0M4NfbP4d1ESnwbere+SRaleWjMHnP79c3QWH+vE8ebsCbbQHw+mhw95IjjHFMleF0hkpTs+W72lPIHmvCBzAbn7HQ03dN/TfiaLHmvphkP5nR59289LkbzVL8P9YKKb9+/jfAUz9LtCLfwG8pH+5mw4AAA=="
  },
  "tools/stem_eval/prepare.py": {
    "sha256": "7de27593740951c7e0be235c8337d3450c28f5fa8550e0b201b8203218a746f3",
    "gzip_base64": "H4sIAAAAAAAC/51WS2/jNhC++1dMuRcZUJRki3YBtz6kbdrTtgGy2EsQCLQ0stlIokpSsbWL/e8dkpIo5rF9+GCb5Mw33zxJxtiNwo4rBA5a9qpA6Opeg+yMkC2v4fbD1W/XUNSieIBCtqWw+xoqJRvgLYg/bu1hB3jqpDIZY2wlGvsXuNoTsMZp/aeW7crpddwcarGD8eCGlpOQPvRG1Csvx/tSyEmKl2XuaOgUCt7KVhS8Fp8whUf6LbnB3IgG9WhCyUdseUv+zMj87Xffp3BUgkQdmdWqxIpEXQAS734avBTtPgXZm643680K6PM1Edg6R0aYdepXS8lpb4R0iA0aTtw5aVtKWS15qSOtTCEvc4Mnk6y9jqhmtWyPJmG6OGDDP6LSpMLW8M0WLj1h+1FcaISPvO7xWimpEnZ96rAwWMKtbPfveRf5AxEaXDJvtOwVtxJEdDJ+xxT+1aM27P6OTefs3om/cZXhY5Ed+SMIDby2rgxglGgaLDMY1TNtOCWIJELWUmgl5Zy+DqhAIz5kDneH3GjiEOc8iePhhFgKd/cU8onY6IY8tv8KYxZ8GYdyYAlqEp0l15nQut/ZPb8OOXgDV11Xo6N/vuMKnEnDm05DwwcoRVWRn7sBlOwpF5R2eM9NcYCjMAfRgmxxbIdK8QazGZmI8HZIfLSGhO90UsIZ7NbwI1UBnMO3by8uLqCShA4E5Jm5dWnXgX1g+3LV/DKHrum1oZpRCnVHlQNGgu7JP0El5WPmg+QLPWseSqES22Ot0dsPqqfs4klok8uH7a+81ujFjRoCh2WLz605dto5sFBZLKRmG+fIdaybJ1khu6ESNSYvdi7hRR1nO5EFjMXY28JndsU2kf0vy1w49+NIhsGVvMI/bFv1cdMhpSE/KSxC9ZzaHfuJ3RO/ABEJErUZKab3XylOMK/StAlePzOxZPqzZxohzQoNb0VFvWFjHaGMYy5/HOfcBi5TCHNnM5cB7WrqrBpzWiMduBZIn4A573J/K5CM/xOmd1QRz8TisR4jV+KEZT7i77mw1C6yd0tITVth8UR/PghWPz/gMNueU2EL2ncyHaduaTs6AGd00zU6WX95YoGGBVoK7Hfp8keqxpAnP9BVjnSpjw8BXh/5oKHX1NbWSk0ti+6KaMhFGowx6pEre2iB41E6H/hJOmuFxgl38qL+xnu59O2YzoURakuh6VU7H7h9PBXYGbh2PxSEzdNRoBqjEKNLeJ539CKgRsnzlkZsnsOWijTPG0phnjMP5F40impzet1kV2rfNzTZbtxJUqIulHCmt3leyiLP1wvNzLYaH1USdnbmY03umaHDrX0fpO5eFOT5NrTSK9rLOvy/GBSJf1QleTv8RgT3YzGotjzy+ISyW9k0rN0inrduS9qw/w3L3cPneQoAAA=="
  },
  "tools/stem_eval/provenance.py": {
    "sha256": "c76f5b079e782f084a86afac81e7bcad8952c85aa74101edd1445074c184c598",
    "gzip_base64": "H4sIAAAAAAAC/51UTY/TMBC951cMPqWQRunHpu2iHtCyCA4rkFi4LKvKcSaNaWJHttN2tep/x45dFRaJAzkktmfmzbyZ5xBCbqQwKAzUVNeoE+i4EFiClr1iCLy0Nm44aqCiBDx2DWfcuAUq3rrATsk9CioYpoSQiLedVB6u4cV5+1NLEVVKttBR4wwQDF/s9uyk+8KCMdQ6im4+3919uv8Ka3iOwD6EMiTXQBid4PKqwtV8lk0Wq+VsgnnB8gIXq4zNZtPJKqOzZZ6TxIdpQ7dDIM0mVwyzopxPGcvYarFcsXlVslm+nObZirGFha3mhQ08RVFUYgW6ptOrPHYVj64HuJJvURtbVKCXBpfRYD1wUw+EfEgqOxQxUQUZAdWgjULaehz3VFIBq3uxAy6AG1RxQ9uipNfBM7WvMp5k0zm8BvcZJVAQMrogXCpK+66kBuMBzxej0PRKnO01Hv3KlurJHZTNuXFzGapNYE+bHgO6O7EsL1yGU4NuTlQ9WZM7TR3hje6rih9jkpq2Iy8cU5/F4NHELlVa9m2n4yFVYnk7da2niVWbMpsdPun1vXIW2jTysLGiWn+gjcYRvAHyQ/yFrrBrrC5CiZ4Xq5HtNn3nmxi4FZTtUJSBHZNty90UL4JLfZjsTdeb+IFsuSEJkPGNfVskn8EeKNyPO6o0OuvH23fvyWMCjt5Q+Ci1vrwLcuDVOdOrNQQ9P4RKHi9TVJRrhO+uJbdKSRVX5NbeLmbsLeRCh9VzCDydMZ9fIp4S2Ep77u2n0Cyrf9Pr/+LqQwfT2DacYUO58NteGOXSluOKN6jXQv7ZhzN/D/EPruRbGBQEQBgA7f3aox0lFVss30Jvg0yNv7Uj/J7sOLjm8qyMoHnfgegXQsDtmt0EAAA="
  },
  "tools/stem_eval/run.py": {
    "sha256": "156fdeb26589aef3eda9530e2d2296f5f2c135c9edf03ae09da7ce9401acc41d",
    "gzip_base64": "H4sIAAAAAAAC/7Uba3PctvG7fgXCfhDPuaMl24kzctgZRZYcN7Gtsey2M5obDkTi7hjzFYLUI5r7791dACT4OEp2W33wkSCwu1jse2HHcc4TnrG8ZOJWhHUlWCmKMo/qML5KBJNxtk7EQlYihQmFKONUZJV8xQpYJVkmrkXJ4rRI7licrUQpslB4juPswVheVoyX64KXUph39ZPEV14qKh7xipsvf8g8M88przbmOZd7qzJPWQFjsE5DYOfWFKClWuVlat7lnWwe6yvYTShkM1LBDsxzXcfRngLP6yjODfCQZ3kWhzyJ/xJzlsSw/QwYEaTxrZ4dhiIRJa/y0qxJRZqXd4GseCXnwEQpqqA7JmFJWAWRuI5DACvvsnBTAp6/hAIKZAObgyrPE2mgluLPOi5FQOSpT/NmEF81c8r8WmQcmG8Wnnx49+7tp4s5Czci/BLUhaxKwVNAu+HPfvhxzm7KuBIBcX1vLxIrOlG3gNPnpYjm7IqHX0QGD3EGS2s8dtyCiOTsaI/BX8qzeCVkxXw6Oy/JeSTdBgJ7yhzz7OEEZ+YBBVFQidvKnc0IRrxqwHhrUbmOBHJTHoBUyRiXsO98dqjw4V/JYynYP3lSi9OyzEvX+ZzJusAdA0aFjlewkilAToNGbbtDXZhnUYyT4Wg1hYjOEHTZmRAoAM5yipgTawEwnmdrwMRXFeiIRVtLVMtaluWgHBlznSsupTNnTgQf6GFdxxUvndkU4ouGBy1ICconGIKbMwKGSm6AjVDg+w0uxrPICACyxOGhcKbwv6F1LKxLsAAVGAMtopJVG8GOT04XF5UoDMgWPe6aZAppS0Tmgta4SsjoLNQQvuIEnt25uOKA/ezTMvYze/bkyfNDBtqvBoCFtozukBl4r7P4z1po5FeiuhEiA8C48WeHL16++On5jy9eakIbOZAg65fOsbNE2g1/fM0fJhKAi8bCHREh6SwVsD/yKwKzpDckvJmD1LcL2h3IvC5Bs31mSe84isvmZdmsNmwmIF4sg1WcCJcYqsfgnPLkGsY8hA6i8F2Lqv3Y0vOA8GtbmtZgG64E4wwR4t5QFpothElcsAhkJAQjeqc53VVWRd4upWw00tr019L4kI6aI+rIVhcJnqfHiwJEwb1vqXOO2qMEJW71DD7YFtVBkDCGP/MO4M6fo3iBE6vS7Z/bbG4mGKYc9Xg4BTqq9Z6PLDY3g8utYkUpqrrM2H3fRB+xQ8QOLq6W8OKgG8lgSzBo1P2ocSYdKhzjlIIwT9MYOaO91qWevwQgyF/4gj+95WPm+eiRhr4PKhEcBGqchTYozfkbfj2EYZMDLxJOGPfuKFHLs+TuFYjkDUYgMOVpuMlLsD3AVx4Dw0jOxDUIK1cyg0qb5pFIQF6KupLOVjvqkMMJRAFwxG30Z44MAv8crzM4iFJobSXN81s1wy2A1NRJpfhg22Gc64lbsF/SVnZ97u/zTGg5oPii4/Rp6ZhzV5O1a1ciQursrEWG4RMyaCcmZEjGU6H8Yvdwuipl1GhuCfPcOhHbeXapQvhEEbCvHXjYjEDEK9Ddhps6+wJWLsmzNTi7HFi9ojC40qf2itUSbWAmblhZW74ffW2EEVsgIdAE9wG7BCJsdoQgIIDD77Cxt2ze5bGoKpAtjBvutzM1hiYt7YwMQLz0DmazLoc0bsWZy/6S5f+EQwboTh7ZW1PpCdl9I+RKlprX6cjsEfSAvKzitRagcaIoEfKZa2tUV8ghgkd1IJmEF2UqZq2t7oQ+OAE8cikgg4mvMaJvIVvmnXy1tuewhLatsF4qjI8KTslsqB3EkqWxxOQOQWsvCNZJrGoaBLZI2IQO5ZARYJ47fkCh1xZJp44uZHsQbLZ5orFCmKJok95kNseh+MWkGBcVX7dvNyJeb6ogziClUbYNrCJICqQrRmIIrHmB8xhMcOE0eAWuUpE0JfIjKZY+pasyjtYiKPMc7R1mnH2w1gyASTMCOv4gmA3CKnn5bKlxQqrnYULrgRETZeUezMmv2wjR2YRlXIDh14uIjSCKmMEGIruOIXmkyN3kisNPtM6e6o/N0tvt2ttfz4JfP/8SnByf/HqKwgzv/z791L6/O//9l+OT307fv9ZvJx/en7198/rtR9va5tLTiC4R9BIIsBCrMRVjd9JU4q/XJq30qiWo0SB7kKJwiWKkkhcFgIAWeYw8kuRYKI3ofvJoz6BRKwVgcU8TWvey9SRfge3IZF7KB/KwT8dvTrvJT8qrcEOKtlrFYcwTO+eyCESpQUq0mv2NfdoINECQwyU5CK1dfvHYv0hHGA8Bk9TJg8QpCeCoiAVSBbNCyY12uDDR2+sdjDnqD2dnv799f+rgITmHzmDap4/H7y/OPnx8d/rxYnSyqbDkZbihAVXqYH639OHSBH2oauhRZ6qhYfgQ1hEfJKQflWTrowD2IacLwAtWTx0Mj3iBgX5zQCefXx97DPPBJkcFTh4X4CvYRQy8BHYZd2TMEWynb6GmxHXelzc9ACljQOHd7r13s+5vshmN3SBUAZcQITQmGEQTJEpXuUyZzKzS72pral1g0gO9BbPhFpUxLqRSsAPz7rDvqdjm4T8vwCJuxO3l0U+UR/dMKoanKVroMw759Exl1QaQ08fEfHsXjyFWv+Y1ssumdzZmtByQi4tPp+cBWL2T384/vH3/6SJAG4dyj+zXECxnrQw8sA6V0oeESUmLpMSvcWkOHQiMdYVhpxwoPkjNJ0ucMPczBVAAZx498+AOcz9HaRLMNrVIZ13U8EqK6aFyYTSj1ZWY486QMqPPRgP14XT0Bbj67vwCfYLOEHHj99tt42IK2BnotPIyhBEn0wO5YHoreSaRegCB7wlyEBMrfMnqtLiz7XBV3nXjUc39y5aE5aVGi+c2LEJ7eqKrZ7WyIG5DUYzVrb1zNfV9Xp3ldRaR2emSUXCtT81h2mkUijmMTIu/EiYT4Q6nj0XFc8LS+vQ4i8QtJYjIcwHso9TLbR3Kpcqyl9006QF7pIsfw9wAqbTjNPIETbzs2/q32r8n6o4Onkfb4J7AtVnb0gxZqR6MYaqnxinpW273h2i89As8u5TOBvkXX7G1l+PQCaBneVRGjQ4Wg3Zkbu+cS9BDd+Vcqs18f7h9eo+VS4vB+8jg/eVsu2TvRMq/8FgnIugZJbzdQQgOjj7JpQoKgDY4yFVSy43/qawt0nVFEvK8Wtj7MSelI52eRlScysM+9T88oGsVhCC2YDrdLuhWVt02RO87OOVN0DD4Xfs1xzAHiwIQSAdhUavPemyuDYhvzE5XcnzzOttVrJo2jnYWMeqRLaPZRTH0QiEECnDoYAJD2FZpdGq3DDSOVq8INATja1GEeZKY4cHa3ndXb9GD1CxKRGl20CWruwuroWRCLDu6GhhHdYh0WFKg1knS3REBYQsjQHuPOY4hZxqcPd/cJK7oTVWCO+bCZ7sBCkjeQ8qd1XaQLwTuclSI7h0MoXQZFR+pdNov+OH4dnS9cmDVhorB1HFxTQgA5qwX2HnrBCyK88RDdzXcw3LsWAJdsrOMpKM/WSW7cTNmgxip5I1hgriF3Lttkxt8RQnik9cyuCeZUJESgNzqImoHcttPdG0Ec4OunQ16JtKi6tUlXUeP7wwWZ9Z2SeGb5UOz3t21hqydwqxnvNUmHyyTWjjMmgkmN1M0g23YKwP80Qxuiq5RHFauckxUTvUxMtaB0Ugs4Lfua2/koHZu2FQVZ3vf5uPe6PIuIz+93/rv/eWWpXUCDu/eVFz3ybQEqxLYtD9nB7Onz58dHBwcec9WW7nDBw4ivmG/fdIEPtob3sSg6yosbu43kHaPnXnJb1DcVQkWgBsbbordLhWIVOSi2jnL2VzFUG27ZaJTs/NvECRpqCo+mhv5/yZ3oaubdYH1NbdlgnYZ/qS7mNpMLxYwfPPNw5w9eTJxoD0Pnq3ybk+ivfmh/H4A1q2oK3foF+zrFK6zWsGyK+GgY1hcY74hMLLHh4Xc5DcBMLiMBaUlmKHwyrd7Dot8hT9aiyhD5DddWuFAsE2iRHoQiPAr6WK0VCGv80uNAxIYS0aQwz2xmbG/swPv4IcxqRx0603zhZn1WA2mQpXqUKmOVaczpUsp8cDu23dlXNIAY9+fWhXweYPK78s7WwMW/3BCVFJ+GzTLe4EafgOITQQzZ4cHPabazT1TxzXGp9P5c3SUtavh1wXbuRbk2nB2scAahSUK5oSuGQvf9srmzKr2+yZgGsE11e9lTY/CR/EcX++NtiRms9lo4sPYhU5jIp7wlN3r5sT+wFzsL4+8w912XWfbp/RDbXrJxDCxHufTisOekEm0wl8599VdAWkuvs28gNKVINgesXsa2u4OYL7CLzYa9l/4VzsR111SKqu2B6/7LCloinE8dK2uVJEiPXvH5ZrM/zl9cSOhaoGoNEEQ5WEQzKyVHo+igOslrrNYGKkH4pBvPupJc9cssg5qx3rT7ccrZzlYaOk3d9AeD6TT1DVwJi5FoTtYceCrr+ZMAqfbG2Z7VInNUN9953unhXN5MMcbDc+Wk6DAmXwrn8BiLcD8LFqL1dLT7PgQqHh+YG0PBzYiKXznVBf5kzuWY2Uqq3JqNDw/0CBZnVFtAZudeYauBvyNrKZ5ozPLRZvwWptTiF/HfJ3lsorDI2ayW2ypHJ+cojfax/4DDysGhkNdP6OaPZHPyKPp3HiaEFMEWJhOHdFB61tuvPQadiD2Xzh4uLabfMiu7rD2hgX+szeMp8CwFbgoda/CkFQKVCfJXk7To3ubaL7J/fmgojlGC3DO0ytN9cHm5fQZNPWJMfarFkarIAt0yLgvZrWnqI00TRbkpwtVfLY1DEbB+heL68MfQJMl9a/tsdtEDdsaN1gzjZdy+oXJ6ce2eB7j1SFW1FfgXDcCW3VVvEKZMqteNYpGUQkefpP2TKPXNe/OnuuKqs1Uy8ZYopDd/eH3aTOgClvjwmF2RdcNtSywk/PPpkSGXXbYSA1RyRsYVbGuxod2CS/8KbT0g4hNZ3q0oz7eJe+kShTedKtuo93zkXnGVZGDbqPJEQ+tiaZRF0MM5YHH+1zmdilZizHyYAJo+w7oA3uBBTqeUTcPL1yhhezcuCUM5qZ7g7I5nUHttI+tvSSvO4iWmj+AZFC0bL70ql0PUdERepuSnoaRhjQadA0hhjGCPVJH8A+LnE3JTvWpOrnWGJTOfTBdbHQ6t1J7EL/ze/WzfjOlwwNE1l5Bp7umlMAw28D1wyvUKLzl3onyR9u19hVNGlC3ijts6xVjbbZZdxAGgq5vOo90X+iy+2zXsb/WaIzrbQ6evC7KeuuLrQPWd6ZjSRVmVO5BAuZi7EEeVl01opuH1GmWqziDaLa/pBP+6gUH7Od+TorXtIkHvSSNBNO0o6aaUF0G/ANWNAkrGiK8uo2RT3PtYRBcQZKWNjeXbFT9tHHJxkkdLNQ9UWQ+Or+oXam+DBY04tzObIYGk7vNk3aFHn/kTQEb4khzbsTKdi477zyZx/X+doE3JWZdhdV3oihextqlbtRZbU8rkbKr4S1dJpeyLpyNmeCJRvHOi2tfm5FaRP3fstKvY8KDyenjwe3tAUcNpSRvQYCxcxA45r8CYV669x8pVWlATTYAAA=="
  },
  "tools/stem_eval/system_tools.py": {
    "sha256": "d31e7ebceba70244ccbb61f9f994939914057085892310ce8c91465a6f67a3b7",
    "gzip_base64": "H4sIAAAAAAAC/5WTTWvcMBCG7/4Vgy61Idj3wB4KKYQSktAEekxke7w7sT7ckdTdUPLfO/5Y7yYtlOoiyZp59frxa6XUjW90RNCpJQ/RexMugFxjZO+2cO0t1ox7qMlpJgzgUwzUSgN8TcNrRIYe2aH5FOD+8+N1qZTKOvYWBh13hmogO3iOcC/bbFkPRsfOsz3uwy5FMlmWtdgBY/DmJz6NXnKnLRaXGcjAAzYp6togbJaOcr+jZjcXTTXUnZXNbeNgjInd2dGx+GikDK8hos0L2GxAXWnek1OnfimBgbGjg6CBXFV+iNVuQVMJGnUBqkqBKyM0zfSkOLWPo9GupXYkvZlQ5LNeARWM9t/VirG1vKTw1JHB/IPe2WuFyPlaP3NYTm69w5Xqj0T8D6p/ov9/qqwpIHxLLpLFL8ye8xNH9WtUfYMoXntoSZinXrsSrggeka2EzIDVzd0DvGijnZzB85Q/ciFqY6Dr7IDbZ+G9qiqjTRq9Ry1ZmtO4hrMVhVUqybwF5yPW3vfluYYYuCGXDtV3cq3fh+XC5b5JpusG9jWCxfFCkoA3vd7KXjuZGO4eylmw+MB8+rcmqOH4GRd2E47Lv3ycKXLjag7c7GJM2WJCFW/ZbwM+CQu9AwAA"
  },
  "tools/stem_eval/vae_reference.py": {
    "sha256": "62fa0c1b7113630a99dbfd3b99ec69be4aa1b5253d895c6774cee3aee5178104",
    "gzip_base64": "H4sIAAAAAAAC/4VUy27bMBC8+yu2OkmJ5MTOpTCiS5Nc2x7aXgwjIMSVTVQiCZJy7L/v8iHFjyA1YMMi9zEzO6ssy76pQXLk1ZuQXL3B08/fwLFRHA0YbNGgbBDehNsBHljjQO3RdExXXNiGGQ6W9bpDYE1DhZyQ23mWZTPRa2UcyKHXR2AWpJ7NZhzbVPvViQ553jGH0tkynZbQ7Ab597U1rEdbL5Zfy7Fd/bAsVjOgD1V/DtEgOCWLhnWwXpRP5a8NTPUMOiYkgQFkzQ4aZRCUJ8Ikh4Tc3zZKOjy4ANkXF+1YYy656OFLDQ+gzHRod0zj+n7jLxbXF9ViA4/xQipir+fCtoTD4Ui1mLOuyxMV/zFMWIQ/rBvwxRhl8uzloLFxyCFmUiWJvXbHS5ZZMUL2ve7hsR7FIgynQt4tP2v3I+X0g3XQIduTSKCVFU7Q3ziYIGDqZ50h4aE+awEVLG9S9xClmfFz2Cnth8qkxM5SznpTwnfiE39DZEtiWUfhICTBk1vM78trXcvU+EQ6pFnW0AuZx/zbFPJBdjElddg68ofY7pxPZgfqFtKrBL8oQ8lrANTudoyZykV9PA4aNrPMGHbM4+FYYr0qV6VvuwpdN9SAu6PGmjLaTjFH1p7qiXFD+LkBx8MrA176LAVe+ezj4Z9vuxuMJC5C7lkn+B0ZL1mQDVyorDhxkRPKb1lPW+Zza1qqfa94fo4zzC2wrrwAZzRDjXFb3ivRQ06uAWEDNW+TsLR5jCfa/vZKkUVQZHRa8V/iz4m0z9gS58mj3o3hjVY5VcUJRqgn9C9dneS4RDTFh2WYM63JQqNCwRXRuFGbG6q6yini/XEzb5Q+5kXsHOfjh01vrcYjo28eF40dhK2rRTH7B6EklcPRBQAA"
  },
  "references/approved_guitar_style.mp3": {
    "sha256": "f511536669b3ebfd7aef2fe70213d51aae92c0ad5635844ea891cb194c37b9e3",
    "gzip_base64": "..."
  },
  "scripts/guitar_run.py": {
    "sha256": "156fdeb26589aef3eda9530e2d2296f5f2c135c9edf03ae09da7ce9401acc41d",
    "gzip_base64": "H4sIAAAAAAAC/7Uba3PctvG7fgXCfhDPuaMl24kzctgZRZYcN7Gtsey2M5obDkTi7hjzFYLUI5r7791dACT4OEp2W33wkSCwu1jse2HHcc4TnrG8ZOJWhHUlWCmKMo/qML5KBJNxtk7EQlYihQmFKONUZJV8xQpYJVkmrkXJ4rRI7licrUQpslB4juPswVheVoyX64KXUph39ZPEV14qKh7xipsvf8g8M88przbmOZd7qzJPWQFjsE5DYOfWFKClWuVlat7lnWwe6yvYTShkM1LBDsxzXcfRngLP6yjODfCQZ3kWhzyJ/xJzlsSw/QwYEaTxrZ4dhiIRJa/y0qxJRZqXd4GseCXnwEQpqqA7JmFJWAWRuI5DACvvsnBTAp6/hAIKZAObgyrPE2mgluLPOi5FQOSpT/NmEF81c8r8WmQcmG8Wnnx49+7tp4s5Czci/BLUhaxKwVNAu+HPfvhxzm7KuBIBcX1vLxIrOlG3gNPnpYjm7IqHX0QGD3EGS2s8dtyCiOTsaI/BX8qzeCVkxXw6Oy/JeSTdBgJ7yhzz7OEEZ+YBBVFQidvKnc0IRrxqwHhrUbmOBHJTHoBUyRiXsO98dqjw4V/JYynYP3lSi9OyzEvX+ZzJusAdA0aFjlewkilAToNGbbtDXZhnUYyT4Wg1hYjOEHTZmRAoAM5yipgTawEwnmdrwMRXFeiIRVtLVMtaluWgHBlznSsupTNnTgQf6GFdxxUvndkU4ouGBy1ICconGIKbMwKGSm6AjVDg+w0uxrPICACyxOGhcKbwv6F1LKxLsAAVGAMtopJVG8GOT04XF5UoDMgWPe6aZAppS0Tmgta4SsjoLNQQvuIEnt25uOKA/ezTMvYze/bkyfNDBtqvBoCFtozukBl4r7P4z1po5FeiuhEiA8C48WeHL16++On5jy9eakIbOZAg65fOsbNE2g1/fM0fJhKAi8bCHREh6SwVsD/yKwKzpDckvJmD1LcL2h3IvC5Bs31mSe84isvmZdmsNmwmIF4sg1WcCJcYqsfgnPLkGsY8hA6i8F2Lqv3Y0vOA8GtbmtZgG64E4wwR4t5QFpothElcsAhkJAQjeqc53VVWRd4upWw00tr019L4kI6aI+rIVhcJnqfHiwJEwb1vqXOO2qMEJW71DD7YFtVBkDCGP/MO4M6fo3iBE6vS7Z/bbG4mGKYc9Xg4BTqq9Z6PLDY3g8utYkUpqrrM2H3fRB+xQ8QOLq6W8OKgG8lgSzBo1P2ocSYdKhzjlIIwT9MYOaO91qWevwQgyF/4gj+95WPm+eiRhr4PKhEcBGqchTYozfkbfj2EYZMDLxJOGPfuKFHLs+TuFYjkDUYgMOVpuMlLsD3AVx4Dw0jOxDUIK1cyg0qb5pFIQF6KupLOVjvqkMMJRAFwxG30Z44MAv8crzM4iFJobSXN81s1wy2A1NRJpfhg22Gc64lbsF/SVnZ97u/zTGg5oPii4/Rp6ZhzV5O1a1ciQursrEWG4RMyaCcmZEjGU6H8Yvdwuipl1GhuCfPcOhHbeXapQvhEEbCvHXjYjEDEK9Ddhps6+wJWLsmzNTi7HFi9ojC40qf2itUSbWAmblhZW74ffW2EEVsgIdAE9wG7BCJsdoQgIIDD77Cxt2ze5bGoKpAtjBvutzM1hiYt7YwMQLz0DmazLoc0bsWZy/6S5f+EQwboTh7ZW1PpCdl9I+RKlprX6cjsEfSAvKzitRagcaIoEfKZa2tUV8ghgkd1IJmEF2UqZq2t7oQ+OAE8cikgg4mvMaJvIVvmnXy1tuewhLatsF4qjI8KTslsqB3EkqWxxOQOQWsvCNZJrGoaBLZI2IQO5ZARYJ47fkCh1xZJp44uZHsQbLZ5orFCmKJok95kNseh+MWkGBcVX7dvNyJeb6ogziClUbYNrCJICqQrRmIIrHmB8xhMcOE0eAWuUpE0JfIjKZY+pasyjtYiKPMc7R1mnH2w1gyASTMCOv4gmA3CKnn5bKlxQqrnYULrgRETZeUezMmv2wjR2YRlXIDh14uIjSCKmMEGIruOIXmkyN3kisNPtM6e6o/N0tvt2ttfz4JfP/8SnByf/HqKwgzv/z791L6/O//9l+OT307fv9ZvJx/en7198/rtR9va5tLTiC4R9BIIsBCrMRVjd9JU4q/XJq30qiWo0SB7kKJwiWKkkhcFgIAWeYw8kuRYKI3ofvJoz6BRKwVgcU8TWvey9SRfge3IZF7KB/KwT8dvTrvJT8qrcEOKtlrFYcwTO+eyCESpQUq0mv2NfdoINECQwyU5CK1dfvHYv0hHGA8Bk9TJg8QpCeCoiAVSBbNCyY12uDDR2+sdjDnqD2dnv799f+rgITmHzmDap4/H7y/OPnx8d/rxYnSyqbDkZbihAVXqYH639OHSBH2oauhRZ6qhYfgQ1hEfJKQflWTrowD2IacLwAtWTx0Mj3iBgX5zQCefXx97DPPBJkcFTh4X4CvYRQy8BHYZd2TMEWynb6GmxHXelzc9ACljQOHd7r13s+5vshmN3SBUAZcQITQmGEQTJEpXuUyZzKzS72pral1g0gO9BbPhFpUxLqRSsAPz7rDvqdjm4T8vwCJuxO3l0U+UR/dMKoanKVroMw759Exl1QaQ08fEfHsXjyFWv+Y1ssumdzZmtByQi4tPp+cBWL2T384/vH3/6SJAG4dyj+zXECxnrQw8sA6V0oeESUmLpMSvcWkOHQiMdYVhpxwoPkjNJ0ucMPczBVAAZx498+AOcz9HaRLMNrVIZ13U8EqK6aFyYTSj1ZWY486QMqPPRgP14XT0Bbj67vwCfYLOEHHj99tt42IK2BnotPIyhBEn0wO5YHoreSaRegCB7wlyEBMrfMnqtLiz7XBV3nXjUc39y5aE5aVGi+c2LEJ7eqKrZ7WyIG5DUYzVrb1zNfV9Xp3ldRaR2emSUXCtT81h2mkUijmMTIu/EiYT4Q6nj0XFc8LS+vQ4i8QtJYjIcwHso9TLbR3Kpcqyl9006QF7pIsfw9wAqbTjNPIETbzs2/q32r8n6o4Onkfb4J7AtVnb0gxZqR6MYaqnxinpW273h2i89As8u5TOBvkXX7G1l+PQCaBneVRGjQ4Wg3Zkbu+cS9BDd+Vcqs18f7h9eo+VS4vB+8jg/eVsu2TvRMq/8FgnIugZJbzdQQgOjj7JpQoKgDY4yFVSy43/qawt0nVFEvK8Wtj7MSelI52eRlScysM+9T88oGsVhCC2YDrdLuhWVt02RO87OOVN0DD4Xfs1xzAHiwIQSAdhUavPemyuDYhvzE5XcnzzOttVrJo2jnYWMeqRLaPZRTH0QiEECnDoYAJD2FZpdGq3DDSOVq8INATja1GEeZKY4cHa3ndXb9GD1CxKRGl20CWruwuroWRCLDu6GhhHdYh0WFKg1knS3REBYQsjQHuPOY4hZxqcPd/cJK7oTVWCO+bCZ7sBCkjeQ8qd1XaQLwTuclSI7h0MoXQZFR+pdNov+OH4dnS9cmDVhorB1HFxTQgA5qwX2HnrBCyK88RDdzXcw3LsWAJdsrOMpKM/WSW7cTNmgxip5I1hgriF3Lttkxt8RQnik9cyuCeZUJESgNzqImoHcttPdG0Ec4OunQ16JtKi6tUlXUeP7wwWZ9Z2SeGb5UOz3t21hqydwqxnvNUmHyyTWjjMmgkmN1M0g23YKwP80Qxuiq5RHFauckxUTvUxMtaB0Ugs4Lfua2/koHZu2FQVZ3vf5uPe6PIuIz+93/rv/eWWpXUCDu/eVFz3ybQEqxLYtD9nB7Onz58dHBwcec9WW7nDBw4ivmG/fdIEPtob3sSg6yosbu43kHaPnXnJb1DcVQkWgBsbbordLhWIVOSi2jnL2VzFUG27ZaJTs/NvECRpqCo+mhv5/yZ3oaubdYH1NbdlgnYZ/qS7mNpMLxYwfPPNw5w9eTJxoD0Pnq3ybk+ivfmh/H4A1q2oK3foF+zrFK6zWsGyK+GgY1hcY74hMLLHh4Xc5DcBMLiMBaUlmKHwyrd7Dot8hT9aiyhD5DddWuFAsE2iRHoQiPAr6WK0VCGv80uNAxIYS0aQwz2xmbG/swPv4IcxqRx0603zhZn1WA2mQpXqUKmOVaczpUsp8cDu23dlXNIAY9+fWhXweYPK78s7WwMW/3BCVFJ+GzTLe4EafgOITQQzZ4cHPabazT1TxzXGp9P5c3SUtavh1wXbuRbk2nB2scAahSUK5oSuGQvf9srmzKr2+yZgGsE11e9lTY/CR/EcX++NtiRms9lo4sPYhU5jIp7wlN3r5sT+wFzsL4+8w912XWfbp/RDbXrJxDCxHufTisOekEm0wl8599VdAWkuvs28gNKVINgesXsa2u4OYL7CLzYa9l/4VzsR111SKqu2B6/7LCloinE8dK2uVJEiPXvH5ZrM/zl9cSOhaoGoNEEQ5WEQzKyVHo+igOslrrNYGKkH4pBvPupJc9cssg5qx3rT7ccrZzlYaOk3d9AeD6TT1DVwJi5FoTtYceCrr+ZMAqfbG2Z7VInNUN9953unhXN5MMcbDc+Wk6DAmXwrn8BiLcD8LFqL1dLT7PgQqHh+YG0PBzYiKXznVBf5kzuWY2Uqq3JqNDw/0CBZnVFtAZudeYauBvyNrKZ5ozPLRZvwWptTiF/HfJ3lsorDI2ayW2ypHJ+cojfax/4DDysGhkNdP6OaPZHPyKPp3HiaEFMEWJhOHdFB61tuvPQadiD2Xzh4uLabfMiu7rD2hgX+szeMp8CwFbgoda/CkFQKVCfJXk7To3ubaL7J/fmgojlGC3DO0ytN9cHm5fQZNPWJMfarFkarIAt0yLgvZrWnqI00TRbkpwtVfLY1DEbB+heL68MfQJMl9a/tsdtEDdsaN1gzjZdy+oXJ6ce2eB7j1SFW1FfgXDcCW3VVvEKZMqteNYpGUQkefpP2TKPXNe/OnuuKqs1Uy8ZYopDd/eH3aTOgClvjwmF2RdcNtSywk/PPpkSGXXbYSA1RyRsYVbGuxod2CS/8KbT0g4hNZ3q0oz7eJe+kShTedKtuo93zkXnGVZGDbqPJEQ+tiaZRF0MM5YHH+1zmdilZizHyYAJo+w7oA3uBBTqeUTcPL1yhhezcuCUM5qZ7g7I5nUHttI+tvSSvO4iWmj+AZFC0bL70ql0PUdERepuSnoaRhjQadA0hhjGCPVJH8A+LnE3JTvWpOrnWGJTOfTBdbHQ6t1J7EL/ze/WzfjOlwwNE1l5Bp7umlMAw28D1wyvUKLzl3onyR9u19hVNGlC3ijts6xVjbbZZdxAGgq5vOo90X+iy+2zXsb/WaIzrbQ6evC7KeuuLrQPWd6ZjSRVmVO5BAuZi7EEeVl01opuH1GmWqziDaLa/pBP+6gUH7Od+TorXtIkHvSSNBNO0o6aaUF0G/ANWNAkrGiK8uo2RT3PtYRBcQZKWNjeXbFT9tHHJxkkdLNQ9UWQ+Or+oXam+DBY04tzObIYGk7vNk3aFHn/kTQEb4khzbsTKdi477zyZx/X+doE3JWZdhdV3oihextqlbtRZbU8rkbKr4S1dJpeyLpyNmeCJRvHOi2tfm5FaRP3fstKvY8KDyenjwe3tAUcNpSRvQYCxcxA45r8CYV669x8pVWlATTYAAA=="
  }
}
# END EMBEDDED BRIDGE SOURCE
identity = hashlib.sha256(json.dumps({k:v["sha256"] for k,v in BRIDGE_SOURCE.items()}, sort_keys=True).encode()).hexdigest()
CODE_ROOT = BASE / ("ace_context_source_" + identity[:16])
verified = {}
for name, payload in BRIDGE_SOURCE.items():
    relative = PurePosixPath(name)
    assert not relative.is_absolute() and ".." not in relative.parts and "\\" not in name
    target = (CODE_ROOT / name).resolve()
    assert target.is_relative_to(CODE_ROOT)
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Source checksum mismatch: {name}"
    if target.exists():
        assert target.is_file() and target.read_bytes() == data, f"Source lokal berubah; jangan timpa: {target}"
    verified[name] = data
for name, data in verified.items():
    target = CODE_ROOT / name
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
sys.path[:0] = [str(CODE_ROOT / "scripts"), str(CODE_ROOT / "tools/stem_eval")]
print("Source bridge siap:", CODE_ROOT)
print("Snapshot:", identity)

spec = importlib.util.spec_from_file_location("colab_weight_cache", CODE_ROOT / "scripts/colab_weight_cache.py")
cache = importlib.util.module_from_spec(spec); spec.loader.exec_module(cache)
sys.modules["colab_weight_cache"] = cache
ASSETS = CODE_ROOT / "configs/assets-base.json"
OLD_ROOT = cache.discover(BASE, ASSETS, ROOT_HINT)
ROOT = OLD_ROOT if OLD_ROOT is not None else CODE_ROOT
UPSTREAM = ROOT / "vendor/ACE-Step-1.5"
MODEL_PYTHON = UPSTREAM / ".venv/bin/python"
cache_info = cache.inventory(ROOT, ASSETS)
print("Aksi:", ACTION, "| Cache ditemukan:", OLD_ROOT)
print("Aset tersedia berdasarkan ukuran:", round(cache_info["present_bytes_by_size"]/1024**3, 2), "dari", round(cache_info["total_bytes"]/1024**3, 2), "GiB")
if ACTION == "backup" and OLD_ROOT is None and not DRY_RUN:
    raise RuntimeError("Bobot lama tidak ditemukan pada VM yang tersambung. Sambungan CPU ini mungkin VM baru. Hanya backup Drive/lokal yang sudah ada yang bisa dipulihkan; file VM yang telah dihapus tidak dapat diambil lewat notebook.")


## Simpan atau pilih arsip cache — CPU cukup

Pastikan tujuan punya sekitar 14.9 GB ruang kosong. `USE_DRIVE=True` memakai Google Drive; untuk penyimpanan persisten lain gunakan False dan isi `ARCHIVE_PATH`. File `.partial` menandai ekspor yang terputus, bukan backup selesai. Aset diverifikasi saat disalin; file lain di VM tidak dikemas. Sebelum pindah akun, pastikan backup selesai dan akun baru dapat mengaksesnya.


In [ ]:
if USE_DRIVE and (ACTION == "backup" or not cache_info["complete_by_size"]) and not DRY_RUN:
    from google.colab import drive
    drive.mount("/content/drive")
archive_name = "ace-base-" + cache_info["manifest_sha256"][:12] + ".tar"
if ARCHIVE_PATH is None:
    archive_base = Path("/content/drive/MyDrive/StemsGeneratingCache") if USE_DRIVE else BASE / "weight_backups"
    ARCHIVE_PATH = archive_base / archive_name
ARCHIVE_PATH = Path(ARCHIVE_PATH).expanduser().resolve()
print("Arsip bobot:", ARCHIVE_PATH)
print("Ukuran arsip sekitar 14.9 GB; pastikan ruang penyimpanan tujuan cukup.")
if ACTION == "backup":
    if DRY_RUN:
        print("DRY RUN: inventaris CPU saja; tidak membaca seluruh bobot atau membuat arsip")
    else:
        backup_report = cache.export(ROOT, ARCHIVE_PATH, ASSETS)
        print(json.dumps(backup_report, indent=2))
        print("Backup selesai. Pastikan akun baru dapat mengakses file ini melalui Drive/shortcut atau salinan lokal.")


## Akun/VM GPU tujuan: pulihkan aset dan siapkan runtime

Sel ini dilewati dalam mode backup. Jika backup tersedia, bobot tidak diunduh dari Hugging Face lagi; membaca arsip dari Drive tetap memindahkan data ke disk VM baru. Dependensi runtime dapat memerlukan unduhan tersendiri. Hasil audio/progres run lama perlu disimpan terpisah; arsip bobot tidak memuatnya.


In [ ]:
if ACTION == "prepare":
    if DRY_RUN:
        print("DRY RUN: checkout, restore, instalasi dan pemeriksaan CUDA dilewati")
    else:
        assert shutil.which("nvidia-smi"), "Pilih runtime GPU di akun baru. Untuk backup CPU gunakan ACTION=backup."
        subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)
        if (ROOT / "configs/assets-base.json").is_file():
            assert hashlib.sha256((ROOT / "configs/assets-base.json").read_bytes()).hexdigest() == cache_info["manifest_sha256"], "Manifest runtime lama berbeda; simpan dan pilih ROOT_HINT yang sesuai"
        if not cache_info["complete_by_size"] and not ARCHIVE_PATH.is_file() and not ALLOW_MODEL_DOWNLOAD:
            raise RuntimeError("Arsip bobot belum terlihat oleh akun ini. Periksa path/shortcut Drive atau akses file dari akun lama. Jika benar-benar tidak ada backup, set ALLOW_MODEL_DOWNLOAD=True untuk unduhan baru.")
        from runtime_environment import runtime_environment
        from stream_process import run_logged
        from system_tools import require_audio_tools
        environment = runtime_environment(ROOT)
        if any(shutil.which(name) is None for name in ("ffmpeg", "ffprobe")):
            run_logged(["apt-get", "update"], check=True)
            run_logged(["apt-get", "install", "-y", "ffmpeg"], check=True)
        require_audio_tools()
        required = cache_info["missing_bytes"] + 5*1024**3 + (8*1024**3 if not MODEL_PYTHON.is_file() else 0)
        assert shutil.disk_usage(ROOT).free >= required, "Ruang disk belum cukup untuk runtime dan bobot"
        # Clone/check pinned source before restoring checkpoints; this is CPU-only.
        run_logged([sys.executable, str(ROOT / "scripts/setup_runtime.py")], env=environment, check=True)
        if not cache_info["complete_by_size"] and ARCHIVE_PATH.is_file():
            restore_report = cache.restore(ROOT, ARCHIVE_PATH, ASSETS)
            print(json.dumps(restore_report, indent=2))
        needs_setup = not MODEL_PYTHON.is_file()
        if not needs_setup:
            probe = subprocess.run([str(MODEL_PYTHON), "-c", "import torch,torchaudio,transformers,acestep.handler"], env=environment, capture_output=True, text=True)
            needs_setup = probe.returncode != 0
        if needs_setup:
            import venv
            bootstrap = ROOT / ".bootstrap"; bootstrap_python = bootstrap / "bin/python"
            if not bootstrap_python.is_file(): venv.EnvBuilder(with_pip=False).create(bootstrap)
            run_logged([sys.executable, "-m", "pip", "--python", str(bootstrap_python), "install", "pip", "uv==0.10.0"], env=environment, check=True)
            run_logged([sys.executable, str(ROOT / "scripts/setup_runtime.py"), "--install"], env=environment, check=True)
        command = [str(MODEL_PYTHON), str(CODE_ROOT / "scripts/model_assets.py"), "--manifest", str(ASSETS), "--upstream", str(UPSTREAM)]
        if ALLOW_MODEL_DOWNLOAD: command += ["--download"]
        run_logged(command, env=environment, check=True)
        from cuda_reference import check_cuda
        print(json.dumps(check_cuda(MODEL_PYTHON), indent=2))


## Piano asli dan gaya gitar yang dipilih

Hanya untuk ACTION=prepare. Unggah piano asli bila tidak tersedia. Referensi gaya gitar dari klip yang dipilih sudah disertakan.


In [ ]:
if ACTION == "prepare":
    from google.colab import files
    from stream_process import run_logged
    missing = [name for name in ("numpy", "scipy", "soundfile") if importlib.util.find_spec(name) is None]
    if missing and not DRY_RUN:
        run_logged([sys.executable, "-m", "pip", "install", "-r", str(CODE_ROOT / "requirements-context-chunks.txt")], check=True)
    assert not [n for n in ("numpy", "scipy", "soundfile") if importlib.util.find_spec(n) is None]
    modules = {}
    for name in ("electric_full_song", "guitar_probe", "ace_context_chunks", "guitar_layers"):
        spec = importlib.util.spec_from_file_location(name, CODE_ROOT / "scripts" / (name + ".py"))
        module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
        sys.modules[name] = module; modules[name] = module
    layers = modules["guitar_layers"]
    STYLE_REFERENCE = CODE_ROOT / "references/approved_guitar_style.mp3"
    SOURCE_PATH = globals().get("SOURCE_PATH")
    if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
        uploaded = files.upload()
        names = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
        assert len(names) == 1, "Upload satu piano asli"
        incoming = ROOT / "runs" / ("input_" + uuid.uuid4().hex[:10]); incoming.mkdir(parents=True)
        SOURCE_PATH = incoming / ("piano" + Path(names[0]).suffix.lower()); SOURCE_PATH.write_bytes(uploaded[names[0]])


## Generasi dua track

`MODE="preview30"` default menghasilkan strumming dan petikan pada piano 0–30 detik. Ubah ke `full90` untuk dua track seluruh lagu (enam panggilan model jika belum ada cache untuk piano 4:03). Kedua track memakai referensi gaya yang sama. Penilaian teknik, akor/timing dan glitch tetap perlu didengar.


In [ ]:
if ACTION == "prepare":
    from IPython.display import Audio, display
    MODE = "preview30"  # "full90" untuk seluruh lagu dengan dua track chunk 90 detik
    START = 0
    if MODE == "preview30":
        LAYERS_OUT = ROOT / "runs" / ("guitar_layers30_" + uuid.uuid4().hex[:10])
        report = layers.generate(ROOT, SOURCE_PATH, LAYERS_OUT, STYLE_REFERENCE, start=START, execute=not DRY_RUN)
        previews = [LAYERS_OUT / n for n in ("strum/guitar_preview.mp3", "pick/guitar_preview.mp3", "guitars_layered.mp3", "piano_guitars_mix.mp3")]
    elif MODE == "full90":
        FULL_LAYERS_OUT = globals().get("FULL_LAYERS_OUT")
        if FULL_LAYERS_OUT is None:
            FULL_LAYERS_OUT = ROOT / "runs" / ("guitar_layers_full90_" + uuid.uuid4().hex[:10])
        LAYERS_OUT = Path(FULL_LAYERS_OUT)
        report = layers.generate_full(ROOT, SOURCE_PATH, LAYERS_OUT, STYLE_REFERENCE, execute=not DRY_RUN)
        mixed = LAYERS_OUT / report.get("mix_export_folder", "not_generated")
        previews = [LAYERS_OUT / role / "stitched/guitar_full.mp3" for role in ("strum", "pick")]
        previews += [mixed / n for n in ("guitars_layered.mp3", "piano_guitars_mix.mp3")]
    else:
        raise ValueError("Pilih MODE preview30 atau full90")
    print("Status:", report["status"], "|", LAYERS_OUT)
    print("Referensi gaya SHA256:", report["reference_audio_sha256"])
    if not DRY_RUN:
        for path in previews:
            print(path.name)
            display(Audio(filename=str(path), normalize=False))


## Simpan hasil

Result ZIP menyimpan WAV utama; kirim feedback ZIP untuk penilaian. Sel ini dilewati dalam mode backup.


In [ ]:
if ACTION == "prepare":
    names = ("guitar_layers_result.zip", "guitar_layers_feedback.zip") if MODE == "preview30" else ("guitar_layers_full_result.zip", "guitar_layers_full_feedback.zip")
    for name in names:
        path = LAYERS_OUT / name
        if path.is_file():
            files.download(str(path))
    if DRY_RUN:
        print("Plan saja; belum ada audio model atau ZIP hasil")
